---
---

## Phase 1: Raw Data Ingestion, Cleaning, and Base Topology Construction

In [ ]:
# ====================================================================
# Step 1 — Clean, Unified Imports (future-proofed & reproducible)
# Why this layout:
# - Grouped logically → faster mental scan while debugging
# - Only one import per lib → avoids shadowing/duplicates
# - Optional heavy deps behind try/except → portable notebook
# - Seeded randomness up-front → exact repeatability across runs
# ====================================================================

# --- Standard Library (filesystem, math, collections, typing) ---
import os
import json
import math
import random
import glob
from pathlib import Path
from collections import defaultdict
from typing import Iterable

# --- Core Data & I/O (fast arrays, tables, .mat reader) ---
import numpy as np                    # vectorized math = speed + clarity
import pandas as pd                   # tabular wrangling
import scipy.io                       # for loading OneWeb .mat files

# --- Graphs & Networks (topology, ISLs) ---
import networkx as nx                 # satellite/ISL graph ops

# --- Visualization (plots that stay crisp inside notebooks) ---
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Image
plt.rcParams["figure.dpi"] = 120      # prevent blurry figures on HiDPI

# --- Optional World Map Visualization (kept optional for portability) ---
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAVE_CARTOPY = True               # gate map-specific code on this flag
except ImportError:
    HAVE_CARTOPY = False
    print("Note: Cartopy not installed → map visualizations will be skipped.")

# --- Machine Learning (keep both clf & reg for flexibility) ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (
    classification_report,            # clear per-class summary
    accuracy_score,                   # simple baseline metric
    balanced_accuracy_score,          # robust under class imbalance
    mean_squared_error,               # for regression diagnostics
    r2_score                          # interpretability for regression
)
from sklearn.base import clone        # safe model copying for experiments
import joblib                          # lightweight model persistence

# --- Reproducibility (one seed to rule them all) ---
RANDOM_SEED = 42                       # change once, propagate everywhere
random.seed(RANDOM_SEED)               # Python RNG
np.random.seed(RANDOM_SEED)            # legacy NumPy RNG
rng = np.random.default_rng(RANDOM_SEED)  # modern, explicit RNG for sampling


---

In [ ]:
# =============================================================================
# Step 2 — Output Folders + Key Constants
#
# Why this step exists:
# - Keep all artifacts (figures/tables/logs/models) under a clean experiment
#   directory so runs are reproducible and comparable.
# - Define scientific constants + simulation defaults once (single source of truth).
# =============================================================================

EXPERIMENT_NAME = "t_1_sec_analysis"

# Main output folder → named after the experiment
OUTPUT_DIR = Path(f"output_{EXPERIMENT_NAME}")
# If re-run, don't crash if the folder already exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------- Scientific Constants --------------------------

# Max link distance (km). Beyond this, satellites can't form ISLs.
# 3000 km is a common LEO cut-off; adjust in sensitivity analyses if needed.
DISTANCE_THRESHOLD_KM = 3000.0

# Speed of light (km/s). Guard against inconsistent re-definitions upstream.
_prev_c = globals().get("SPEED_OF_LIGHT_KM_S", None)
if _prev_c is not None:
    assert math.isclose(_prev_c, 299_792.458, rel_tol=1e-9), (
        "Error: SPEED_OF_LIGHT_KM_S already defined with a different value."
    )
SPEED_OF_LIGHT_KM_S = 299_792.458  # underscores improve readability

# -------------------------------- Region Labels ------------------------------
# Keep one canonical list + alias map so downstream code is consistent.
CANONICAL_REGIONS = [
    "Europe", "Asia", "Africa",
    "North America", "South America",
    "Australia", "Other",
]

REGION_ALIASES = {
    "Oceania": "Australia",
    "Ocean/Other": "Other",
    "Other/Ocean": "Other",
}

# Pre-compute fast lookups:
# 1) alias (lowercased) -> canonical
_REGION_ALIASES_LC = {str(k).strip().casefold(): v for k, v in REGION_ALIASES.items()}
# 2) canonical (lowercased) -> canonical (ensures proper casing even if input is "europe")
_CANONICAL_LC_TO_CANONICAL = {c.casefold(): c for c in CANONICAL_REGIONS}
# 3) canonical set for membership checks
_CANONICAL_SET = set(CANONICAL_REGIONS)

def normalize_region_name(region_name: str) -> str:
    """
    Standardize messy region names:
    - Trims spaces
    - Case-insensitive
    - Maps aliases to canonical
    - Restores canonical casing for known canonical names
    """
    if pd.isna(region_name):  # handle NaN gracefully
        return region_name
    name = str(region_name).strip()
    key = name.casefold()
    # Alias mapping first, then canonical casing, otherwise return as-is
    if key in _REGION_ALIASES_LC:
        return _REGION_ALIASES_LC[key]
    if key in _CANONICAL_LC_TO_CANONICAL:
        return _CANONICAL_LC_TO_CANONICAL[key]
    return name

def check_region_consistency(df: pd.DataFrame, column_name: str = "Region") -> bool:
    """
    Validate that `df[column_name]` contains only canonical region names
    *after* normalization. Prints unexpected values (if any) for quick fixes.
    """
    if column_name not in df.columns:
        print(f"Warning: column '{column_name}' not found in the data.")
        return False

    normalized_unique = (
        df[column_name]
        .dropna()
        .map(normalize_region_name)
        .unique()
    )
    unknowns = set(normalized_unique).difference(_CANONICAL_SET)
    if unknowns:
        print(f" Warning: Found unexpected region names in '{column_name}':")
        print(f"   -> {sorted(unknowns)}")
        return False

    print(f" Column '{column_name}' passed the region check. All good.")
    return True

# Final confirmation so it's obvious where artifacts will go
print(f" Configuration complete. Results will be saved in: {os.path.abspath(OUTPUT_DIR)}")

# ----------------------- Simulation Parameters (Defaults) --------------------
# Default values aligned with the report scenario; can be overridden by a params file.
#QUEUE_CAPACITY_PKTS     = 1000
SERVICE_RATE_NORMAL_PPS = 30
SERVICE_RATE_STRESS_PPS = 30
TEMPORAL_WINDOW_S       = 5

# If a JSON params file exists in the OUTPUT_DIR, use it to override defaults
# (traceable & reproducible runs).
params_path = OUTPUT_DIR / "sim_params.json"
if params_path.exists():
    try:
        with open(params_path, "r", encoding="utf-8") as f:
            _params = json.load(f)
        for k in ("QUEUE_CAPACITY_PKTS",
                  "SERVICE_RATE_NORMAL_PPS",
                  "SERVICE_RATE_STRESS_PPS",
                  "TEMPORAL_WINDOW_S"):
            if k in _params:
                # Cast to int for safety; JSON may store strings.
                globals()[k] = int(_params[k])
        print(" Loaded simulation params from sim_params.json")
    except Exception as e:
        print(" Note: could not read sim_params.json → using defaults.", e)

print(
    #f" Params → capacity={QUEUE_CAPACITY_PKTS}, "
    f"normal={SERVICE_RATE_NORMAL_PPS}pps, "
    f"stress={SERVICE_RATE_STRESS_PPS}pps, "
    f"window={TEMPORAL_WINDOW_S}s"
)


---

In [ ]:
# ====================================================================
# Step 3: Load Raw Satellite Data (.mat Files)
#
# The data I’m working with comes in MATLAB .mat format.
# Each file stores different pieces of information:
#   - positions (lat, lon of each satellite)
#   - link distances (a matrix of all possible inter-satellite distances)
#
# The main goals of this step:
#   1. Load the raw .mat files safely
#   2. Make sure the data makes sense (no NaNs, wrong shapes, etc.)
#   3. Convert it into Python-friendly structures (pandas, numpy)
#   4. Clean the link matrix → symmetric + realistic
#   5. Prepare both node info (positions) and edge info (distances/delays)
#
# Basically: after this step I should have a clean network graph skeleton.
# ====================================================================

# Build filenames based on the experiment prefix.
# Why? Because I sometimes want to switch datasets (t_1_sec, t_10_sec, etc.)
# and I don’t want to manually rename paths every time.
DATA_PREFIX = "t_1_sec"
position_filename = Path(f"Satellite_lat_lon_{DATA_PREFIX}.mat")
links_filename = Path(f"Links_Oneweb_{DATA_PREFIX}.mat")

# Print which files are being used → really helpful later when I forget
# which dataset I was working on.
print(f" Loading data for prefix: '{DATA_PREFIX}'")
print(f"  - Position file: {position_filename}")
print(f"  - Links file:    {links_filename}\n")

# Fail fast: if the files are missing, stop here with a clear error.
# Otherwise I’d waste time debugging downstream code.
if not position_filename.exists() or not links_filename.exists():
    raise FileNotFoundError(" Required .mat files not found. Please make sure they are available.")
else:
    print(" Files found. Loading data...")

try:
    # Load the MATLAB files into Python dicts.
    # (Each key in the dict corresponds to a variable name inside MATLAB.)
    satellite_positions = scipy.io.loadmat(position_filename)
    satellite_links = scipy.io.loadmat(links_filename)

    # Extract lat/lon vectors + link distance matrix.
    # Flattening is important because MATLAB sometimes stores them as 2D column vectors.
    latitudes = np.asarray(satellite_positions['sat_lat'], dtype=float).flatten()
    longitudes = np.asarray(satellite_positions['sat_lon'], dtype=float).flatten()
    links_matrix = np.asarray(satellite_links['links'], dtype=float)

    print(" Data loaded successfully from .mat files.")

    # --- Integrity checks ---
    num_satellites = len(latitudes)

    # These are basically "tripwires". If the input data is corrupted or
    # doesn’t match my expectations, I’d rather the code crash here
    # with a clear message than give me wrong results later.
    assert len(longitudes) == num_satellites, "Mismatch between latitudes and longitudes!"
    assert links_matrix.shape == (num_satellites, num_satellites), "Links matrix shape is incorrect."
    assert np.isfinite(latitudes).all() and np.isfinite(longitudes).all(), "NaN in lat/lon"
    assert (-90 <= latitudes).all() and (latitudes <= 90).all(), "Latitude out of range"
    assert (-180 <= longitudes).all() and (longitudes <= 180).all(), "Longitude out of range"

    # Put satellite info into a DataFrame (nicer to work with than raw arrays).
    nodes_df = pd.DataFrame({
        "node_id": np.arange(num_satellites),
        "lat_deg": latitudes,
        "lon_deg": longitudes
    })

    # --- Clean the link matrix ---
    # The raw distance matrix isn’t always perfectly symmetric,
    # because of small floating-point differences or errors.
    # But in physics: distance(A,B) == distance(B,A).
    # So here I force it to be symmetric:
    U = np.triu(links_matrix, 1)   # keep only upper triangle (ignore duplicates)
    sym_links = U + U.T            # mirror it to get full symmetric matrix
    np.fill_diagonal(sym_links, np.nan)  # satellites can’t connect to themselves

    # Filter out obviously broken values:
    MIN_LINK_KM = 10.0
    # <10 km → impossible (two LEO sats can’t be that close)
    sym_links[sym_links < MIN_LINK_KM] = np.nan
    # >3000 km → cut-off I decided earlier (otherwise the graph gets unrealistic)
    sym_links[sym_links > DISTANCE_THRESHOLD_KM] = np.nan
    # Keep a clean copy for downstream steps
    sym = sym_links.copy()

    # --- Edge list (for graph work) ---
    # A matrix is fine for math, but for graph algorithms (NetworkX),
    # an edge list is way more convenient: (node u, node v, distance).
    u_indices, v_indices = np.where(np.triu(np.isfinite(sym_links), 1))
    edges_df = pd.DataFrame({
        "u": u_indices,
        "v": v_indices,
        "Distance_km": sym_links[u_indices, v_indices]
    })

    # Add propagation delay (distance / c).
    # This is one of the key ML features later, so I compute it now.
    if 'SPEED_OF_LIGHT_KM_S' in globals():
        edges_df["propagation_Time"] = edges_df["Distance_km"] / SPEED_OF_LIGHT_KM_S

    # --- More sanity checks ---
    # I keep these because edge-cleaning bugs can be very sneaky.
    assert not edges_df.empty, "No valid edges found after cleaning!"
    assert not edges_df.duplicated(subset=["u", "v"]).any(), "Duplicate edges found!"
    assert np.nanmax(sym_links) <= DISTANCE_THRESHOLD_KM + 1e-9, "Found links > 3000 km!"

    # --- Quick summary ---
    # These printouts are mainly for me to “eyeball” the dataset
    # and make sure nothing looks crazy.
    finite_mask = np.isfinite(links_matrix)
    ut_finite = (finite_mask & np.triu(np.ones_like(links_matrix, dtype=bool), 1)).sum()
    lt_finite = (finite_mask & np.tril(np.ones_like(links_matrix, dtype=bool), -1)).sum()
    near_zero = (finite_mask & (links_matrix < MIN_LINK_KM)).sum()

    print("\n---  Data Summary ---")
    print(f"Total satellites: {num_satellites}")
    print(f"Links matrix size: {links_matrix.shape}")
    print(f"Finite link values: {finite_mask.sum()} (upper: {ut_finite}, lower: {lt_finite})")
    print(f"Very short links (< {MIN_LINK_KM} km): {near_zero}")
    print(f"Valid edges after cleaning: {len(edges_df)}")

    degrees = np.isfinite(sym_links).sum(axis=1)
    print(f"Satellite degree stats: min = {degrees.min()}, mean = {degrees.mean():.1f}, max = {degrees.max()}")

except KeyError as e:
    # Specific error handling:
    # If a MATLAB file is missing a variable (like sat_lat),
    # I don’t want a cryptic crash — I want a clear message.
    print(" ERROR: A variable is missing in one of the .mat files.")
    print(f"The file is likely structured differently than expected. Missing variable: {e}")
    raise


In [ ]:
# ====================================================================
# Step 4: Build Initial Network Graph (Satellites & Positions)
#
# Goal: take the raw lat/lon data and turn it into a clean table
# that maps satellite IDs → coordinates. This table is the backbone
# for everything else: plotting, graphs, and ML features.
#
# Important: I want both a "human-friendly ID" (starts at 1, good for reports)
# and a "programmer ID" (starts at 0, good for numpy/NetworkX).
# ====================================================================

if 'nodes_df' in globals():
    # If I already have nodes_df from Step 3, I just reuse it.
    # Renaming columns makes it clearer in tables and plots.
    positions_df = nodes_df.rename(columns={
        'node_id': 'Node_ID',
        'lat_deg': 'Latitude_deg',
        'lon_deg': 'Longitude_deg'
    }).copy()  # .copy() avoids weird side effects on the original df

    # Add a "Satellite_ID" that starts from 1.
    # This is only for readability in reports (professors prefer Sat-1, Sat-2… not Sat-0).
    positions_df['Satellite_ID'] = positions_df['Node_ID'] + 1

    # Reorder columns → easier to read
    positions_df = positions_df[['Satellite_ID', 'Node_ID', 'Latitude_deg', 'Longitude_deg']]

else:
    # Fallback plan: if nodes_df doesn’t exist (e.g., I skipped Step 3),
    # I rebuild the table from the raw arrays.
    satellite_ids = np.arange(1, num_satellites + 1)   # human-friendly
    node_ids = satellite_ids - 1                       # 0-indexed for code
    positions_df = pd.DataFrame({
        "Satellite_ID": satellite_ids,
        "Node_ID": node_ids,
        "Latitude_deg": latitudes.astype(float),
        "Longitude_deg": longitudes.astype(float)
    })

# --- Quick sanity checks ---
# These are boring but necessary; they prevent hard-to-find bugs later.

# Node_IDs must be a perfect sequence from 0..N-1.
assert positions_df['Node_ID'].min() == 0, "Node_ID must start at 0"
assert positions_df['Node_ID'].max() == len(positions_df) - 1, "Node_ID range has gaps!"

# Make sure no edge in edges_df points to a non-existent node.
if 'edges_df' in globals():
    max_node_used = int(np.max(edges_df[['u', 'v']].to_numpy()))
    assert positions_df['Node_ID'].max() >= max_node_used, \
        "edges_df references a satellite that isn’t in positions_df!"

# Duplicate lat/lon isn’t necessarily a bug (can happen in polar constellations),
# but I want to at least be aware of it.
duplicate_coords = positions_df.duplicated(subset=['Latitude_deg', 'Longitude_deg']).sum()
if duplicate_coords > 0:
    print(f"Note: {duplicate_coords} satellites share the same coordinates (expected in some planes).")

# --- Output ---
print("---  Satellite Positions Table (First 5 Rows) ---")
display(positions_df.head())
print("--------------------------------------------------")

# Save to CSV → this is my first real checkpoint file.
# Later I can reload this directly, instead of re-parsing .mat files.
positions_csv_path = OUTPUT_DIR / "satellite_positions.csv"
positions_df.to_csv(
    positions_csv_path,
    index=False,          # don’t need the pandas row numbers
    float_format='%.6f'   # enough precision to keep lat/lon accurate
)

print(f" Satellite positions saved to:\n  {positions_csv_path.absolute()}")


---

In [ ]:
# ====================================================================
# Step 5: Final Checks on Satellite Data
#
# At this point I already have positions_df (nodes) and edges_df (links).
# Before moving on, I want to double-check that the data is internally consistent.
# These are the last tripwires: if something is broken here, better to
# catch it now than deep inside routing experiments later.
# ====================================================================

if 'edges_df' in globals():
    # Collect all nodes that appear in the edge list
    used_nodes = set(edges_df['u']).union(set(edges_df['v']))
    # And compare against the nodes I actually know about
    known_nodes = set(positions_df['Node_ID'])

    # If the edge list references a node that doesn’t exist in positions_df,
    # that’s a ghost satellite → it would definitely crash pathfinding later.
    missing_nodes = used_nodes.difference(known_nodes)

    # Ideally this should be empty. If not, I print a few so I know what went wrong.
    print(" Missing Node_IDs (should be empty):", sorted(list(missing_nodes))[:10])

# --- ID uniqueness checks ---
# Each satellite must have its own Satellite_ID and Node_ID.
# If there are duplicates, it means I messed up the indexing earlier.
is_sat_id_unique = positions_df['Satellite_ID'].is_unique
is_node_id_unique = positions_df['Node_ID'].is_unique
print(" Satellite_IDs unique:", is_sat_id_unique,
      "|  Node_IDs unique:", is_node_id_unique)

# --- Lat/Lon ranges ---
# This isn’t an error check but more of a sanity check for me.
# Seeing the ranges gives me a feel for coverage:
# - If it’s a polar constellation, I expect to hit close to ±90° latitude.
# - Longitudes should always span -180° to 180°.
lat_min = positions_df['Latitude_deg'].min()
lat_max = positions_df['Latitude_deg'].max()
lon_min = positions_df['Longitude_deg'].min()
lon_max = positions_df['Longitude_deg'].max()
print(" Latitude range:", f"{lat_min:.2f}", "to", f"{lat_max:.2f}",
      "| Longitude range:", f"{lon_min:.2f}", "to", f"{lon_max:.2f}")


---

In [ ]:
# ====================================================================
# Step 6: Clean and Save Final Satellite Link Matrix
#
# Goal: make sure the inter-satellite link matrix is realistic,
# symmetric, and ready to use. After this, I’ll save it as a CSV
# so I don’t have to repeat all the cleaning steps later.
# ====================================================================

# --- 0. Pick a starting matrix ---
# If I already built 'sym' earlier, reuse it (safer, already cleaned).
# Otherwise, fall back to the raw links_matrix and rebuild symmetry.
links_matrix_raw = links_matrix.copy()
if 'sym' in globals():
    symmetric_matrix = sym.copy()
else:
    U = np.triu(links_matrix_raw, 1)   # keep upper triangle only
    symmetric_matrix = U + U.T         # mirror to make it symmetric
    np.fill_diagonal(symmetric_matrix, np.nan)  # no self-links

# --- 1. Clean obvious garbage values ---
# Negative or zero distances don’t make sense → mark them as missing.
symmetric_matrix[symmetric_matrix <= 0] = np.nan

# Very short distances (<10 km) are also unrealistic for LEO satellites.
# I treat them as data errors and remove them.
MIN_LINK_KM = 10.0
symmetric_matrix[symmetric_matrix < MIN_LINK_KM] = np.nan

# --- 2. Apply a maximum distance cap (optional) ---
# This cap (3000 km by default) simulates hardware limits
# on maximum ISL range. It’s not always needed, but it’s
# useful if I want to explore more “realistic” constraints.
MAX_LINK_KM = 3000

if isinstance(MAX_LINK_KM, (int, float)) and np.isfinite(MAX_LINK_KM):
    before = np.isfinite(symmetric_matrix).sum()
    symmetric_matrix[symmetric_matrix > MAX_LINK_KM] = np.nan
    after = np.isfinite(symmetric_matrix).sum()
    # Helpful to see how aggressive the cap was
    print(f" MAX_LINK_KM applied — kept {after} of {before} links")

print(" Link matrix cleaned: removed invalid, too-short, and (optionally) too-long distances.")

# --- 3. Quick stats on the cleaned network ---
finite_mask = np.isfinite(symmetric_matrix)

# Upper vs lower triangle counts should match in a symmetric matrix.
upper_links = (finite_mask & np.triu(np.ones_like(symmetric_matrix, dtype=bool), 1)).sum()
lower_links = (finite_mask & np.tril(np.ones_like(symmetric_matrix, dtype=bool), -1)).sum()

# Degree = number of neighbors per satellite
degrees = np.isfinite(symmetric_matrix).sum(axis=1)

print(f" Total valid links: {finite_mask.sum()} (upper: {upper_links}, lower: {lower_links})")
print(f" Degree of satellites: min = {int(degrees.min())}, "
      f"mean = {degrees.mean():.1f}, max = {int(degrees.max())}")

# --- 4. Save to CSV (checkpoint) ---
# I always save major cleaned datasets as CSVs.
# Reason: if I mess something up later, I don’t have to redo all the cleaning.
node_ids = np.arange(num_satellites)
final_matrix_df = pd.DataFrame(symmetric_matrix, index=node_ids, columns=node_ids)
final_matrix_df.index.name = "Node_ID"

matrix_csv_path = OUTPUT_DIR / "satellite_link_matrix_cleaned.csv"
final_matrix_df.to_csv(matrix_csv_path, float_format="%.6f")

print(f" Cleaned satellite link matrix saved to:\n  {matrix_csv_path.absolute()}")


---

In [ ]:
# ====================================================================
# Step 7: Build Satellite Adjacency List
#
# Why this step?
# A link matrix or edge list is fine for math, but many algorithms
# (routing, ML neighbor features, etc.) are easier if I have a direct
# “who-is-connected-to-who” list. That’s basically an adjacency list.
#
# End goal here: a DataFrame that shows for each satellite:
#   - Node_ID (0-based, for code)
#   - Satellite_ID (1-based, for humans)
#   - its list of neighbors
#   - its degree (number of neighbors)
#
# I’ll also save it as CSV for reproducibility.
# ====================================================================

# --- 1. Check prerequisites ---
if 'edges_df' not in globals() or edges_df.empty:
    raise RuntimeError(" edges_df is missing. Build it in an earlier step before continuing.")

# --- 2. Create empty neighbor lists ---
# Using a dict keyed by Node_ID. It’s memory-cheap and fast to fill.
num_nodes = int(max(edges_df['u'].max(), edges_df['v'].max())) + 1
neighbors_dict = {node_id: [] for node_id in range(num_nodes)}

# --- 3. Fill neighbor lists ---
# Looping over edges: for every link (u,v), add v to u’s list and u to v’s list.
# I use itertuples() instead of iterrows() → faster for big DataFrames.
for row in edges_df.itertuples(index=False):
    u = int(row.u)
    v = int(row.v)
    neighbors_dict[u].append(v)
    neighbors_dict[v].append(u)

# --- 4. Deduplicate and sort ---
# Even though edges_df *shouldn’t* have duplicates, I don’t want to risk
# subtle errors. set() removes duplicates, sorted() gives me consistent order.
for node_id in neighbors_dict:
    neighbors_dict[node_id] = sorted(set(neighbors_dict[node_id]))

print(" Neighbors dictionary created successfully.")

# --- 5. Convert to DataFrame ---
neighbors_df = pd.DataFrame({
    "Node_ID": list(neighbors_dict.keys()),
    "Neighbor_IDs": list(neighbors_dict.values())
}).sort_values("Node_ID").reset_index(drop=True)

# --- 6. Add Satellite_ID (human-friendly, 1-based) ---
if 'positions_df' in globals() and 'Node_ID' in positions_df.columns:
    sat_map = positions_df.set_index('Node_ID')['Satellite_ID']
    neighbors_df["Satellite_ID"] = neighbors_df["Node_ID"].map(sat_map)
else:
    # Fallback: assume Satellite_ID = Node_ID + 1
    neighbors_df["Satellite_ID"] = neighbors_df["Node_ID"] + 1

# --- 7. Add degree (neighbor count) ---
neighbors_df["Degree"] = neighbors_df["Neighbor_IDs"].apply(len)

# --- 8. Quick sample output ---
print("\n---  Adjacency List (First 5 Rows) ---")
display(neighbors_df.head())
print("----------------------------------------")

# --- 9. Save to CSV ---
# CSV doesn’t natively store lists. Trick: convert the neighbor list
# into a JSON string. That way I can parse it back later if needed.
neighbors_df["Neighbor_IDs_JSON"] = neighbors_df["Neighbor_IDs"].apply(json.dumps)

final_neighbors_df = neighbors_df[["Node_ID", "Satellite_ID", "Degree", "Neighbor_IDs_JSON"]]
neighbors_csv_path = OUTPUT_DIR / "satellite_adjacency_list.csv"
final_neighbors_df.to_csv(neighbors_csv_path, index=False)

print(f" Adjacency list saved to:\n  {neighbors_csv_path.absolute()}")

# --- 10. Final validation (handshaking lemma) ---
# In graph theory: sum of all degrees = 2 × number of edges.
# If this holds, my adjacency list is guaranteed consistent with edges_df.
deg_sum = int(neighbors_df["Degree"].sum())
num_edges = len(edges_df)

print(f" Validation: Total degree = {deg_sum} vs 2 × #edges = {2 * num_edges} ->",
      "OK" if deg_sum == 2 * num_edges else " MISMATCH")


---

In [ ]:
# ====================================================================
# Step 8: Region Tagging of Satellites
#
# Goal: figure out which part of the world each satellite is flying over.
# This is useful for region-based analysis later (e.g., traffic patterns).
#
# Main idea:
# - If GeoPandas is available, use it to check if a sat is “within” a continent.
# - If not, fall back to a simple default (Region = "Other").
# - Finally, normalize the messy region names so they match my canonical list.
# ====================================================================

# --- A. Try importing GeoPandas ---
# GeoPandas is amazing for spatial stuff, but it’s heavy and often tricky to install.
# I don’t want my whole notebook to crash just because it’s missing,
# so I mark it as optional.
try:
    import geopandas as gpd
    from shapely.geometry import Point
    HAVE_GPD = True
except ImportError:
    HAVE_GPD = False
    print("Note: GeoPandas not available — all satellites will default to Region = 'Other'.")

# --- B. Helper: tag satellites with GeoPandas ---
def _tag_with_geopandas(positions_df):
    """
    Use continent polygons from GeoPandas to tag each satellite
    with the region it’s currently over.
    """

    # Load a world map dataset
    try:
        world_path = gpd.datasets.get_path('naturalearth_lowres')
        world = gpd.read_file(world_path)
    except Exception:
        # Fallback: fetch a Natural Earth dataset from the web if the local one is missing
        NE_COUNTRIES_ZIP = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
        world = gpd.read_file(NE_COUNTRIES_ZIP)

    # Standardize column names to lowercase (less chance of key errors later)
    world.rename(columns={col: col.lower() for col in world.columns}, inplace=True)

    # Make sure we actually have a continent column
    if "continent" not in world.columns:
        for alt in ("region_un", "region_wb", "regionun"):
            if alt in world.columns:
                world["continent"] = world[alt]
                break
        else:
            raise ValueError("No 'continent' column found in the world map data.")

    # Dissolve = merge all country polygons into big continent polygons
    continents = world[["continent", "geometry"]].dissolve(by="continent").reset_index()

    # Turn satellites into geometric points (lat/lon) so GeoPandas can work with them
    geometry = gpd.points_from_xy(positions_df['Longitude_deg'], positions_df['Latitude_deg'], crs="EPSG:4326")
    geo_positions_df = gpd.GeoDataFrame(positions_df.copy(), geometry=geometry, crs="EPSG:4326")

    # The magic step: spatial join → “which continent polygon contains this satellite point?”
    satellites_with_regions = gpd.sjoin(geo_positions_df, continents, how="left", predicate="within")

    # Clean up output
    result = satellites_with_regions.drop(columns=["geometry", "index_right"], errors='ignore')
    result = result.rename(columns={"continent": "Region"})

    # If no continent matched → it must be over ocean
    result["Region"] = result["Region"].fillna("Ocean")

    return result

# --- C. Try tagging satellites ---
if HAVE_GPD:
    try:
        positions_df = _tag_with_geopandas(positions_df)
    except Exception as e:
        # Last line of defense → don’t crash if geopandas fails
        print(f"GeoPandas failed — falling back to Region = 'Other'. Reason: {e}")
        positions_df = positions_df.copy()
        positions_df["Region"] = "Other"
else:
    # If GeoPandas not installed at all
    positions_df = positions_df.copy()
    positions_df["Region"] = "Other"

# --- D. Normalize region names ---
# My thesis doesn’t care about fine-grained categories like "Ocean" or "Antarctica".
# I collapse these into a generic "Other" bucket.
if 'REGION_ALIASES' in globals():
    REGION_ALIASES.setdefault("Ocean", "Other")
    REGION_ALIASES.setdefault("Antarctica", "Other")

if 'normalize_region_name' in globals():
    positions_df["Region"] = positions_df["Region"].map(normalize_region_name)

positions_df["Region"] = positions_df["Region"].replace({
    "Ocean": "Other",
    "Antarctica": "Other"
})

# --- E. Optional region sanity check ---
if 'check_region_consistency' in globals():
    _ok = check_region_consistency(positions_df, "Region")

# --- F. Show quick stats ---
print("\n---  Satellite Count per Region ---")
try:
    display(positions_df["Region"].value_counts().sort_index())
except Exception:
    print(positions_df["Region"].value_counts().sort_index())
print("------------------------------------------------")

# --- G. Save to CSV ---
# Important to save because running geopandas is slow.
# Later I can just load this file and skip the geospatial join step.
positions_csv_path = OUTPUT_DIR / "satellite_positions_with_regions.csv"
positions_df.to_csv(positions_csv_path, index=False)
print(f" Region-tagged satellite data saved to:\n  {positions_csv_path.absolute()}")


---
---

## Phase 2: Regional Sampling, Connectivity Assurance, and Visualization

In [ ]:
# ====================================================================
# Step 9: Reload Cleaned Data and Perform Consistency Checks
#
# Why this step?
# I don’t want to re-run all the cleaning every time I restart.
# Instead, I reload the saved checkpoint files and check that they
# are still consistent. If something is off, I’d rather find it now
# than in the middle of an experiment.
# ====================================================================

# --- 1. Load cleaned link matrix ---
mat_path = OUTPUT_DIR / "satellite_link_matrix_cleaned.csv"
mat = pd.read_csv(mat_path, index_col=0)

# Pandas often reads numeric column labels as strings → forcing them
# back to int avoids subtle bugs in graph indexing later.
mat.index = mat.index.astype(int)
mat.columns = mat.columns.astype(int)

# --- Rebuild edges_df from the matrix (convenience) ---
# This makes the notebook restart-friendly: I can regenerate edges_df
# directly from the saved matrix without touching the raw .mat files.
ui, vi = np.where(np.triu(np.isfinite(mat.values.astype(float)), 1))
edges_df = pd.DataFrame({
    'u': ui,
    'v': vi,
    'Distance_km': mat.values[ui, vi].astype(float)
})
# Recompute propagation delay instead of trusting older values,
# just in case the distance cleaning changed.
edges_df['propagation_Time'] = edges_df['Distance_km'] / SPEED_OF_LIGHT_KM_S

# --- 2. Reload satellite positions (basic version) ---
positions_path = OUTPUT_DIR / "satellite_positions.csv"
pos = pd.read_csv(positions_path)

# This assert enforces my global rule: Satellite_ID = Node_ID + 1.
# If this fails, all downstream mapping between IDs breaks.
assert (pos['Satellite_ID'] == pos['Node_ID'] + 1).all(), \
    "Mismatch in Satellite_ID vs Node_ID in positions.csv"

# --- 3. Reload positions with region info ---
positions_with_regions_path = OUTPUT_DIR / "satellite_positions_with_regions.csv"
posw = pd.read_csv(positions_with_regions_path)

# Same rule check here, to confirm the enriched file didn’t drift.
assert (posw['Satellite_ID'] == posw['Node_ID'] + 1).all(), \
    "Mismatch in positions_with_regions.csv"

# --- 4. Reload adjacency list ---
adj_path = OUTPUT_DIR / "satellite_adjacency_list.csv"
adj = pd.read_csv(adj_path)

# Neighbor lists were stored as JSON strings → convert back now,
# so I can actually use them in Python as lists.
adj['Neighbor_IDs'] = adj['Neighbor_IDs_JSON'].apply(json.loads)

# Handshaking lemma: Σ(degrees) must equal 2 × #edges.
# If this doesn’t hold, then something is inconsistent between
# my adjacency list and edge list.
assert adj['Degree'].sum() == 2 * len(edges_df), \
    " Degree sum does not match number of edges"

print(" All data files loaded and verified successfully.")


---

In [ ]:
# ====================================================================
# Step 10: Quota-Based Sampling by Region (Delay-Aware Traffic Prep)
#
# Why this step?
# I don’t want to pick satellites at random for traffic experiments.
# Instead, I want the sampling to roughly reflect real-world internet
# usage across regions. Otherwise, my traffic model would be biased
# (e.g., too many sources in Europe, not enough in Africa).
#
# Idea:
# - Define quotas (# of satellites per region I want to sample).
# - Clean up the region names so they match my canonical list.
# - Make sure sampled sats are actually connected (Degree > 0).
# - Save the final list for reproducibility in later stages.
# ====================================================================

# --- 1. Define region quotas (raw vs canonical) ---
SAMPLING_QUOTAS_RAW = {
    "North America": 40,
    "Europe": 20,
    "Asia": 10,
    "South America": 8,
    "Africa": 8,
    "Oceania": 6,                   # I use "Australia" in my cleaned data
    "Ocean": 5,                     # folded into "Other"
    "Seven seas (open ocean)": 0,   # deliberately ignored
    "Antarctica": 5                 # also folded into "Other"
}

# These aliases keep the quotas consistent with my canonical region labels
QUOTA_ALIASES = {
    "Oceania": "Australia",
    "Ocean": "Other",
    "Seven seas (open ocean)": "Other",
    "Antarctica": "Other"
}

# defaultdict(int) avoids a bunch of if/else when merging quotas
SAMPLING_QUOTAS = defaultdict(int)
for region_name, count in SAMPLING_QUOTAS_RAW.items():
    canon_name = QUOTA_ALIASES.get(region_name, region_name)
    SAMPLING_QUOTAS[canon_name] += count

# Sanity check: warn if I asked for a region that doesn’t exist in the data
regions_in_data = set(positions_df["Region"].unique())
for region in list(SAMPLING_QUOTAS.keys()):
    if region not in regions_in_data:
        print(f" Region '{region}' not found in the data — quota ({SAMPLING_QUOTAS[region]}) will be skipped.")

# --- 2. Make sure degree info is present ---
# Degree is critical because I only want satellites that can actually forward traffic.
# If positions_df doesn’t have it yet, merge from neighbors_df.
if 'Degree' not in positions_df.columns:
    positions_df = positions_df.merge(neighbors_df[["Satellite_ID", "Degree"]],
                                      on="Satellite_ID", how="left").fillna({'Degree': 0})
    positions_df['Degree'] = positions_df['Degree'].astype(int)
    print(" Degree information added to positions_df.")

# --- 3. Sample satellites by region quotas ---
rng = random.Random(RANDOM_SEED)  # seed ensures reproducibility
initial_selected_ids = []

print("\n---  Selecting Satellites by Region Quotas ---")
for region, quota in SAMPLING_QUOTAS.items():
    if quota <= 0:
        continue  # skip regions where quota = 0

    # Only consider satellites in this region *and* connected to the network
    in_region = positions_df["Region"] == region
    connected = positions_df["Degree"] > 0
    candidates = positions_df.loc[in_region & connected, "Satellite_ID"].tolist()

    if len(candidates) == 0:
        print(f"   Region '{region}': no satellites with Degree > 0. Skipping.")
        continue

    # Don’t oversample: cap at available satellites
    take = min(quota, len(candidates))
    if take < quota:
        print(f"  Region '{region}': quota = {quota}, available = {len(candidates)} → taking {take}.")

    # rng.sample → reproducible and without replacement
    selected = rng.sample(candidates, take) if take < len(candidates) else candidates
    initial_selected_ids.extend(selected)

print("--------------------------------------------")

# --- 4. Finalize selected set ---
selected_satellite_ids = sorted(set(initial_selected_ids))  # enforce uniqueness

# I need Node_IDs too, since the graph logic depends on them
sat_to_node = positions_df.set_index('Satellite_ID')['Node_ID'].to_dict()
selected_node_ids = sorted(sat_to_node[sid] for sid in selected_satellite_ids)

target_total = sum(SAMPLING_QUOTAS.values())

print("\n Quota Sampling Summary:")
print(f"  - Target total (sum of quotas): {target_total}")
print(f"  - Actual selected satellites: {len(selected_satellite_ids)}")

# Quick cross-check: see how many sats per region I actually ended up with
sel_df = positions_df[positions_df['Satellite_ID'].isin(selected_satellite_ids)]
region_counts = sel_df['Region'].value_counts().sort_index()

print("\n Satellites Selected Per Region:")
print(region_counts)

# --- 5. Save selection to CSV ---
# Saving is crucial: without this snapshot I’d risk re-sampling
# a different set of satellites next time.
selection_csv = OUTPUT_DIR / "initial_sample_nodes.csv"
(sel_df[['Satellite_ID', 'Node_ID', 'Region', 'Degree']]
 .sort_values(['Region', 'Node_ID'])
 .to_csv(selection_csv, index=False))

print(f"\n Initial satellite sample saved to:\n  {selection_csv.absolute()}")


---

In [ ]:
# ====================================================================
# Step 11: Refine Sample to Ensure Connectivity
#
# Why this step?
# After quota sampling, I might end up with satellites that are
# “lonely” (no neighbor inside the sample) or with a fragmented
# sample (multiple disconnected groups). Either case is bad because:
#   - A source with no in-sample neighbor can’t really send traffic.
#   - If the sample is split into islands, end-to-end routing fails.
#
# So here I apply a *policy* to fix that. I can choose between:
#   - 'ensure_neighbor': each node must have at least one peer in the sample
#   - 'ensure_connected': the sample must form one connected component (strict)
#   - 'off': no refinement
#
# In practice, I’ll use "ensure_connected" for experiments,
# but I keep the policy switchable to test alternatives.
# ====================================================================

# --- 1. Pick refinement policy ---
TIGHTEN_POLICY = "ensure_connected"

# --- 2. Make sure I have a sample to refine ---
if 'selected_node_ids' not in globals():
    if 'selected_satellite_ids' in globals():
        # Safer to convert Satellite_ID → Node_ID now, since Node_ID is what
        # networkx and routing logic depend on.
        sat_to_node = positions_df.set_index('Satellite_ID')['Node_ID'].to_dict()
        selected_node_ids = sorted(sat_to_node[sid] for sid in selected_satellite_ids)
    else:
        # If I restart the notebook, I want the script to recover gracefully
        # by reloading from the CSV snapshot instead of failing.
        sample_path = OUTPUT_DIR / "initial_sample_nodes.csv"
        if sample_path.exists():
            sample_df = pd.read_csv(sample_path)
            selected_node_ids = sorted(sample_df['Node_ID'].tolist())
            selected_satellite_ids = sorted(sample_df['Satellite_ID'].tolist())
        else:
            raise RuntimeError(" No sample found. Please run the sampling step first.")

# --- 3. Build network graph for connectivity checks ---
if 'G_dist' in globals():
    # Reuse existing graph to save time, but copy it so I don’t mess up the original.
    G0 = G_dist.copy()
else:
    if 'edges_df' not in globals() or edges_df.empty:
        # No graph, no connectivity check → fail early instead of producing nonsense.
        raise RuntimeError(" edges_df is missing. Please build it first.")
    G0 = nx.Graph()
    # Adding all nodes first avoids silent errors if some nodes have no edges.
    G0.add_nodes_from(range(int(edges_df[['u', 'v']].to_numpy().max()) + 1))
    G0.add_edges_from(edges_df[['u', 'v']].itertuples(index=False, name=None))

print(f" Refinement policy: '{TIGHTEN_POLICY}'")
print(f"  - Sample size (before): {len(selected_node_ids)}")

# --- 4. Extract subgraph of sample nodes ---
S = set(selected_node_ids)
subgraph = G0.subgraph(S).copy()
# I explicitly add nodes back because networkx drops isolates by default,
# but I still want to track them for pruning policies.
subgraph.add_nodes_from(S)

print(f"  - Subgraph size: {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges")

# --- 5. Apply refinement policy ---
initial_sample_size = len(selected_node_ids)
final_nodes = list(selected_node_ids)

if TIGHTEN_POLICY == "ensure_neighbor":
    # Removing isolates can create new isolates, so I loop until stable.
    changed = True
    while changed:
        isolates = list(nx.isolates(subgraph))
        if not isolates:
            changed = False
        else:
            subgraph.remove_nodes_from(isolates)
    final_nodes = sorted(subgraph.nodes())

elif TIGHTEN_POLICY == "ensure_connected":
    # I only keep the largest connected component to guarantee
    # that all sampled nodes can reach each other.
    if subgraph.number_of_nodes() > 0:
        components = list(nx.connected_components(subgraph))
        if components:
            largest = max(components, key=len)
            final_nodes = sorted(largest)
            print(f"  - Found {len(components)} components → keeping largest (size {len(largest)})")
        else:
            final_nodes = []
    else:
        final_nodes = []

elif TIGHTEN_POLICY == "off":
    # Useful for debugging: lets me compare with and without refinement.
    pass
else:
    raise ValueError("Unknown TIGHTEN_POLICY")

selected_node_ids = sorted(final_nodes)

# --- 6. Compare region counts before vs after ---
def region_counts(node_ids):
    df = positions_df[positions_df['Node_ID'].isin(node_ids)]
    return df['Region'].value_counts().sort_index()

before_counts = region_counts(S)
after_counts = region_counts(selected_node_ids)

print("\n Refinement Complete")
print(f"  - Sample size before: {initial_sample_size}")
print(f"  - Sample size after:  {len(selected_node_ids)}")

print("\n Region Counts (Before):")
print(before_counts.to_string())

print("\n Region Counts (After):")
print(after_counts.to_string())

# --- 7. Save refined sample ---
# Degree_in_subgraph is critical: it shows how well-connected a node is
# within the *experiment sample*, which can differ from its global degree.
degree_in_subgraph = dict(subgraph.degree(selected_node_ids))

refined_df = positions_df[positions_df['Node_ID'].isin(selected_node_ids)].copy()
refined_df['Degree_in_subgraph'] = refined_df['Node_ID'].map(degree_in_subgraph).fillna(0).astype(int)

refined_csv = OUTPUT_DIR / "final_experiment_nodes.csv"
refined_df[['Satellite_ID', 'Node_ID', 'Region', 'Degree', 'Degree_in_subgraph']]\
    .sort_values(['Region', 'Node_ID'])\
    .to_csv(refined_csv, index=False)

print(f"\n Refined sample saved to:\n  {refined_csv.absolute()}")


---

In [ ]:
# ====================================================================
# Step 12: Visualize the Final Sampled Satellite Network on a Map
#
# Why this step?
# After all the data prep and sampling, I want to see the network.
# A plot makes it obvious if my sample is balanced, connected,
# or accidentally clustered in one region. It’s also good for my thesis
# because visuals are easier to interpret than raw tables.
# ====================================================================

print(" Preparing and plotting the sampled satellite network...")

# --- 1. Make sure I have the final sample ---
if 'selected_node_ids' not in globals():
    refined_csv = OUTPUT_DIR / "final_experiment_nodes.csv"
    if refined_csv.exists():
        refined_df = pd.read_csv(refined_csv)
        selected_node_ids = sorted(refined_df["Node_ID"].tolist())
        selected_satellite_ids = sorted(refined_df["Satellite_ID"].tolist())
    else:
        # Fail early instead of drawing a meaningless empty map
        raise RuntimeError(" Refined satellite sample not found. Run the refinement step first.")

# --- 2. Filter edges to only those inside the sample ---
if 'edges_df' not in globals() or edges_df.empty:
    raise RuntimeError(" edges_df is missing. Please rebuild it before plotting.")

selected_set = set(selected_node_ids)
# Only keep edges where both endpoints are in my final sample
subgraph_edges_df = edges_df[
    edges_df['u'].isin(selected_set) & edges_df['v'].isin(selected_set)
].copy()

# --- 3. Precompute coordinates lookup ---
# Looking up coordinates in a dict is much faster than searching
# the DataFrame inside the plotting loop.
pos_map = positions_df.set_index("Node_ID")[["Longitude_deg", "Latitude_deg"]].to_dict("index")

# --- 4. Optional edge filter (for clarity only) ---
PLOT_MAX_KM = 3000.0
if "Distance_km" in subgraph_edges_df.columns:
    # Long edges can clutter the map, so I drop them here purely for visualization
    edges_to_draw = subgraph_edges_df[subgraph_edges_df["Distance_km"] <= PLOT_MAX_KM].copy()
    print(f" Drawing edges (<= {PLOT_MAX_KM} km): {len(edges_to_draw)}")
else:
    # Fallback: if distance info is missing, better to plot all than none
    print("Note: 'Distance_km' not found — drawing all available edges.")
    edges_to_draw = subgraph_edges_df.copy()

# Limit edge count so the map doesn’t turn into a solid blue mess
EDGE_DRAW_CAP = 5000
if EDGE_DRAW_CAP is not None and len(edges_to_draw) > EDGE_DRAW_CAP:
    # Random subset is enough to show the pattern without overplotting
    edges_to_draw = edges_to_draw.sample(EDGE_DRAW_CAP, random_state=RANDOM_SEED)

# --- 5. Set up the map canvas ---
fig = plt.figure(figsize=(16, 8))  # wide aspect ratio works best for world maps

if 'HAVE_CARTOPY' in globals() and HAVE_CARTOPY:
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_global()
    # Using muted land/ocean colors to keep satellites/edges visible on top
    ax.add_feature(cfeature.LAND, zorder=0, edgecolor='black', facecolor='#d2b48c')
    ax.add_feature(cfeature.OCEAN, zorder=0, facecolor='#add8e6')
    ax.add_feature(cfeature.COASTLINE, zorder=1)
    ax.add_feature(cfeature.BORDERS, zorder=1, linestyle=':')
    try:
        # Gridlines help orient the reader, but sometimes Cartopy bugs out
        ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    except Exception:
        ax.gridlines(linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
else:
    # Fallback if Cartopy isn’t installed — not as pretty, but good enough
    print("Note: Cartopy not available — using plain Matplotlib.")
    ax = plt.axes()
    ax.set_xlabel("Longitude (deg)")
    ax.set_ylabel("Latitude (deg)")
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)

# --- 6. Plot all satellites (background context) ---
# Showing the full constellation in gray gives context:
# the reader can see which satellites were *not* chosen.
ax.scatter(
    positions_df["Longitude_deg"], positions_df["Latitude_deg"],
    s=8, c='gray', alpha=0.5,
    transform=ccrs.PlateCarree() if ('HAVE_CARTOPY' in globals() and HAVE_CARTOPY) else None,
    label="All Satellites"
)

# --- 7. Plot sampled satellites (highlighted) ---
# Making them red with black edges so they pop visually on the map.
selected_df = positions_df[positions_df["Node_ID"].isin(selected_node_ids)]
ax.scatter(
    selected_df["Longitude_deg"], selected_df["Latitude_deg"],
    s=50, c='red', edgecolors='black',
    transform=ccrs.PlateCarree() if ('HAVE_CARTOPY' in globals() and HAVE_CARTOPY) else None,
    label="Final Sampled Satellites", zorder=10
)

# --- 8. Draw edges between sampled satellites ---
for row in edges_to_draw.itertuples(index=False):
    u, v = int(row.u), int(row.v)
    lon_u, lat_u = pos_map[u]["Longitude_deg"], pos_map[u]["Latitude_deg"]
    lon_v, lat_v = pos_map[v]["Longitude_deg"], pos_map[v]["Latitude_deg"]

    if 'HAVE_CARTOPY' in globals() and HAVE_CARTOPY:
        # Geodetic transform draws great-circle arcs (the correct physical path)
        ax.plot([lon_u, lon_v], [lat_u, lat_v],
                color='cyan', linewidth=1.0, alpha=0.9,
                transform=ccrs.Geodetic(), zorder=5)
    else:
        # Plain Matplotlib can’t do great-circle, so I settle for straight lines
        ax.plot([lon_u, lon_v], [lat_u, lat_v],
                color='cyan', linewidth=0.8, alpha=0.6, zorder=5)

# --- 9. Final touches ---
ax.legend(loc="lower left")
ax.set_title("Final Sampled Satellite Network Topology", fontsize=18, pad=20)

# Save high-res figure so it’s ready for thesis/report use
map_filepath = OUTPUT_DIR / "final_sampled_topology_map.png"
plt.savefig(map_filepath, dpi=200, bbox_inches='tight')
print(f"\n Network map saved to:\n  {map_filepath.absolute()}")


---

In [ ]:
# ====================================================================
# Step 13: Dijkstra Algorithm (Baseline)
#
# Why this step?
# I need a classical baseline to compare against ML-based routing later.
# Dijkstra is the natural choice: it gives the minimum-latency path
# if all I care about is propagation time. This lets me benchmark
# how much “smarter” (if at all) my ML agents are.
# ====================================================================

print(" Building the final weighted graph and running path analysis...")

# --- 1. Ensure I have the final refined sample ---
if 'selected_node_ids' not in globals():
    refined_csv = OUTPUT_DIR / "final_experiment_nodes.csv"
    if refined_csv.exists():
        # Reload snapshot so this script can run standalone
        refined_df = pd.read_csv(refined_csv)
        selected_node_ids = sorted(refined_df["Node_ID"].tolist())
        selected_satellite_ids = sorted(refined_df["Satellite_ID"].tolist())
    elif 'selected_satellite_ids' in globals():
        # Sometimes I only have Satellite_IDs → convert them to Node_IDs
        mapping = positions_df.set_index('Satellite_ID')["Node_ID"].to_dict()
        selected_node_ids = sorted(mapping[sid] for sid in selected_satellite_ids)
    else:
        raise RuntimeError(" No refined selection found. Run the refinement step first.")

# --- 2. Filter edges to only those between selected nodes ---
if 'edges_df' not in globals() or edges_df.empty:
    raise RuntimeError(" edges_df is missing. Please rebuild it.")

selected_set = set(selected_node_ids)
subgraph_edges_df = edges_df[
    edges_df["u"].isin(selected_set) & edges_df["v"].isin(selected_set)
].copy()

# --- 3. Guarantee latency weight exists ---
# Dijkstra needs a cost metric. For this baseline,
# I’m using propagation_Time = distance / c.
if 'propagation_Time' not in subgraph_edges_df.columns:
    subgraph_edges_df["propagation_Time"] = (
        subgraph_edges_df["Distance_km"] / SPEED_OF_LIGHT_KM_S
    )

# --- 4. Build weighted graph ---
# Keeping it undirected since ISLs are bidirectional.
# Adding both distance and delay as edge weights for flexibility later.
G_subgraph = nx.from_pandas_edgelist(
    subgraph_edges_df,
    source="u", target="v",
    edge_attr=["Distance_km", "propagation_Time"]
)

print(f" Graph built:")
print(f"  - Nodes: {G_subgraph.number_of_nodes()}")
print(f"  - Edges: {G_subgraph.number_of_edges()}")

# --- 5. Pick test source (Europe) and target (North America) ---
# Europe ↔ NA is a realistic benchmark path because it’s one of the
# busiest intercontinental corridors for Internet traffic.
positions_df = pd.read_csv(OUTPUT_DIR / "satellite_positions_with_regions.csv")
sampled_df = positions_df[positions_df["Node_ID"].isin(selected_node_ids)]

eu_nodes = sampled_df[sampled_df["Region"] == "Europe"]["Node_ID"].tolist()
na_nodes = sampled_df[sampled_df["Region"] == "North America"]["Node_ID"].tolist()

if eu_nodes and na_nodes:
    # For reproducibility: pick the lowest-ID satellite in each region
    source_node = min(eu_nodes)
    target_node = min(na_nodes)

    # Handy for printing human-readable Sat IDs instead of just Node_IDs
    node_to_sat = positions_df.set_index("Node_ID")["Satellite_ID"].to_dict()

    print(f"\n Finding fastest path from:")
    print(f"    Node {source_node} (Europe, Sat {node_to_sat[source_node]})")
    print(f"    → Node {target_node} (North America, Sat {node_to_sat[target_node]})")

    try:
        # Core baseline: shortest path by propagation time
        path_nodes = nx.dijkstra_path(
            G_subgraph, source=source_node, target=target_node,
            weight="propagation_Time"
        )
        path_Time = nx.dijkstra_path_length(
            G_subgraph, source=source_node, target=target_node,
            weight="propagation_Time"
        )
        path_dist_km = nx.dijkstra_path_length(
            G_subgraph, source=source_node, target=target_node,
            weight="Distance_km"
        )

        path_sat_ids = [node_to_sat[n] for n in path_nodes]

        print("\n---  Fastest Path Results ---")
        print(f"Path (Node_IDs):      {path_nodes}")
        print(f"Path (Satellite_IDs): {path_sat_ids}")
        print(f"Hop Count:            {len(path_nodes) - 1}")
        print(f"Total Distance:       {path_dist_km:.2f} km")
        # Reporting in ms since that’s the standard in networking
        print(f"Propagation Delay:    {path_Time * 1000:.4f} ms")
        print("-----------------------------------")

    except nx.NetworkXNoPath:
        # Important: sometimes refinement cuts Europe off from NA.
        # Better to print a clear message than silently fail.
        print(f" No path found between Node {source_node} and Node {target_node}.")
else:
    # This happens if sampling/refinement dropped one of the regions entirely
    print("\n Could not find nodes in both Europe and North America.")
    print("   Try adjusting the sampling quotas or refinement settings.")


---

In [ ]:
# ====================================================================
# Step 14: Save Final Subgraph Adjacency List
#
# Why this step?
# I want a clean adjacency list of only the *final sample* satellites.
# This is useful because:
#   - It’s lighter than keeping the full constellation.
#   - It gives me exactly the neighbor relationships used in experiments.
#   - Saving it as CSV makes the experiment reproducible.
# ====================================================================

print(" Building and saving the adjacency list for the final satellite subgraph...")

# --- 1. Load refined satellite sample (if needed) ---
if 'selected_node_ids' not in globals():
    sel_path = OUTPUT_DIR / "final_experiment_nodes.csv"
    if sel_path.exists():
        _sel = pd.read_csv(sel_path)
        selected_node_ids = sorted(_sel['Node_ID'].tolist())
        selected_satellite_ids = sorted(_sel['Satellite_ID'].tolist())
    else:
        # Better to fail here than generate an empty/misleading adjacency list
        raise RuntimeError(" No refined selection found. Run the refinement step first.")

# --- 2. Rebuild subgraph if missing ---
if 'G_subgraph' not in globals():
    if 'edges_df' not in globals() or edges_df.empty:
        raise RuntimeError(" edges_df is missing. Please rebuild it first.")
    
    # Filtering ensures I only keep edges where *both* endpoints are in the sample.
    selected_set = set(selected_node_ids)
    sub_edges = edges_df[edges_df['u'].isin(selected_set) & edges_df['v'].isin(selected_set)]
    # Rebuilding makes the script restart-friendly, so I don’t depend on previous steps.
    G_subgraph = nx.from_pandas_edgelist(sub_edges, source='u', target='v')

# --- 3. Extract neighbors for each node ---
# Using networkx’s adjacency() is the simplest way to guarantee
# I capture the actual graph structure instead of manually parsing edges.
final_neighbors_dict = {
    int(node): sorted(list(neighbors.keys()))  # sorted for consistency
    for node, neighbors in G_subgraph.adjacency()
}

# --- 4. Convert neighbor dict → DataFrame ---
# DataFrames make it easier to enrich with metadata (Satellite_ID, Degree, etc.)
# and are the natural format for saving to CSV.
final_neighbors_df = pd.DataFrame({
    "Node_ID": list(final_neighbors_dict.keys()),
    "Neighbor_IDs_in_Sample": list(final_neighbors_dict.values())
}).sort_values("Node_ID").reset_index(drop=True)

# --- 5. Add extra info (Satellite_ID + Degree) ---
# Adding Satellite_ID makes the table human-readable.
# Degree_in_Sample is useful for quickly checking connectivity balance.
node_to_sat = positions_df.set_index('Node_ID')['Satellite_ID'].to_dict()
final_neighbors_df["Satellite_ID"] = final_neighbors_df["Node_ID"].map(node_to_sat)
final_neighbors_df["Degree_in_Sample"] = final_neighbors_df["Neighbor_IDs_in_Sample"].apply(len)

# --- 6. Convert neighbor lists to JSON + save ---
# Lists can’t be stored cleanly in CSV. JSON encoding ensures
# I can reload them later without losing structure.
final_neighbors_df["Neighbor_IDs_JSON"] = final_neighbors_df["Neighbor_IDs_in_Sample"].apply(json.dumps)

# Keep only the clean final columns
out_cols = ["Node_ID", "Satellite_ID", "Degree_in_Sample", "Neighbor_IDs_JSON"]
final_df_to_save = final_neighbors_df[out_cols]

# Save snapshot for reproducibility
adj_list_path = OUTPUT_DIR / "final_subgraph_adjacency_list.csv"
final_df_to_save.to_csv(adj_list_path, index=False)

print(f"\n Final subgraph adjacency list saved to:\n  {adj_list_path.absolute()}")

# --- 7. Preview a few rows ---
# Quick sanity check: if this looks wrong, I know immediately before using it downstream.
display(final_df_to_save.head())


---

In [ ]:
# ====================================================================
# Step 15: Load and Validate the Edge List
#
# Why this step?
# The edge list is the backbone of everything: routing, ML features,
# connectivity checks. If it’s messy (self-loops, duplicates, wrong order),
# my later results would be untrustworthy. So here I make sure it’s:
#   - Loaded reliably (from any of several possible sources),
#   - Put into one standard “u < v” format,
#   - Clean of impossible edges (self-loops, bad distances),
#   - Saved back in canonical form for future steps.
# ====================================================================

print(" Loading and validating the satellite edge list...")

# I may have multiple edge list formats around, so I prepare paths for both.
std_edges_path = OUTPUT_DIR / "edges_df.csv"
alt_edges_path = OUTPUT_DIR / "satellite_link_edge_list.csv"

# --- 1. Define load functions ---
# I keep flexible loaders so I don’t get stuck if one file is missing.
# This way the notebook is restart-proof and can adapt to available data.

def _load_from_alt():
    df = pd.read_csv(alt_edges_path, dtype={"From_ID": int, "To_ID": int, "Distance_km": float})
    # Renaming keeps column names consistent with my convention (u, v).
    return df.rename(columns={"From_ID": "u", "To_ID": "v"})

def _load_from_std():
    df = pd.read_csv(std_edges_path)
    # Explicit types avoid subtle pandas dtype issues.
    df["u"] = df["u"].astype(int)
    df["v"] = df["v"].astype(int)
    df["Distance_km"] = df["Distance_km"].astype(float)
    return df

def _build_from_matrix():
    # Ultimate fallback: reconstruct edges from the cleaned link matrix.
    mat_path = OUTPUT_DIR / "satellite_link_matrix_cleaned.csv"
    mat = pd.read_csv(mat_path, index_col=0)
    mat.index = mat.index.astype(int)
    mat.columns = mat.columns.astype(int)
    M = mat.values.astype(float)
    # Grab all valid links from the upper triangle to avoid duplicates.
    ui, vi = np.where(np.triu(np.isfinite(M), 1))
    df = pd.DataFrame({"u": ui, "v": vi, "Distance_km": M[ui, vi].astype(float)})
    return df

# --- 2. Try loading edge list ---
try:
    if alt_edges_path.exists():
        edges_df = _load_from_alt()
        source_used = alt_edges_path.name
    elif std_edges_path.exists():
        edges_df = _load_from_std()
        source_used = std_edges_path.name
    else:
        # If nothing is available, reconstruct from the matrix instead of failing.
        edges_df = _build_from_matrix()
        source_used = "built_from_matrix"
    print(f" Edge list loaded from: {source_used}")
except FileNotFoundError:
    raise FileNotFoundError(" No edge list or matrix found. Please run Phase 1 first.")

# --- 3. Canonicalize edge directions ---
# For undirected graphs, (A,B) and (B,A) should not both exist.
# My rule: always store as (min, max) → guarantees consistency.
edges_df["u"] = edges_df["u"].astype(int)
edges_df["v"] = edges_df["v"].astype(int)
swap_mask = edges_df["u"] > edges_df["v"]
if swap_mask.any():
    u_old = edges_df.loc[swap_mask, "u"].copy()
    edges_df.loc[swap_mask, "u"] = edges_df.loc[swap_mask, "v"]
    edges_df.loc[swap_mask, "v"] = u_old

# --- 4. Remove invalid or duplicate edges ---
# Self-loops (u == v) make no sense in a physical satellite network.
self_loops = (edges_df["u"] == edges_df["v"]).sum()
if self_loops:
    print(f" Removing {self_loops} self-loops.")
    edges_df = edges_df[edges_df["u"] != edges_df["v"]]

# After enforcing u < v, true duplicates can show up → drop them.
before = len(edges_df)
edges_df = edges_df.drop_duplicates(subset=["u", "v"]).reset_index(drop=True)
duplicates_removed = before - len(edges_df)
if duplicates_removed:
    print(f" Dropped {duplicates_removed} duplicate edges.")

# --- 5. Validate physical constraints ---
# Distance must be realistic (positive, >= 10 km). This avoids
# bugs in routing and nonsense results in delay calculations.
assert (edges_df["Distance_km"] > 0).all(), " Found non-positive distances!"
assert edges_df["Distance_km"].min() >= 10.0 - 1e-9, " Found distance < 10 km!"

# --- 6. Cross-check node IDs ---
# Defensive check: ensures edges don’t reference non-existent satellites.
if 'positions_df' in globals():
    max_node = positions_df["Node_ID"].max()
    assert edges_df[["u", "v"]].min().min() >= 0, " Found negative Node_IDs!"
    assert edges_df[["u", "v"]].max().max() <= max_node, " Node_IDs exceed position data!"

# --- 7. Ensure propagation delay column exists ---
# Needed as a weight for Dijkstra and ML features.
if "propagation_Time" not in edges_df.columns:
    edges_df["propagation_Time"] = edges_df["Distance_km"] / SPEED_OF_LIGHT_KM_S

# --- 8. Final integrity checks ---
# At this point the edge list *must* be canonical and duplicate-free.
assert (edges_df["u"] < edges_df["v"]).all(), " Edge list is not canonical (u < v)!"
assert not edges_df.duplicated(subset=["u", "v"]).any(), " Duplicate edges remain!"

print(" Data validation passed (u < v, no self-loops, distances valid).")

# --- 9. Save standardized copies ---
# Overwriting ensures that next time, loading will start from clean data,
# not raw messy CSVs.
edges_df.to_csv(std_edges_path, index=False, float_format="%.6f")
edge_list_df = edges_df.rename(columns={"u": "From_ID", "v": "To_ID"})
edge_list_df.to_csv(alt_edges_path, index=False, float_format="%.6f")

print(f" Edge lists saved to:\n  - {std_edges_path}\n  - {alt_edges_path}")

# --- 10. Quick preview ---
# Sanity check: seeing the head of the file often catches subtle mistakes early.
print("\n---  Edge List Overview ---")
print(f"Total number of unique edges: {len(edges_df)}")
display(edge_list_df.head(5))


---

In [ ]:
# ====================================================================
# Step 16 : Validate Degrees and Distance Ranges
#
# Why this step?
# Even after cleaning, it’s easy for subtle mistakes to creep in
# (mismatched adjacency vs. edge list, wrong node ranges, bad distances).
# These checks don’t change the data — they give me confidence that
# everything is consistent before I build traffic models or ML features.
# ====================================================================

print(" Final checks on adjacency and edge list consistency...")

# --- 1. Degree vs. edge count (handshaking lemma) ---
# The sum of all node degrees must equal exactly 2 × number of edges.
# If not, something is off: maybe a corrupted adjacency file or
# an error in neighbor calculation.
adj_path = OUTPUT_DIR / "satellite_adjacency_list.csv"
if adj_path.exists():
    adj_df = pd.read_csv(adj_path)

    # Defensive coding: if Degree column is missing, rebuild it
    # from the saved neighbor JSONs instead of failing.
    if "Degree" not in adj_df.columns and "Neighbor_IDs_JSON" in adj_df.columns:
        adj_df["Degree"] = adj_df["Neighbor_IDs_JSON"].apply(json.loads).apply(len)

    total_degree = adj_df["Degree"].sum()
    expected_degree = 2 * len(edges_df)

    print(f" Degree Check: Total Degree = {total_degree} | 2 × #Edges = {expected_degree}")
    if total_degree != expected_degree:
        # I don’t assert here — just warn — because at this stage I’d rather
        # see the mismatch than crash the notebook.
        print(" Warning: Degree sum does not match 2 × edge count!")
else:
    print(" Adjacency list not found. Skipping degree check.")

# --- 2. Node ID ranges ---
# Quick scan: all node IDs should be >= 0 and ≤ max in positions_df.
# This catches silly errors like a negative index or an out-of-range link.
min_node_id = edges_df[["u", "v"]].to_numpy().min()
max_node_id = edges_df[["u", "v"]].to_numpy().max()
print(f" Node ID Range in Edge List: {min_node_id} to {max_node_id}")

# --- 3. Distance sanity range ---
# My earlier cleaning rules enforced 10 km ≤ distance ≤ ~3000 km.
# Checking here confirms those rules were actually applied,
# and also gives me a sense of the network’s geographic scale.
min_dist = edges_df["Distance_km"].min()
max_dist = edges_df["Distance_km"].max()
print(f" Distance Range: {min_dist:.2f} km to {max_dist:.2f} km")


---

In [ ]:
# ====================================================================
# Step 17: Load, Merge, Validate, and Save Final Master Dataset
#
# Why this step?
# Up to now, I’ve been working with separate pieces: positions,
# adjacency lists, regions, degrees. For experiments, I need one
# *self-contained master dataset*. This makes the workflow:
#   - Easier to reproduce,
#   - Less error-prone (everything is in one CSV),
#   - Ready for ML pipelines.
#
# The checks here are deliberately strict: if something is wrong,
# I want to catch it now before using this dataset in simulations.
# ====================================================================

print(" Loading and merging satellite sample data...")

positions_path = OUTPUT_DIR / "satellite_positions_with_regions.csv"
adjacency_path = OUTPUT_DIR / "final_subgraph_adjacency_list.csv"

try:
    # --- 1. Load inputs ---
    positions_df = pd.read_csv(positions_path)   # has lat/lon/region
    sampled_df = pd.read_csv(adjacency_path)     # has final subgraph neighbors
    print(" Files loaded successfully.")

    # Adjacency lists are stored as JSON strings → convert back to Python lists
    if "Neighbor_IDs" not in sampled_df.columns and "Neighbor_IDs_JSON" in sampled_df.columns:
        sampled_df["Neighbor_IDs"] = sampled_df["Neighbor_IDs_JSON"].apply(json.loads)

    # --- 2. Merge with positions ---
    # Goal: enrich adjacency info with coordinates + region.
    merge_columns = ["Node_ID", "Latitude_deg", "Longitude_deg", "Region"]
    if "Degree" in positions_df.columns:   # include global degree if available
        merge_columns.append("Degree")

    master_df = pd.merge(
        sampled_df,
        positions_df[merge_columns],
        on="Node_ID",
        how="left",            # keep all sampled nodes no matter what
        validate="one_to_one"  # sanity: each Node_ID must match exactly one row
    )

    # --- 3. Validation checks ---
    # These checks are intentionally strict to guarantee dataset integrity.

    # Check 1: Required columns must exist
    required_columns = [
        "Node_ID", "Satellite_ID", "Region",
        "Latitude_deg", "Longitude_deg", "Degree_in_Sample"
    ]
    if "Neighbor_IDs_JSON" not in master_df.columns:
        if "Neighbor_IDs" in master_df.columns:
            master_df["Neighbor_IDs_JSON"] = master_df["Neighbor_IDs"].apply(json.dumps)
        else:
            raise ValueError("Missing 'Neighbor_IDs_JSON' and 'Neighbor_IDs'.")
    missing_columns = [col for col in required_columns if col not in master_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    # Check 2: Satellite_ID must always = Node_ID + 1 (my consistency rule)
    assert (master_df["Satellite_ID"] == master_df["Node_ID"] + 1).all(), \
        " Satellite_ID must be Node_ID + 1"

    # Check 3: No missing values in critical columns
    check_cols = required_columns + ["Neighbor_IDs_JSON"]
    assert master_df[check_cols].isnull().sum().sum() == 0, " Missing values in required columns"

    # Check 4: Degree_in_Sample must equal actual neighbor list length
    if "Neighbor_IDs" in master_df.columns:
        neighbor_counts = master_df["Neighbor_IDs"].apply(len)
        assert (neighbor_counts == master_df["Degree_in_Sample"]).all(), \
            " Degree_in_Sample does not match neighbor list"

    # Check 5: Closure check → all neighbors must also be in the sample
    selected_nodes = set(master_df["Node_ID"])
    all_neighbors = set(n for neighbors in master_df["Neighbor_IDs"] for n in neighbors)
    missing_neighbors = all_neighbors - selected_nodes
    assert len(missing_neighbors) == 0, f" Missing nodes in sample: {list(missing_neighbors)[:10]}"

    print(" All validation checks passed.")
    print(f" Total satellites in final sample: {len(master_df)}")

    # --- 4. Save master dataset ---
    # At this point, the dataset is “golden”: clean, validated, final.
    final_output_df = master_df[required_columns + ["Neighbor_IDs_JSON"]] \
        .sort_values(["Region", "Node_ID"]).reset_index(drop=True)

    final_path = OUTPUT_DIR / "master_sample_dataframe.csv"
    backup_path = OUTPUT_DIR / "final_sample_master.csv"  # backup copy

    final_output_df.to_csv(final_path, index=False, float_format="%.6f")
    backup_path.write_text(final_output_df.to_csv(index=False))  # redundant safety

    print(f"\n Final dataset saved to:\n   - {final_path.name}\n   - {backup_path.name}")

    # --- 5. Preview ---
    # Always worth eyeballing the first few rows before trusting it downstream.
    print("\n Preview of Final Dataset:")
    display(final_output_df.head())

# --- 6. Error handling ---
except FileNotFoundError as e:
    print(f" File not found → {e.filename}")
    print("   Make sure you have completed the previous saving steps first.")
except AssertionError as e:
    print(f" Validation failed: {e}")
except ValueError as e:
    print(f" Error: {e}")
except Exception as e:
    print(f" Unexpected error: {e}")


---
---

## Phase 3: Robust Master Loading and Final Data Validation

In [ ]:
# ====================================================================
# Step 18: Load and Validate Satellite Positions Before Normalization
#
# Why this step?
# Before I start scaling/normalizing features (for ML),
# I need to be 100% sure the raw positions data is:
#   - Complete (no missing values),
#   - Consistent (IDs match rules, no duplicates),
#   - Physically valid (coords in range, degrees make sense).
#
# If something is wrong here and I normalize it blindly,
# I’d just be amplifying bad data → bad models. So this is
# the “gatekeeper” step: fail fast if anything looks suspicious.
# ====================================================================

print(" Starting Phase 4: Data Normalization")
print("  Warning: Some CSV files may be overwritten in this phase.")

# --- 1. Load the positions file ---
positions_filepath = OUTPUT_DIR / "satellite_positions_with_regions.csv"

try:
    # Being explicit about dtypes avoids pandas “guessing wrong”
    # (like turning Node_ID into float). I only load the columns I need.
    usecols = ["Satellite_ID", "Node_ID", "Latitude_deg", "Longitude_deg", "Region"]
    positions_df = pd.read_csv(
        positions_filepath,
        usecols=usecols,
        dtype={
            "Satellite_ID": int,
            "Node_ID": int,
            "Latitude_deg": float,
            "Longitude_deg": float,
            "Region": "string"  # pandas StringDtype is safer than plain object
        }
    )
    print(" Positions file loaded successfully.")

    # --- 2. Ensure Degree column exists ---
    # Degree is a core feature (connectivity). If it’s missing,
    # I try to recover it from the adjacency list. If that fails,
    # I fall back to Degree=0 with a warning (better than crashing).
    if "Degree" not in positions_df.columns:
        adj_path = OUTPUT_DIR / "satellite_adjacency_list.csv"
        if adj_path.exists():
            adj = pd.read_csv(adj_path)
            adj["Neighbor_IDs"] = adj["Neighbor_IDs_JSON"].apply(json.loads)
            if "Degree" not in adj.columns:
                adj["Degree"] = adj["Neighbor_IDs"].apply(len)

            # Merge degree info into positions_df by Node_ID
            if "Node_ID" in adj.columns:
                positions_df = positions_df.merge(adj[["Node_ID", "Degree"]], on="Node_ID", how="left")
            else:  # fallback: use Satellite_ID if Node_ID missing
                positions_df = positions_df.merge(adj[["Satellite_ID", "Degree"]], on="Satellite_ID", how="left")

            # Fill NaN with 0 in case some nodes had no neighbors
            positions_df["Degree"] = positions_df["Degree"].fillna(0).astype(int)
            print(" Degree column merged from adjacency list.")
        else:
            positions_df["Degree"] = 0
            print("  Adjacency list not found → Degree set to 0.")

    # --- 3. Data validation ---
    # These checks are strict on purpose. If any fail, I stop immediately
    # instead of silently pushing bad data forward.

    # Check 1: ID rules
    assert positions_df["Satellite_ID"].is_unique, " Satellite_ID is not unique!"
    assert positions_df["Node_ID"].is_unique, " Node_ID is not unique!"
    assert (positions_df["Satellite_ID"] == positions_df["Node_ID"] + 1).all(), \
        " Satellite_ID must equal Node_ID + 1"

    # Check 2: No missing values in core columns
    required_cols = ["Satellite_ID", "Node_ID", "Latitude_deg", "Longitude_deg", "Region", "Degree"]
    assert positions_df[required_cols].isnull().sum().sum() == 0, " Missing values in important columns"

    # Check 3: Geographic ranges
    assert positions_df["Latitude_deg"].between(-90, 90).all(), " Latitude out of range"
    assert positions_df["Longitude_deg"].between(-180, 180).all(), " Longitude out of range"

    # Check 4: Region names must be canonical
    if 'normalize_region_name' in globals():
        positions_df["Region"] = positions_df["Region"].map(normalize_region_name)
    if 'check_region_consistency' in globals():
        ok = check_region_consistency(positions_df, "Region")
        assert ok, " Region names are not consistent!"

    print(" Data passed all validation checks.")

# --- Error handling ---
except FileNotFoundError:
    print(f" ERROR: File not found → {positions_filepath}")
except AssertionError as e:
    print(f" VALIDATION FAILED: {e}")
except Exception as e:
    print(f" An unexpected error occurred: {e}")

# --- 4. Quick inspection (only runs if data is valid) ---
if 'positions_df' in globals():
    print("\n---  Satellite Counts by Region ---")
    try:
        display(positions_df['Region'].value_counts().sort_index())
    except Exception:
        print(positions_df['Region'].value_counts().sort_index())

    print("\n---  First 5 Rows of Data ---")
    try:
        display(positions_df.head(5))
    except Exception:
        print(positions_df.head(5).to_string(index=False))


---

In [ ]:
# ====================================================================
# Step 19: Load and Validate the Satellite Link Distance Matrix
#
# Why this step?
# The Satellite Link Distance matrix is the backbone of my network model.
# If this matrix is corrupted (not symmetric, wrong labels, bad values),
# then literally every routing or ML step that follows will be wrong.
#
# So here I load the matrix carefully and apply strict checks:
#   - Must be square (same # of rows and cols).
#   - No self-links (diagonal = NaN).
#   - Row/col labels must line up (Node_ID alignment).
#   - Must be symmetric (distance A→B == B→A).
#   - Distances must respect physical rules (≥ 10 km).
#
# Basically: if it passes this checkpoint, I can trust it.
# ====================================================================

print(" Loading and validating the cleaned link matrix...")

matrix_filepath = OUTPUT_DIR / "satellite_link_matrix_cleaned.csv"

try:
    # --- 1. Load matrix ---
    # index_col=0 → keeps Node_IDs as row labels (not just plain numbers 0..N).
    matrix_df = pd.read_csv(matrix_filepath, index_col=0)

    # Pandas sometimes loads header numbers as strings → I force them to int.
    matrix_df.index = matrix_df.index.astype(int)
    matrix_df.columns = matrix_df.columns.astype(int)

    # Defensive move: coerce any non-numeric junk into NaN instead of crashing.
    matrix_df = matrix_df.apply(pd.to_numeric, errors="coerce")

    print(" Link matrix loaded successfully.")

    # --- 2. Validation checks ---
    # Fail fast if *anything* is suspicious.

    # Check 1: Must be square.
    assert matrix_df.shape[0] == matrix_df.shape[1], " Matrix is not square!"

    # Check 2: No self-links allowed → diagonal must be NaN.
    assert matrix_df.isna().values.diagonal().all(), " Diagonal is not all NaN!"

    # Check 3: Row/col labels must match perfectly (otherwise alignment is broken).
    assert (matrix_df.index.values == matrix_df.columns.values).all(), \
        " Row/column labels are not aligned!"

    # Check 4: Symmetry is critical (undirected graph assumption).
    M = matrix_df.values.astype(float)
    mask = np.isfinite(M) & np.isfinite(M.T)
    max_diff = 0.0 if not mask.any() else np.nanmax(np.abs(M[mask] - M.T[mask]))
    assert max_diff <= 1e-9, f" Matrix is not symmetric! Max diff = {max_diff:.3e}"

    # Check 5: Distances must be > 0 and ≥ 10 km (from earlier cleaning rule).
    finite_vals = M[np.isfinite(M)]
    assert (finite_vals > 0).all(), " Found non-positive distances!"
    assert finite_vals.min() >= 10.0 - 1e-9, f" Found distance < 10 km (min = {finite_vals.min():.4f})"

    print(" All validation checks passed (square, aligned, symmetric, min ≥ 10 km).")

except FileNotFoundError:
    print(f" ERROR: File not found → {matrix_filepath}")
except AssertionError as e:
    print(f" VALIDATION FAILED: {e}")
except Exception as e:
    print(f" Unexpected error: {e}")

# --- 3. Quick overview (if everything passed) ---
if 'matrix_df' in globals():
    print("\n---  Matrix Overview ---")
    print(f"Matrix shape: {matrix_df.shape}")

    # Quick cross-check: edge count in matrix vs edge list.
    if 'edges_df' in globals() and not edges_df.empty:
        edges_from_matrix = np.isfinite(matrix_df.values).sum() // 2
        print(f"Edges from matrix (upper triangle): {edges_from_matrix}")
        print(f"Edges from edges_df: {len(edges_df)}")

    # Showing top-left corner helps spot obvious corruption quickly.
    print("\n Top-left 5x5 of the matrix (Node_ID x Node_ID):")
    display(matrix_df.iloc[:5, :5])


---

In [ ]:
# ====================================================================
# Step 20 : Load or Rebuild Master Dataset
#
# Why this step?
# The master dataset is the single source of truth for my traffic model.
# It ties together everything I’ve built so far:
#   - Node_ID ↔ Satellite_ID mapping
#   - Geographic region tags
#   - Adjacency (who can talk to who)
#   - Degree information
#
# If this dataset is missing or broken, the whole simulation pipeline
# falls apart. That’s why this step:
#   1. First tries to load the master file from disk.
#   2. If missing, rebuilds it from adjacency + positions data.
#   3. Normalizes columns so the downstream code always sees
#      the same consistent schema.
#   4. Runs strict validation checks before declaring it “ready”.
# ====================================================================

print(" Initializing the traffic simulation model...")

# --- 1. Try to Load the Master Dataset ---
# Why: Re-using a saved master file is faster and guarantees consistency
# with previous runs. This prevents silent re-sampling of satellites.
master_candidates = [
    OUTPUT_DIR / "master_sample_dataframe.csv",
    OUTPUT_DIR / "final_sample_master.csv"
]

master_df = None
for path in master_candidates:
    if path.exists():
        master_df = pd.read_csv(path)
        print(f" Master dataset loaded from: {path.name}")
        break

# --- 2. Rebuild if not found ---
# Why: I don’t want my workflow to die just because the master file
# wasn’t saved. As long as adjacency + positions are available,
# I can regenerate the master table.
if master_df is None:
    print(" Master dataset not found. Attempting to rebuild it...")
    adj_path = OUTPUT_DIR / "final_subgraph_adjacency_list.csv"
    pos_path = OUTPUT_DIR / "satellite_positions_with_regions.csv"

    if adj_path.exists() and pos_path.exists():
        adj = pd.read_csv(adj_path)
        pos = pd.read_csv(pos_path)

        # Neighbor lists are stored as JSON strings for portability → parse back to Python lists.
        if "Neighbor_IDs" not in adj.columns and "Neighbor_IDs_JSON" in adj.columns:
            adj["Neighbor_IDs"] = adj["Neighbor_IDs_JSON"].apply(json.loads)

        # Merge positions info (lat/lon + region) into adjacency info (connectivity).
        master_df = adj.merge(
            pos[["Node_ID", "Satellite_ID", "Region"]],
            on="Node_ID",
            how="left",
            validate="one_to_one"  # ensures no duplicate Node_IDs sneak in
        )
        print(" Master dataset rebuilt from adjacency + position files.")
    else:
        raise FileNotFoundError(" No master dataset found and input files are missing.")

# --- 3. Normalize / Fill Missing Columns ---
# Why: Different save paths may have produced slightly different schemas.
# This step enforces one canonical format so later steps don’t break.
if "Satellite_ID" not in master_df.columns and "Node_ID" in master_df.columns:
    master_df["Satellite_ID"] = master_df["Node_ID"] + 1

if "Neighbor_IDs_JSON" not in master_df.columns and "Neighbor_IDs" in master_df.columns:
    master_df["Neighbor_IDs_JSON"] = master_df["Neighbor_IDs"].apply(json.dumps)

if "Neighbor_IDs" not in master_df.columns and "Neighbor_IDs_JSON" in master_df.columns:
    master_df["Neighbor_IDs"] = master_df["Neighbor_IDs_JSON"].apply(json.loads)

if "Degree_in_Sample" not in master_df.columns and "Neighbor_IDs" in master_df.columns:
    master_df["Degree_in_Sample"] = master_df["Neighbor_IDs"].apply(len).astype(int)

# --- 4. Final Checks ---
# Why: Fail fast if anything is inconsistent. This ensures the master
# dataset is safe to use in the simulation and reproducible.
required_columns = ["Node_ID", "Satellite_ID", "Region", "Degree_in_Sample", "Neighbor_IDs_JSON"]
missing_columns = [col for col in required_columns if col not in master_df.columns]

assert not missing_columns, f" Missing required columns: {missing_columns}"
assert (master_df["Satellite_ID"] == master_df["Node_ID"] + 1).all(), \
    " Satellite_ID must equal Node_ID + 1"

print(f" Master dataset is ready with {len(master_df)} rows.")


---
---

## Phase 4: Traffic Modeling and Spatio-Temporal Analysis

In [ ]:
# ====================================================================
# Step 21: Define Diurnal Traffic Model and Simulation Parameters
#
# Why this step?
# My simulation isn’t useful unless traffic looks like the *real world*.
# Traffic is not constant — it rises and falls with local time of day
# (Europe busy at noon, America at night, etc.).
#
# This step builds a **diurnal traffic model**:
#   - A function mapping (region, UTC hour) → average arrival rate λ
#   - Global parameters for simulation (duration, packet size, RNG seed)
#
# The goal is to capture realistic variability so routing algorithms
# face believable workloads, not flat synthetic noise.
# ====================================================================

# --- 1. Define traffic rate model ---
# Why: Regions follow their own day/night cycles. This function encodes
# a very simple version of that so the network “feels alive”.
if 'get_traffic_rate_lambda' not in globals():
    def get_traffic_rate_lambda(region: str, utc_hour: int) -> float:
        """
        Diurnal traffic model (step-function style).
        Returns λ = expected arrival rate based on region + time of day.
        """

        # Normalize input → avoids bugs from lowercase/spacing issues.
        r = (region or "").strip().title()
        h = int(utc_hour) % 24  # wrap hour into [0–23]

        # Each region has a “busy window” (awake) and “quiet window” (asleep).
        # Numbers are chosen heuristically to represent relative intensity.
        if r == "Europe":
            return 6.0 if 7 <= h < 19 else 2.0
        if r == "North America":
            return 5.0 if (h >= 19 or h < 7) else 1.5
        if r == "Asia":
            return 5.5 if 1 <= h < 13 else 2.0
        if r == "Africa":
            return 4.5 if 7 <= h < 19 else 1.5
        if r == "South America":
            return 4.0 if 10 <= h < 22 else 1.5
        if r == "Australia":
            return 3.0 if (h >= 21 or h < 9) else 1.0

        # Fallback: oceans / “other” regions → always low demand
        return 1.0

    get_traffic_rate = get_traffic_rate_lambda  # short alias

# --- 2. Define global simulation knobs ---
# Why: These control the whole experiment environment. Keeping them global
# (and seeded) ensures repeatability across runs.
if 'UTC_HOUR' not in globals():
    # Chosen time of day. 14:00 UTC ≈ Europe afternoon (high demand).
    UTC_HOUR = 14

if 'SIMULATION_DURATION_S' not in globals():
    # Run length of the simulated experiment (in seconds).
    SIMULATION_DURATION_S = 180

if 'AVG_PACKET_SIZE_KB' not in globals():
    # Simplification: treat all packets as ~1.5 KB (typical average size).
    AVG_PACKET_SIZE_KB = 1.5

if 'rng' not in globals():
    # Random generator for traffic arrivals. Seeded for reproducibility.
    rng = np.random.default_rng(RANDOM_SEED)

# --- 3. Sanity check ---
# Why: Better to fail here if a parameter is missing than halfway into simulation.
print(f" Traffic model ready: UTC_HOUR={UTC_HOUR}, DURATION={SIMULATION_DURATION_S}s")

assert 'SIMULATION_DURATION_S' in globals()
assert 'rng' in globals()
assert 'get_traffic_rate_lambda' in globals()



---

In [ ]:
# ====================================================================
# Step 22: Generate Traffic Events (Poisson Simulation)
#
# Why this step?
# Now that each satellite has an average traffic rate λ (from Step 21),
# I need to *instantiate* traffic over time. In other words:
#   - Turn average rates into actual packet arrivals.
#   - Use randomness to reflect bursty, real internet flows.
#
# The standard tool for this is a **Poisson process**:
#   - Well-known model for random arrivals in networks/queues.
#   - Produces realistic variation around the mean λ.
#   - Keeps the math tractable for later queuing analysis.
# ====================================================================

print(" Generating traffic events using a Poisson process...")

# --- 1. Ensure RNG is ready ---
# Why: traffic simulation must be reproducible. If rng is missing,
# I reinitialize it with the fixed RANDOM_SEED.
if 'rng' not in globals() or not hasattr(rng, 'poisson'):
    rng = np.random.default_rng(RANDOM_SEED)

# --- 2. Confirm dataset has required columns ---
# Why: I can’t simulate if I don’t know which node, which sat, and which region.
required_cols = ["Node_ID", "Satellite_ID", "Region"]
missing_cols = [col for col in required_cols if col not in master_df.columns]
if missing_cols:
    raise ValueError(f" master_df is missing columns: {missing_cols}")

# --- 3. Attach λ (packets per sec) per node ---
# Why: each satellite inherits its λ from the diurnal model
# so that “busy” regions emit more packets than “quiet” ones.
master_df["lambda_pps"] = master_df["Region"].apply(
    lambda region: get_traffic_rate_lambda(region, UTC_HOUR)
)
master_df["Lambda"] = master_df["lambda_pps"]  # shorthand
assert (master_df["Lambda"] >= 0).all(), " Negative λ found!"

# --- 4. Simulate arrivals second by second ---
# Why: a discrete-event model lets me track packet counts per node over time,
# which I’ll later feed into queueing and routing simulations.
simulation_results = []
N = len(master_df)

for t in range(1, SIMULATION_DURATION_S + 1):
    # Core of Poisson process: for each node, draw packets ~ Poisson(λ).
    # Why Poisson? Because it captures random bursts while preserving
    # the long-run mean rate.
    packet_counts = rng.poisson(lam=master_df["Lambda"].values)

    # Build one row per node for this second t
    second_df = pd.DataFrame({
        "Time": t,
        "Node_ID": master_df["Node_ID"].values,
        "Satellite_ID": master_df["Satellite_ID"].values,
        "Region": master_df["Region"].values,
        "NumPackets_Generated": packet_counts.astype("int32")
    })
    simulation_results.append(second_df)

print(f" Simulation complete. Time simulated: {SIMULATION_DURATION_S} seconds.")

# --- 5. Combine per-second results ---
traffic_events_df = pd.concat(simulation_results, ignore_index=True)

# Optimization: most rows will be zeros (no arrivals).
# Why keep them? They waste disk/memory. I filter them out.
traffic_events_df = traffic_events_df[traffic_events_df["NumPackets_Generated"] > 0].reset_index(drop=True)

# --- 6. Save compressed output ---
# Why gzip? Traffic event files are *huge*. Compression saves space
# and pandas can still load them directly later.
events_path = OUTPUT_DIR / "traffic_events.csv.gz"
traffic_events_df.to_csv(events_path, index=False, compression="gzip")
print(f" Traffic events saved to:\n   {events_path.absolute()}")

# --- 7. Sanity check packet counts ---
expected_packets = master_df["Lambda"].sum() * SIMULATION_DURATION_S
actual_packets = int(traffic_events_df["NumPackets_Generated"].sum())
print(f" Total packets: expected ≈ {expected_packets:.1f} | actual = {actual_packets}")
assert (traffic_events_df["NumPackets_Generated"] >= 0).all()
print(" All packet counts are non-negative integers.")

# --- 8. Quick preview ---
print("\n Sample of Generated Traffic Events:")
display(traffic_events_df.head())


---

In [ ]:
# ====================================================================
# Step 23: Analyze and Visualize Traffic Simulation Results
#
# Why this step?
# Traffic generation (Step 22) produced raw packet events, but raw logs
# don’t tell me much. I need to:
#   - Validate: did the Poisson arrivals behave as expected (λ ≈ λ̂)?
#   - Summarize: how much traffic did each region produce on average?
#   - Visualize: create a bar chart that clearly shows regional load.
#
# This is both a **debugging checkpoint** and a **result I can use in my thesis**.
# ====================================================================

print(" Analyzing and visualizing simulation results...")

# --- 1. Load events file ---
# Why: I can’t analyze traffic until I bring the generated events back into memory.
if 'traffic_events_df' not in globals():
    gz_path = OUTPUT_DIR / "traffic_events.csv.gz"
    csv_path = OUTPUT_DIR / "traffic_events_log.csv"

    if gz_path.exists():
        traffic_events_df = pd.read_csv(gz_path)
        print(f" Loaded traffic events from: {gz_path.name}")
    elif csv_path.exists():
        traffic_events_df = pd.read_csv(csv_path)
        print(f" Loaded traffic events from: {csv_path.name}")
    else:
        raise FileNotFoundError(" No traffic events found. Run the traffic generation step first.")

# --- 2. Ensure Region info is present ---
# Why: my analysis is regional. If the Region column was lost,
# I merge it back from master_df so I can group results by region.
if not {"Region", "Node_ID"}.issubset(traffic_events_df.columns):
    traffic_events_df = traffic_events_df.merge(
        master_df[["Node_ID", "Satellite_ID", "Region"]],
        on="Satellite_ID",
        how="left",
        validate="many_to_one" # one satellite, many events
    )
    print(" Merged Region and Node_ID from master_df.")

# --- 3. Compute λ̂ (observed rate) per node ---
# Why: I want to check whether the Poisson arrivals actually match
# the theoretical λ I fed into the simulation.
T = int(SIMULATION_DURATION_S)

packets_per_node = (
    traffic_events_df
    .groupby("Node_ID")["NumPackets_Generated"]
    .sum()
    .rename("total_packets")
    .reset_index()
)

node_rates = master_df[["Node_ID", "Region"]].merge(
    packets_per_node, on="Node_ID", how="left"
)
node_rates["total_packets"] = node_rates["total_packets"].fillna(0).astype(int)
node_rates["lambda_hat"] = node_rates["total_packets"] / T  # observed λ̂

# --- 4. Aggregate by region ---
# Why: the experiment is global, but analysis is clearer per region.
regional_traffic_summary = (
    node_rates
    .groupby("Region")["lambda_hat"]
    .agg(count="count", mean="mean", sum="sum")
    .reset_index()
    .rename(columns={
        "mean": "Mean_Packets_per_Second",
        "sum": "Total_Packets_per_Second"
    })
    .sort_values("Mean_Packets_per_Second", ascending=False)
)

print(f"\n Mean Traffic Per Region at {UTC_HOUR}:00 UTC")
display(regional_traffic_summary)

# Sanity check: compare input λ vs output λ̂
if "Lambda" in master_df.columns:
    model_means = master_df.groupby("Region")["Lambda"].mean().rename("Model_Mean_Lambda").reset_index()
    comparison_df = model_means.merge(
        regional_traffic_summary[["Region", "Mean_Packets_per_Second"]],
        on="Region", how="left"
    )
    display(comparison_df)  # they should be close (not identical, but close)

# --- 5. Plot results ---
# Why: visuals are easier to communicate in a thesis than raw tables.
plt.figure(figsize=(10, 6))
plt.bar(
    regional_traffic_summary["Region"],
    regional_traffic_summary["Mean_Packets_per_Second"],
    color="skyblue", edgecolor="black"
)
plt.ylabel("Mean Packets per Second (λ̂)")
plt.xlabel("Region")
plt.title(f"Simulated Mean Traffic by Region at {UTC_HOUR}:00 UTC")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()

# --- 6. Save results ---
# Why: these outputs are checkpoints for future steps (routing, queues).
summary_path = OUTPUT_DIR / "traffic_summary_by_region.csv"
regional_traffic_summary.to_csv(summary_path, index=False)
print(f"\n Regional summary saved to:\n   {summary_path.absolute()}")

events_gz = OUTPUT_DIR / "traffic_events.csv.gz"
if events_gz.exists():
    print(f" Full traffic event log already exists at:\n   {events_gz.absolute()}")
else:
    csv_log_path = OUTPUT_DIR / "traffic_events_log.csv"
    traffic_events_df.to_csv(csv_log_path, index=False)
    print(f" Full traffic event log saved to:\n   {csv_log_path.absolute()}")

chart_path = OUTPUT_DIR / "traffic_by_region_barchart.png"
plt.savefig(chart_path, dpi=200, bbox_inches="tight")
print(f" Bar chart saved to:\n   {chart_path.absolute()}")

plt.show()


---

In [ ]:
# ====================================================================
# Step 24: Load and Enrich Simulated Traffic Data
#
# Why this step?
# The Poisson simulator (Step 22) gave me raw event logs:
#   - When each packet was generated
#   - Which satellite produced it
#
# But raw logs alone aren’t enough. For realistic analysis I need to:
#   - Add back metadata (Node_ID, Region, etc.) from the master dataset
#   - Optimize data types (millions of rows → memory footprint matters!)
#   - Sanity check completeness (every event should map to a Region)
#
# This step turns “raw packet events” into a structured dataset that is
# ready for queueing simulations and routing experiments.
# ====================================================================

print(" Loading the simulated traffic event log...")

# --- 1. Load traffic events ---
# Why: Without the events, I have nothing to enrich.
events_gz = OUTPUT_DIR / "traffic_events.csv.gz"
events_csv = OUTPUT_DIR / "traffic_events_log.csv"

try:
    if events_gz.exists():
        traffic_events_df = pd.read_csv(events_gz)
        print(f" Traffic event log loaded from: {events_gz.name}")
    elif events_csv.exists():
        traffic_events_df = pd.read_csv(events_csv)
        print(f" Traffic event log loaded from: {events_csv.name}")
    else:
        raise FileNotFoundError(str(events_csv))
except FileNotFoundError:
    print(f" ERROR: No traffic events file found at '{events_csv}'.")
    print("   → Run the traffic generation step first.")
    raise

# --- 2. Load or rebuild master dataset ---
# Why: I need metadata (Region, Node_ID, etc.) to interpret events.
master_candidates = [
    OUTPUT_DIR / "master_sample_dataframe.csv",
    OUTPUT_DIR / "final_sample_master.csv"
]

master_df = None
for p in master_candidates:
    if p.exists():
        master_df = pd.read_csv(p)
        print(f" Master dataset loaded from: {p.name}")
        break

if master_df is None:
    print(" Master dataset not found, rebuilding it...")
    adj_path = OUTPUT_DIR / "final_subgraph_adjacency_list.csv"
    pos_path = OUTPUT_DIR / "satellite_positions_with_regions.csv"
    if not adj_path.exists() or not pos_path.exists():
        raise FileNotFoundError(" Cannot rebuild master_df — missing adjacency or positions.")

    adj = pd.read_csv(adj_path)
    pos = pd.read_csv(pos_path)

    if "Node_ID" not in adj.columns and "Satellite_ID" in adj.columns:
        adj["Node_ID"] = adj["Satellite_ID"].astype(int) - 1

    keep_cols = [col for col in ["Node_ID", "Satellite_ID", "Region"] if col in pos.columns]
    master_df = adj.merge(pos[keep_cols], on=["Node_ID", "Satellite_ID"], how="left")

    if "Region" not in master_df.columns:
        master_df["Region"] = "Other"

    assert (master_df["Satellite_ID"] == master_df["Node_ID"] + 1).all(), \
        " Mapping error: Satellite_ID != Node_ID + 1"

    print(" Master dataset rebuilt successfully.")

# --- 3. Enrich events with metadata ---
# Why: Events need context (where? which region? which Node_ID?).
traffic_df = traffic_events_df.copy()

if "Region" not in traffic_df.columns:
    traffic_df = traffic_df.merge(
        master_df[["Satellite_ID", "Region"]],
        on="Satellite_ID", how="left", validate="many_to_one"
    )
    print(" Region added from master_df.")

if "Node_ID" not in traffic_df.columns:
    traffic_df = traffic_df.merge(
        master_df[["Satellite_ID", "Node_ID"]],
        on="Satellite_ID", how="left", validate="many_to_one"
    )
    print(" Node_ID added from master_df.")

# --- 4. Optimize data types ---
# Why: Millions of rows → memory & speed matter.
astype_map = {}
for col in ["Time", "Satellite_ID", "Node_ID", "NumPackets_Generated"]:
    if col in traffic_df.columns:
        astype_map[col] = "int32"

if astype_map:
    traffic_df = traffic_df.astype(astype_map)

if "Region" in traffic_df.columns:
    traffic_df["Region"] = traffic_df["Region"].astype("category")

# Sanity check: no missing Region values
if "Region" in traffic_df.columns:
    missing = traffic_df["Region"].isna().sum()
    if missing > 0:
        print(f" Warning: {missing} rows have missing Region values.")

print(" Data types optimized.")

# --- 5. Preview final enriched dataset ---
# Why: Always inspect a slice before trusting a big dataset.
print("\n Enriched Traffic DataFrame (Sample):")
display(traffic_df.head())


---

In [ ]:
# ====================================================================
# Step 25: Plot Smoothed Regional Traffic Time-Series
#
# Why this step?
# The raw events (Step 22) are noisy: each second is a random Poisson draw.
# If I plot those directly, I’d just see jagged spikes.
#
# To understand actual traffic patterns, I need to:
#   1. Compute true per-second *mean* traffic per satellite in each region
#   2. Fill missing seconds so the time-series is continuous
#   3. Smooth the noise with a moving average (10s window)
#
# Result: a clean plot that shows the *trend* of traffic over time,
# not just random fluctuations. This plot is thesis-ready.
# ====================================================================

print(" Plotting the corrected time-series of mean regional traffic...")

# --- 1. Load Traffic Events File ---
# Why: All later steps depend on the per-second traffic counts.
if 'traffic_events_df' not in globals():
    gz_path = OUTPUT_DIR / "traffic_events.csv.gz"
    csv_path = OUTPUT_DIR / "traffic_events_log.csv"
    if gz_path.exists():
        traffic_events_df = pd.read_csv(gz_path)
        print(f" Loaded events from: {gz_path.name}")
    elif csv_path.exists():
        traffic_events_df = pd.read_csv(csv_path)
        print(f" Loaded events from: {csv_path.name}")
    else:
        raise FileNotFoundError(" Traffic events not found. Run the simulation first.")

# --- 2. Load Master DataFrame (for Region info) ---
# Why: I need Region membership to group satellites correctly.
if 'master_df' not in globals():
    for p in [OUTPUT_DIR / "master_sample_dataframe.csv", OUTPUT_DIR / "final_sample_master.csv"]:
        if p.exists():
            master_df = pd.read_csv(p)
            print(f" Loaded master_df from: {p.name}")
            break
    else:
        adj_path = OUTPUT_DIR / "final_subgraph_adjacency_list.csv"
        pos_path = OUTPUT_DIR / "satellite_positions_with_regions.csv"
        if adj_path.exists() and pos_path.exists():
            adj = pd.read_csv(adj_path)
            pos = pd.read_csv(pos_path)
            if "Node_ID" not in adj.columns and "Satellite_ID" in adj.columns:
                adj["Node_ID"] = adj["Satellite_ID"].astype(int) - 1
            master_df = adj.merge(pos[["Node_ID", "Satellite_ID", "Region"]],
                                  on=["Node_ID", "Satellite_ID"], how="left")
            print("🔧 Rebuilt master_df from adjacency and positions.")
        else:
            raise FileNotFoundError(" master_df not available and cannot rebuild.")

# --- 3. Enrich Traffic Events with Region ---
# Why: Events must carry Region info to aggregate properly.
traffic_df = traffic_events_df.copy()
if "Region" not in traffic_df.columns:
    traffic_df = traffic_df.merge(
        master_df[["Satellite_ID", "Region"]],
        on="Satellite_ID", how="left", validate="many_to_one"
    )

# --- 4. Compute True Per-Second Mean Traffic per Region ---
# Why: Normalize by #satellites per region to avoid bias.
region_counts = master_df.groupby("Region")["Node_ID"].nunique().sort_index()

# Total packets per region per second
packet_sums = traffic_df.groupby(["Time", "Region"])["NumPackets_Generated"].sum().unstack(fill_value=0)

# Reindex to ensure continuous time-series
T = int(SIMULATION_DURATION_S) if 'SIMULATION_DURATION_S' in globals() else traffic_df["Time"].max()
time_index = pd.Index(range(1, T + 1), name="Time")
packet_sums = packet_sums.reindex(index=time_index, fill_value=0)
packet_sums = packet_sums.reindex(columns=sorted(region_counts.index.tolist()), fill_value=0)

# Divide totals by #satellites per region → per-satellite mean traffic
true_mean_timeseries = packet_sums.div(region_counts, axis=1)

# --- 5. Apply Moving Average (10s) ---
# Why: Smooths Poisson noise → reveals underlying trends.
smoothed_timeseries_df = true_mean_timeseries.rolling(window=10, min_periods=1).mean()

# --- 6. Plot the Time-Series ---
fig, ax = plt.subplots(figsize=(14, 7))
smoothed_timeseries_df.plot(ax=ax, linewidth=2.0)

ax.set_title(f"Mean Traffic Over Time by Region (UTC {UTC_HOUR}:00)", fontsize=16)
ax.set_xlabel("Time (seconds into simulation)")
ax.set_ylabel("Avg Packets per Second per Satellite")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(title="Region")
plt.tight_layout()

# --- 7. Save Outputs ---
plot_path = OUTPUT_DIR / "traffic_timeseries_smoothed.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
print(f"\n Smoothed time-series plot saved to:\n   {plot_path.absolute()}")

true_csv = OUTPUT_DIR / "traffic_timeseries_true_mean.csv"
smoothed_csv = OUTPUT_DIR / "traffic_timeseries_smoothed.csv"
true_mean_timeseries.to_csv(true_csv, float_format="%.6f")
smoothed_timeseries_df.to_csv(smoothed_csv, float_format="%.6f")
print(f" Time-series data saved to:\n   - {true_csv.name}\n   - {smoothed_csv.name}")

plt.show()


---

In [ ]:
# ====================================================================
# Step 26: Plot Stacked Area Chart of Total Regional Traffic
#
# Why this step?
# The smoothed line chart (Step 25) showed *per-satellite mean traffic*,
# but now I want to see how much each region contributes to the *total*
# global traffic load. 
#
# A stacked area chart is ideal:
#   - Each band = one region’s traffic
#   - Band thickness = that region’s contribution
#   - Top edge of the stack = total network traffic
#
# This lets me answer: “Who dominates global load at this time of day?”
# ====================================================================

print(" Plotting the stacked area chart of total regional traffic...")

# --- 1. Load Traffic Events (if needed) ---
# Why: I need the raw per-event data, enriched with Region, to build totals.
if 'traffic_df' not in globals():
    gz_path = OUTPUT_DIR / "traffic_events.csv.gz"
    csv_path = OUTPUT_DIR / "traffic_events_log.csv"

    if gz_path.exists():
        traffic_events_df = pd.read_csv(gz_path)
        print(f" Loaded events from: {gz_path.name}")
    elif csv_path.exists():
        traffic_events_df = pd.read_csv(csv_path)
        print(f" Loaded events from: {csv_path.name}")
    else:
        raise FileNotFoundError(" No traffic events found. Run the traffic generation step first.")

    traffic_df = traffic_events_df.copy()

    # If Region info is missing, bring it in from master_df
    if "Region" not in traffic_df.columns:
        master_df = None
        for p in [OUTPUT_DIR / "master_sample_dataframe.csv", OUTPUT_DIR / "final_sample_master.csv"]:
            if p.exists():
                master_df = pd.read_csv(p)
                print(f" Loaded master_df from: {p.name}")
                break

        if master_df is None:
            adj_path = OUTPUT_DIR / "final_subgraph_adjacency_list.csv"
            pos_path = OUTPUT_DIR / "satellite_positions_with_regions.csv"
            if adj_path.exists() and pos_path.exists():
                adj = pd.read_csv(adj_path)
                pos = pd.read_csv(pos_path)
                if "Node_ID" not in adj.columns and "Satellite_ID" in adj.columns:
                    adj["Node_ID"] = adj["Satellite_ID"].astype(int) - 1
                master_df = adj.merge(pos[["Node_ID", "Satellite_ID", "Region"]],
                                      on=["Node_ID", "Satellite_ID"], how="left")
                print(" Rebuilt master_df from adjacency and positions.")
            else:
                raise FileNotFoundError(" master_df not available and cannot rebuild.")

        # Add Region info to traffic_df
        traffic_df = traffic_df.merge(
            master_df[["Satellite_ID", "Region"]],
            on="Satellite_ID", how="left", validate="many_to_one"
        )

# --- 2. Validate Required Columns ---
required_cols = ["Time", "Region", "NumPackets_Generated"]
if not all(col in traffic_df.columns for col in required_cols):
    print(" ERROR: 'traffic_df' is missing required columns. Re-run previous steps.")
else:
    # --- 3. Aggregate Total Traffic by Region ---
    # Why: Summing (not averaging) shows *absolute load* per region.
    grouped = traffic_df.groupby(["Time", "Region"])["NumPackets_Generated"].sum()
    timeseries_total_df = grouped.unstack(fill_value=0)

    # Fill missing seconds so the timeline is continuous
    T = SIMULATION_DURATION_S if 'SIMULATION_DURATION_S' in globals() else traffic_df["Time"].max()
    timeseries_total_df = timeseries_total_df.reindex(index=range(1, T + 1), fill_value=0)

    if timeseries_total_df.empty:
        print(" WARNING: Aggregated data is empty. Skipping plot.")
    else:
        print(" Data prepared for plotting.")

        # --- 4. Plot the Stacked Area Chart ---
        plt.figure(figsize=(12, 7))
        ax = plt.gca()
        timeseries_total_df.plot.area(
            ax=ax,
            stacked=True,        # key: bands add up to global total
            linewidth=0.5,
            alpha=0.8
        )

        if 'UTC_HOUR' not in globals():
            UTC_HOUR = 14
        ax.set_title(f"Composition of Total Network Traffic by Region (UTC {UTC_HOUR}:00)", fontsize=16)
        ax.set_xlabel("Time (seconds into simulation)")
        ax.set_ylabel("Total Packets Generated per Second")
        ax.grid(True, linestyle="--", alpha=0.6)
        ax.legend(title="Region")
        plt.tight_layout()

        # --- 5. Save Outputs ---
        plot_path = OUTPUT_DIR / "traffic_stacked_area_by_region.png"
        plt.savefig(plot_path, dpi=200, bbox_inches="tight")
        print(f"\n Stacked area chart saved to:\n   {plot_path.absolute()}")

        totals_csv = OUTPUT_DIR / "traffic_totals_timeseries_by_region.csv"
        timeseries_total_df.to_csv(totals_csv, float_format="%.6f")
        print(f" Aggregated time-series saved to:\n   {totals_csv.absolute()}")

        plt.show()


---

In [ ]:
# ====================================================================
# Step 27: Final Sanity Checks on Simulation Results
#
# Why this step?
# The traffic generator is stochastic (Poisson), so small mismatches are expected.
# But big mismatches or structural errors (too many rows, wrong scale) would
# undermine everything downstream. 
#
# These final checks give me confidence that:
#   1. The simulation is producing traffic at the right *scale* (λ̂ ≈ λ).
#   2. The dataset shape is logically consistent (rows ≤ T × N).
# ====================================================================


# --- 1. Compare total simulated traffic (λ̂) vs expected model (λ) ---
# Why: This validates that the Poisson generator is “on target”.
if "Lambda" in master_df.columns:
    lambda_hat_sum = node_rates["lambda_hat"].sum()   # observed average rate from events
    lambda_model_sum = master_df["Lambda"].sum()      # theoretical rate from model

    print(f" Sum of simulated λ̂ (from events): {lambda_hat_sum:.4f}")
    print(f" Sum of expected λ (from model):    {lambda_model_sum:.4f}")
    print(" Difference:", abs(lambda_hat_sum - lambda_model_sum))
    # Expectation: close but not exact — random fluctuations are fine.
    # If the gap is large, it’s a red flag: maybe λ was misapplied.


# --- 2. Check dataset size vs theoretical maximum ---
# Why: At most, each of N satellites can produce one row per second (T seconds).
# So the total number of rows must be ≤ T × N.
N = len(master_df)              # satellites in the sample
T = SIMULATION_DURATION_S       # seconds simulated
max_possible_rows = T * N
actual_rows = len(traffic_events_df)

print(f"\n Number of traffic event rows: {actual_rows}")
print(f" Max possible rows (T × N):    {max_possible_rows}")
print(f" Check: {actual_rows} <= {max_possible_rows} ? →", actual_rows <= max_possible_rows)
# If this fails, something is fundamentally wrong with the logging logic.


---
---

## Phase 5: Queue Simulation in the Normal Scenario and Results Analysis

In [ ]:
# ====================================================================
# Step 28: Initialize Queueing Simulation
#
# Why this step?
# The Poisson traffic log is in a long table (events per sat/time).
# A queueing simulation needs a dense time × satellite matrix, plus
# pre-allocated arrays for queue length and drops. This step reshapes
# and validates everything so the actual queue simulation (next step)
# runs fast and error-free.
# ====================================================================


print(" Initializing the queueing simulation...")

# --- 1. Load Traffic Events ---
# Why: Without the event log, there is no demand to simulate. 
# The loader is flexible (CSV or GZ) so the simulation can always start.
events_gz = OUTPUT_DIR / "traffic_events.csv.gz"
events_csv = OUTPUT_DIR / "traffic_events_log.csv"

try:
    if events_csv.exists():
        traffic_events_df = pd.read_csv(events_csv)
        print(f" Traffic events loaded from: {events_csv.name}")
    elif events_gz.exists():
        traffic_events_df = pd.read_csv(events_gz)
        print(f" Traffic events loaded from: {events_gz.name}")
    else:
        raise FileNotFoundError(" Could not find traffic events CSV or GZ file.")

    # Load master_df for satellite info — needed for consistent sat list
    master_df = None
    for path in [OUTPUT_DIR / "master_sample_dataframe.csv", OUTPUT_DIR / "final_sample_master.csv"]:
        if path.exists():
            master_df = pd.read_csv(path)
            print(f" Loaded satellite master info from: {path.name}")
            break
    
    # If not found, rebuild from adjacency + positions
    if master_df is None:
        adj_path = OUTPUT_DIR / "final_subgraph_adjacency_list.csv"
        pos_path = OUTPUT_DIR / "satellite_positions_with_regions.csv"
        if adj_path.exists() and pos_path.exists():
            adj = pd.read_csv(adj_path)
            pos = pd.read_csv(pos_path)
            if "Node_ID" not in adj.columns and "Satellite_ID" in adj.columns:
                adj["Node_ID"] = adj["Satellite_ID"].astype(int) - 1
            master_df = adj.merge(
                pos[["Node_ID", "Satellite_ID", "Region"]],
                on=["Node_ID", "Satellite_ID"],
                how="left",
                validate="one_to_one"
            )
            print(" Rebuilt master_df from adjacency and positions.")
        else:
            raise FileNotFoundError(" Master DF not found and cannot be rebuilt.")

except FileNotFoundError as e:
    print(e)
    raise


# --- 2. Validate Required Columns ---
# Why: The simulation logic relies on these three fields being present.
required_cols = {"Time", "Satellite_ID", "NumPackets_Generated"}
missing = required_cols - set(traffic_events_df.columns)
assert not missing, f" Missing columns in traffic_events_df: {missing}"
assert "Satellite_ID" in master_df.columns, " master_df is missing Satellite_ID"


# --- 3. Determine Simulation Duration ---
# Why: Need to know T (number of steps) in advance to size arrays properly.
if 'SIMULATION_DURATION_S' in globals():
    num_seconds = int(SIMULATION_DURATION_S)
else:
    config_path = OUTPUT_DIR / "traffic_config.json"
    if config_path.exists():
        try:
            cfg = json.loads(config_path.read_text())
            num_seconds = int(cfg.get("SIMULATION_DURATION_S", traffic_events_df["Time"].max()))
        except:  # fallback if JSON parsing fails
            num_seconds = int(traffic_events_df["Time"].max())
    else:
        num_seconds = int(traffic_events_df["Time"].max())

print(f" Simulation time: {num_seconds} seconds.")


# --- 4. Create Arrival Matrix (Time × Satellite) ---
# Why: Step-by-step simulation is far more efficient if arrivals are in a
# dense [time, sat] matrix. Pivoting also ensures every (t, sat) slot exists.
print(" Creating arrival matrix...")

time_index = pd.Index(range(1, num_seconds + 1), name="Time")
arrivals_df = traffic_events_df.pivot_table(
    index="Time",
    columns="Satellite_ID",
    values="NumPackets_Generated",
    aggfunc="sum",   # combine duplicates within same (t, sat)
    fill_value=0
).reindex(index=time_index, fill_value=0)

# Guarantee all satellites appear as columns, even if they never generated traffic.
all_sat_ids = sorted(master_df['Satellite_ID'].unique())
arrivals_df = arrivals_df.reindex(columns=all_sat_ids, fill_value=0)
arrivals_df = arrivals_df.astype("int32")  # compact & fast


# --- 5. Initialize Queue and Drop Tracking ---
# Why: Pre-allocate state arrays so the simulation loop can update in-place.
num_satellites = len(all_sat_ids)
queue_lengths = np.zeros((num_seconds + 1, num_satellites), dtype=np.int32)
dropped_packets = np.zeros((num_seconds + 1, num_satellites), dtype=np.int32)

# Parameters: router buffers and service capacity
BUFFER_CAPACITY_PACKETS = 20000
SERVICE_RATE_PPS = 1000
print(f" Queues initialized with buffer = {BUFFER_CAPACITY_PACKETS}, service rate = {SERVICE_RATE_PPS} pps")


# --- 6. Sanity Check: Matrix Sum vs Raw Log ---
# Why: If totals don’t match, something was lost in pivoting. Must match exactly.
raw_sum = int(traffic_events_df["NumPackets_Generated"].sum())
matrix_sum = int(arrivals_df.values.sum())
print(f" Total packets — Raw = {raw_sum}, Matrix = {matrix_sum} → {'OK' if raw_sum == matrix_sum else 'Mismatch!'}")


# --- 7. Map Satellite_ID → Column Index ---
# Why: Simulation loop will need O(1) access to columns via integer indices.
sat_id_to_col = {sid: j for j, sid in enumerate(all_sat_ids)}


# --- 8. Preview Arrival Matrix ---
# Why: A quick visual confirmation before launching into heavy simulation.
print("\n Arrival matrix preview (first few seconds):")
display(arrivals_df.head())


---

In [ ]:
# ====================================================================
# Step 29: Run the Queueing Simulation
#
# Why this step?
# Generating traffic was only half the story — now I need to see how
# that traffic interacts with limited buffers and service rates. This
# step evolves the state of every satellite queue second-by-second,
# tracking arrivals, service, and drops. Without this, I wouldn’t know
# whether my constellation can actually carry the generated demand.
# ====================================================================

print(" Running the queueing simulation...")

# --- 1. Initialize the Queue State ---
# Why: Start with empty buffers, pre-allocate arrays for efficiency.
q_previous = np.zeros(num_satellites, dtype=np.int32)

queue_lengths   = np.zeros((num_seconds + 1, num_satellites), dtype=np.int32)
dropped_packets = np.zeros((num_seconds + 1, num_satellites), dtype=np.int32)
serviced_packets= np.zeros((num_seconds + 1, num_satellites), dtype=np.int32)


# --- 2. Run the Simulation Loop (one second at a time) ---
# Why: Each second, queues evolve via service → arrival → drop.
for t in range(num_seconds):
    arrivals = arrivals_df.iloc[t].to_numpy(dtype=np.int32, copy=False)

    # A) Serve backlog first — ensures old packets leave before new ones join.
    served = np.minimum(q_previous, SERVICE_RATE_PPS)

    # B) Remove served packets.
    q_after_service = q_previous - served

    # C) Add new arrivals on top of what remains.
    q_after_arrival = q_after_service + arrivals

    # D) Detect buffer overflow. Excess beyond capacity = drops.
    excess  = q_after_arrival - BUFFER_CAPACITY_PACKETS
    dropped = np.maximum(0, excess)

    # E) Final queue = arrivals admitted, capped at capacity.
    q_next = np.minimum(q_after_arrival, BUFFER_CAPACITY_PACKETS)

    # Record metrics for this second.
    queue_lengths[t + 1]   = q_next
    dropped_packets[t + 1] = dropped
    serviced_packets[t + 1]= served

    # Quick invariant check (only first 3 sec to save runtime):
    # (old queue - served + arrivals - dropped) must equal new queue.
    if t < 3:
        expected = q_previous - served + arrivals - dropped
        assert np.array_equal(q_next, expected), f" Queue math error at t={t}"

    # Update state for next second.
    q_previous = q_next

print(" Simulation finished.")


# --- 3. Convert to Long Format DataFrames ---
# Why: Wide arrays are efficient, but long format is easier for analysis/plots.
print(" Formatting results...")

queue_df = pd.DataFrame(queue_lengths[1:], index=arrivals_df.index, columns=all_sat_ids)
queue_df_long = queue_df.reset_index().melt(id_vars="Time", var_name="Satellite_ID", value_name="Queue_Length")

drop_df = pd.DataFrame(dropped_packets[1:], index=arrivals_df.index, columns=all_sat_ids)
drop_df_long = drop_df.reset_index().melt(id_vars="Time", var_name="Satellite_ID", value_name="Packets_Dropped")

serve_df = pd.DataFrame(serviced_packets[1:], index=arrivals_df.index, columns=all_sat_ids)
serve_df_long = serve_df.reset_index().melt(id_vars="Time", var_name="Satellite_ID", value_name="Packets_Served")

simulation_results_df = (
    queue_df_long
    .merge(drop_df_long, on=["Time", "Satellite_ID"])
    .merge(serve_df_long, on=["Time", "Satellite_ID"])
)


# --- 4. Save to Disk ---
# Why: Second-by-second queue states are a *primary dataset* for congestion analysis.
out_path = OUTPUT_DIR / "queue_simulation_results.csv.gz"
simulation_results_df.to_csv(out_path, index=False, compression="gzip")
print(f" Results saved to: {out_path.absolute()}")


# --- 5. Summary ---
# Why: High-level KPIs (drop %, mean queue, p95 queue) tell the story of congestion.
total_arrivals = int(arrivals_df.sum().sum())
total_dropped  = int(dropped_packets.sum())
drop_percent   = 100.0 * total_dropped / max(1, total_arrivals)

mean_q = float(queue_lengths[1:].mean())
p95_q  = float(np.percentile(queue_lengths[1:], 95))

print(f" Queueing Summary:")
print(f"   - Total Arrivals: {total_arrivals:,}")
print(f"   - Dropped Packets: {total_dropped:,} ({drop_percent:.3f}%)")
print(f"   - Average Queue Length: {mean_q:.2f}")
print(f"   - 95th Percentile Queue Length: {p95_q:.0f}")


# --- 6. Preview ---
# Why: Sanity check a sample of the final table before moving to plots/analysis.
print("\n Sample of Queue Simulation Results:")
display(simulation_results_df.head())


---

In [ ]:
# ====================================================================
# Step 30: Analyzing Queue Simulation Results
#
# Why this step?
# Step 29 produced the raw queue dynamics (per-satellite, per-second).
# That dataset is powerful but too detailed to interpret directly.
# Step 30 enriches and aggregates it — adding context (regions, traffic
# rates, arrivals), then plotting the KPIs (queue length, drops).
# This is the first step where I can *see congestion emerge*.
# ====================================================================

print(" Analyzing queue simulation results...")

# --- 1. Load queue simulation results ---
# Why: Bring in the per-satellite, per-second queue/drops/serves log.
results_file = OUTPUT_DIR / "queue_simulation_results.csv.gz"
simulation_results_df = pd.read_csv(results_file)

# --- 2. Load satellite metadata (region, ID mapping) ---
# Why: Without region info, I can’t interpret traffic patterns.
master_file = OUTPUT_DIR / "master_sample_dataframe.csv"
master_df = pd.read_csv(master_file)

# --- 3. Add traffic model (lambda per region) ---
# Why: Theoretical λ values are the baseline — they tell me how much
# traffic each region was *supposed* to generate. I’ll compare this
# to what actually flowed and queued.
def get_traffic_rate_lambda(region, utc_hour=14):
    region = (region or "").strip().title()
    h = utc_hour % 24
    return {
        "Europe":        6.0 if 7 <= h < 19 else 2.0,
        "North America": 5.0 if h >= 19 or h < 7 else 1.5,
        "Asia":          5.5 if 1 <= h < 13 else 2.0,
        "Africa":        4.5 if 7 <= h < 19 else 1.5,
        "South America": 4.0 if 10 <= h < 22 else 1.5,
        "Australia":     3.0 if h >= 21 or h < 9 else 1.0,
    }.get(region, 1.0)  # Default for "Other"

master_df["Lambda"] = master_df["Region"].apply(lambda r: get_traffic_rate_lambda(r, utc_hour=14))

# --- 4. Merge simulation results with region info ---
# Why: Enrich per-satellite queue states with region + λ, so I can
# aggregate by geography or compare to expected traffic.
merged_df = simulation_results_df.merge(
    master_df[["Satellite_ID", "Region", "Lambda"]],
    on="Satellite_ID", how="left"
)

# --- 5. Add packet generation logs ---
# Why: Queue state alone doesn’t show demand. By merging in arrivals
# (NumPackets_Generated), I get the full picture:
#   arrivals → queue → service → drops.
events_file = OUTPUT_DIR / "traffic_events.csv.gz"
events_df = pd.read_csv(events_file)

merged_df = merged_df.merge(
    events_df[["Time", "Satellite_ID", "NumPackets_Generated"]],
    on=["Time", "Satellite_ID"], how="left"
).fillna({"NumPackets_Generated": 0})

# --- 6. Plot: System-level KPIs ---
# Why: Two key congestion signals are:
#   (a) average queue length (pressure building in the system)
#   (b) total drops per second (hard failures under overload).
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
fig.suptitle("Queueing Behavior Over Time (UTC 14:00)", fontsize=16)

# Plot 1: Average queue length (mean across all sats).
avg_queue = merged_df.groupby("Time")["Queue_Length"].mean()
avg_queue.plot(ax=ax1, linewidth=1.5)
ax1.set_title("Average Queue Length Across All Satellites")
ax1.set_ylabel("Queue Length")
ax1.grid(True, linestyle="--", alpha=0.6)

# Plot 2: Total drops (sum across all sats).
total_drops = merged_df.groupby("Time")["Packets_Dropped"].sum()
total_drops.plot(ax=ax2, color='red', linewidth=1.5)
ax2.set_title("Total Packets Dropped Per Second")
ax2.set_xlabel("Time (seconds)")
ax2.set_ylabel("Dropped Packets")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout(rect=[0, 0, 1, 0.95])

# --- 7. Save enriched dataset + plots ---
merged_df.to_csv(OUTPUT_DIR / "full_simulation_results.csv", index=False)
plt.savefig(OUTPUT_DIR / "queueing_performance_plot.png", dpi=200)
plt.show()


---

## Phase 6: Feature Engineering for Machine Learning

In [ ]:
# ===========================================================
# Step 31: Feature Engineering
#
# Why this step?
# I'm turning raw simulation output (queues, drops, arrivals)
# into meaningful features that capture the satellite's state.
# These will become inputs for ML models that make routing or
# congestion control decisions.
# ===========================================================

print(" Starting feature engineering...")

# --- 1. Model Config ---
# These constants reflect physical constraints and thresholds.
BUFFER_CAPACITY = 20000      # Max number of packets a queue can hold
SERVICE_RATE    = 1000       # Packets a satellite can serve per second
BUSY_THRESHOLD  = 5          # Used to mark a node as "busy" in binary form

# --- 2. Load Simulation Results ---
# Using a fallback pattern for robustness — handles older steps or crashes.
primary_path = OUTPUT_DIR / "full_simulation_results.csv"
backup_path  = OUTPUT_DIR / "queue_simulation_results.csv.gz"

if primary_path.exists():
    df = pd.read_csv(primary_path)
    print(f" Loaded results from {primary_path.name}")
elif backup_path.exists():
    df = pd.read_csv(backup_path)
    print(f" Loaded fallback results from {backup_path.name}")
else:
    raise FileNotFoundError(" No simulation results found!")

# --- 3. Check Required Columns ---
# If essential fields like queues and drops are missing, modeling can't proceed.
required = {"Time", "Satellite_ID", "Queue_Length", "Packets_Dropped"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Handle older datasets that didn't log traffic arrivals.
if "NumPackets_Generated" not in df.columns:
    df["NumPackets_Generated"] = 0

# --- 4. Single-Second Static Features ---
# Each one captures something about *right now*, at time t:

# A) Normalized queue size → queue occupancy from 0.0 to 1.0
df["Queue_Ratio"] = df["Queue_Length"] / BUFFER_CAPACITY
df["Queue_Ratio"] = df["Queue_Ratio"].clip(0, 1)  # Safety bound

# B) Is the node actively receiving traffic? (binary)
df["Is_Busy"] = (df["NumPackets_Generated"] > BUSY_THRESHOLD).astype("int8")

# C) Did any drops occur? (binary congestion signal)
df["Did_Drop_Packets"] = (df["Packets_Dropped"] > 0).astype("int8")

# D) Processor utilization, if available
if "Packets_Served" in df.columns:
    df["Served_Ratio"] = df["Packets_Served"] / SERVICE_RATE
    df["Served_Ratio"] = df["Served_Ratio"].clip(0, 1)
else:
    df["Packets_Served"] = 0
    df["Served_Ratio"]   = 0.0

# E) Arrival rate vs. service capacity
# If >1.0 → system is overloaded
df["Arrival_to_Service_Ratio"] = df["NumPackets_Generated"] / SERVICE_RATE
df["Arrival_to_Service_Ratio"] = df["Arrival_to_Service_Ratio"].clip(lower=0)

# F) Proxy delay = queue size / service rate (in seconds)
df["Queueing_Delay_Proxy_s"] = df["Queue_Length"] / SERVICE_RATE

# --- 5. Rolling Features (past trend over last 5s) ---
# These give the model a short memory: "is this satellite's condition worsening?"

# Order matters: must sort by sat ID and then time
df = df.sort_values(["Satellite_ID", "Time"]).reset_index(drop=True)

def past_rolling_mean(data, column):
    # groupby ensures we treat each satellite separately.
    # shift(1) ensures we don't leak future info (only use t−1 to t−5)
    return data.groupby("Satellite_ID")[column] \
               .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())

# Rolling averages of key variables:
df["Roll5_Q_Mean"]       = past_rolling_mean(df, "Queue_Length")         # Smoothed congestion
df["Roll5_Drop_Mean"]    = past_rolling_mean(df, "Packets_Dropped")      # Smoothed drops
df["Roll5_Arrival_Mean"] = past_rolling_mean(df, "NumPackets_Generated") # Smoothed load

# --- 6. Sanity Checks ---
# These guarantee the engineered features are usable and meaningful.
assert df["Queue_Ratio"].between(0, 1).all(), "Queue ratio out of bounds!"

for flag in ["Is_Busy", "Did_Drop_Packets"]:
    values = df[flag].dropna().unique()
    assert set(values).issubset({0, 1}), f"{flag} has non-binary values: {values}"

# --- 7. Save to Disk ---
# This dataset will be used in Phase 10 and beyond (supervised ML training)
feature_path = OUTPUT_DIR / "features_dataset.csv"
df.to_csv(feature_path, index=False, float_format="%.6f")
print(f"\n Features saved to: {feature_path.absolute()}")

# --- 8. Preview the Features ---
# A quick view of the final dataset (raw + engineered fields)
print("\n Preview of Engineered Features:")
display(df[[
    "Time", "Satellite_ID", "NumPackets_Generated", "Packets_Served",
    "Queue_Length", "Packets_Dropped",
    "Queue_Ratio", "Is_Busy", "Did_Drop_Packets",
    "Arrival_to_Service_Ratio", "Queueing_Delay_Proxy_s",
    "Roll5_Q_Mean", "Roll5_Drop_Mean", "Roll5_Arrival_Mean"
]].head())


---

In [ ]:
# ===========================================================
# Step 32: Analyze Feature Distributions
#
# Why this step?
# Feature quality directly affects model quality.
# I’m verifying normalization, checking binary imbalance, and
# getting a feel for the data before any modeling or label generation.
# ===========================================================

# --- 1. Queue Ratio Range ---
# Why?
# The ML model assumes normalized inputs — this confirms that `Queue_Ratio` is in [0,1]
print("Queue_Ratio stats:")
print("   Min:", df["Queue_Ratio"].min())  # Expected ≈ 0.0
print("   Max:", df["Queue_Ratio"].max())  # Expected ≈ 1.0

# --- 2. Binary Flag Distributions ---
# Why?
# If features like `Is_Busy` or `Did_Drop_Packets` are too imbalanced (e.g. 99% zeros),
# they’ll dominate the loss function or make the model biased toward the majority class.

print("\nBusy Status Counts:")
print("  ", df["Is_Busy"].value_counts().to_dict())

# Interpretation:
# → Shows how often satellites were considered “busy”.
# → If the 1s are rare (e.g., <10%), you’ll need sampling or class weights during training.

print("\nDrop Status Counts:")
print("  ", df["Did_Drop_Packets"].value_counts().to_dict())

# Interpretation:
# → Drops are *very rare*, as they represent severe congestion.
# → This feature may act as a proxy “alarm bell”.
# → You might use it as a **label** in classification, or weight it higher in cost-sensitive training.



---

In [ ]:
# ===========================================================
# Step 33 — Engineer a Time-Aware "Delay_Trend" Feature
# This feature captures how delay is *changing over time*
# for each satellite, using only past information (no leakage).
# ===========================================================

print(" Engineering the 'Delay_Trend' feature...")

# --- 1. Load the Existing Features File ---
# Try different paths in case previous steps saved to backup files
feature_paths = [
    OUTPUT_DIR / "features_dataset.csv",
    OUTPUT_DIR / "full_simulation_results.csv",
    OUTPUT_DIR / "queue_simulation_results.csv.gz"
]

features_df = None
for path in feature_paths:
    if path.exists():
        features_df = pd.read_csv(path)
        print(f" Loaded features from: {path.name}")
        break

if features_df is None:
    raise FileNotFoundError(" Could not find any features or results file.")

# --- 2. Setup Parameters ---
# These define how we convert queue info into delay estimates.
SERVICE_RATE = 1000      # packets per second → used for delay = queue / rate
TimeTEP = 1.0          # seconds → time resolution of the simulation
ROLLING_WINDOW = 5       # seconds → smooth window for past trend

# --- 3. Instantaneous Delay Estimate ---
# We estimate queueing delay in seconds: Delay = Queue_Length / SERVICE_RATE
# This converts the queue size to "how long a packet waits".
features_df["Delay_s"] = features_df["Queue_Length"] / SERVICE_RATE

# --- 4. Raw Delay Change per Second (Δ delay) ---
# Before computing differences, sort the rows by Satellite_ID and Time
# to ensure the `.diff()` is applied correctly per satellite.
features_df = features_df.sort_values(["Satellite_ID", "Time"]).reset_index(drop=True)

# Compute raw delay change: Δdelay = Delay(t) - Delay(t−1)
# This captures if the delay is increasing (positive) or decreasing (negative)
features_df["Delay_Trend_Raw"] = (
    features_df
    .groupby("Satellite_ID")["Delay_s"]
    .diff()           # Difference per satellite
    .fillna(0.0)      # First row will be NaN → replace with 0
)

# --- 5. Smooth the Raw Trend (using past-only window) ---
# To remove noise, we compute a rolling average over the *past* 5 seconds
# `.shift(1)` ensures no future info leaks into the current value
features_df["Delay_Trend"] = (
    features_df
    .groupby("Satellite_ID")["Delay_Trend_Raw"]
    .transform(lambda s: s.shift(1).rolling(window=ROLLING_WINDOW, min_periods=1).mean())
    .fillna(0.0)  # Still safe for early rows
)

# --- 6. Final Checks ---
# These make sure the feature has no missing or invalid values
assert features_df["Delay_Trend"].isna().sum() == 0, "Delay_Trend has missing values"
assert np.isfinite(features_df["Delay_Trend"]).all(), "Delay_Trend has non-finite values"

# --- 7. Save to Disk ---
# Save the new feature into a fresh dataset for ML training
output_path = OUTPUT_DIR / "features_dataset_with_trend.csv"
features_df.to_csv(output_path, index=False, float_format="%.6f")
print(f" Delay trend feature saved to:\n   {output_path.name}")

# --- 8. Preview Result ---
# Let’s inspect the relationship between the raw queue, estimated delay,
# and the new delay trend features (raw + smoothed)
print("\n Sample of Delay Trend Features:")
display(features_df[[
    "Time", "Satellite_ID", "Queue_Length",  # Raw queue size
    "Delay_s", "Delay_Trend_Raw", "Delay_Trend"  # Engineered delay-based features
]].head())


---

In [ ]:
# =============================================================
# Step 34 — Final Cleanup: Filling Missing Values & Optimizing
# This step finalizes the dataset before ML training:
# - Fill missing values from rolling features (no NaNs allowed)
# - Reduce memory usage (float64 → float32, etc.)
# - Save the cleaned and optimized dataset
# =============================================================

print(" Finalizing and saving the features dataset...")

# --- 1. Fill Missing Values in Rolling Averages ---
# Why? → Rolling features like "Roll5_Q_Mean" may have NaNs for the first few seconds
# (e.g., t=0, t=1, ...). ML models can't train on missing values → must fill them.

rolling_cols = ["Roll5_Q_Mean", "Roll5_Drop_Mean", "Roll5_Arrival_Mean"]

for col in rolling_cols:
    if col in features_df.columns:
        # Why fill with 0.0? Because in the early seconds, there's no history.
        # So, a flat "0.0" trend is both safe and semantically correct.
        features_df[col] = features_df[col].fillna(0.0).astype("float32")

# --- 2. Optimize Data Types for Size & Speed ---
# Why? → Pandas loads all numbers as float64 by default, but:
# - float32 and int32 are MUCH smaller in memory
# - They’re still precise enough for all our features
# - Greatly speeds up training and reduces RAM usage (especially in Colab)

# Reduce int columns (timestamps, queue counts, drop counts, flags)
int_columns = [
    "Time", "Satellite_ID", "Queue_Length",
    "Packets_Dropped", "Packets_Served",
    "Is_Busy", "Did_Drop_Packets"
]

for col in int_columns:
    if col in features_df.columns:
        features_df[col] = features_df[col].astype("int32")

# Reduce float columns (ratios, delays, smoothed features)
float_columns = [
    "Lambda", "NumPackets_Generated", "Queue_Ratio",
    "Served_Ratio", "Arrival_to_Service_Ratio", "Queueing_Delay_Proxy_s",
    "Delay_s", "Delay_Trend_Raw", "Delay_Trend"
]

for col in float_columns:
    if col in features_df.columns:
        features_df[col] = features_df[col].astype("float32")

# --- 3. Save the Clean Final Feature Set ---
# Why? → This will be the main input to your ML model training (e.g., Phase 10+)
# We save it in a reliable, memory-optimized format (plain CSV)
final_path = OUTPUT_DIR / "features_dataset_final.csv"
features_df.to_csv(final_path, index=False)

print(f" Cleaned features saved to:\n   {final_path.absolute()}")


---

In [ ]:

# =============================================================
# Step 35 — Analyze Congestion and Delay Relationship
#
# Why I’m doing this:
# I want to see how buffer occupancy (queue ratio) and traffic load
# (arrival rate) are connected to the delay trend.
# Numbers alone don’t give much intuition, so I’ll visualize them.
#
# Outputs:
#   - 2 plots (histogram + boxplot) to explain the relationship
#   - feature_analysis_summary.csv with key stats + correlation
# =============================================================

print("Analyzing and visualizing the new engineered features...")

# --- 1. Load dataset (be robust) ---
# Try multiple possible filenames — depends on which earlier step was run.
if 'features_df' not in globals():
    feature_files = [
        OUTPUT_DIR / "features_dataset_final.csv",
        OUTPUT_DIR / "features_dataset_with_trend.csv",
        OUTPUT_DIR / "features_dataset.csv",
        OUTPUT_DIR / "full_simulation_results.csv"
    ]
    for path in feature_files:
        if path.exists():
            features_df = pd.read_csv(path)
            print(f"Loaded features from: {path.name}")
            break
    else:
        raise FileNotFoundError("No valid features file found.")

# --- 2. Column sanity check ---
required_columns = {"Queue_Ratio", "NumPackets_Generated", "Delay_Trend"}
missing = required_columns - set(features_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# --- 3. Setup plots ---
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(14, 6))
fig.suptitle("Feature Analysis: Congestion and Delay", fontsize=18)

# === Plot 1 — Histogram: Queue Ratio Distribution ===
# Queue ratio shows how “full” the buffers usually are.
# Clipping between 0–1 avoids weird outliers >1.
ax1.hist(
    features_df["Queue_Ratio"].clip(0, 1),
    bins=50,
    range=(0, 1),
    color="purple",
    alpha=0.7
)
mean_qr = features_df["Queue_Ratio"].mean()
ax1.axvline(mean_qr, color="black", linestyle="--", label=f"Mean = {mean_qr:.4f}")
ax1.set_title("Queue Ratio (Buffer Occupancy)")
ax1.set_xlabel("Queue Ratio (0 = Empty, 1 = Full)")
ax1.set_ylabel("Time Steps")
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

# === Plot 2 — Boxplot: Arrivals vs Delay Trend ===
# Hypothesis: more arrivals → worse delay trend (queues filling up).
# I group arrival counts into bins, then show the distribution
# of delay trend per bin.
arrival_bins = np.arange(0, int(features_df["NumPackets_Generated"].max()) + 2)
arrival_categories = pd.cut(
    features_df["NumPackets_Generated"],
    bins=arrival_bins,
    right=False,
    include_lowest=True
)
grouped = features_df.groupby(arrival_categories, observed=True)["Delay_Trend"]
pairs = [(k, s.dropna().values) for k, s in grouped if len(s) > 0]
pairs.sort(key=lambda t: (t[0].left, t[0].right))

# Prepare boxplot data
keys = [k for k, _ in pairs]
data = [vals for _, vals in pairs]
pos  = [k.left + 0.5 for k in keys]

assert len(data) > 0 and len(data) == len(pos), "No data for boxplot"

# Draw boxplot with means overlaid
ax2.boxplot(
    data,
    positions=pos,
    widths=0.8,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", color="black"),
    medianprops=dict(color="red", linewidth=2),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    flierprops=dict(marker=".", color="gray", alpha=0.3)
)
binned_means = np.array([np.nanmean(v) if len(v) > 0 else np.nan for v in data])
ax2.plot(pos, binned_means, lw=2, label="Mean Delay Trend", color="orange")

ax2.set_title("Arrivals vs. Delay Trend")
ax2.set_xlabel("Packets Arriving per Second")
ax2.set_ylabel("Delay Trend (s/s)")
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend()

# --- 4. Save summary stats ---
# Store a few key numbers to back up the visuals:
#   - mean + 95th percentile of queue ratio
#   - correlation between arrivals and delay trend
summary = {
    "Mean_Queue_Ratio": round(mean_qr, 6),
    "P95_Queue_Ratio": round(np.percentile(features_df["Queue_Ratio"], 95), 6),
    "Pearson_Corr(Arrivals, Delay_Trend)": round(np.corrcoef(
        features_df["NumPackets_Generated"], features_df["Delay_Trend"]
    )[0, 1], 6)
}
summary_path = OUTPUT_DIR / "feature_analysis_summary.csv"
pd.DataFrame([summary]).to_csv(summary_path, index=False)

print(f"\nSummary statistics saved to: {summary_path.name}")
print(summary)

# --- 5. Show plots ---
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()



---
---

## Phase 7: Generation of 'Congestion' and 'Stress' Scenarios and Network Metrics

In [ ]:
# ====================================================================
# Step 36 — High-Load “Congestion” Traffic Model
#
# Why this step?
# I want a stress-test scenario that actually pushes buffers and exposes
# congestion behavior. The earlier diurnal model is realistic but mild;
# this one deliberately cranks λ (packets/s) during regional peak hours.
#
# Design notes:
# - Simple, readable rules per region (off-peak + peak windows).
# - Peaks can cross midnight (e.g., 21→09), so I handle wrap-around.
# - Keep it deterministic via RANDOM_SEED so runs are reproducible.
# - Save the exact config to JSON so I can trace results later.
# ====================================================================

print("Initializing the CONGESTION traffic model...")

# --- 1) Defaults (only set if missing) ---
# I lock these here so I can re-run cells without accidentally changing the scenario.
if 'UTC_HOUR' not in globals():
    UTC_HOUR = 14      # Pick one hour; the rules below will map this hour to a rate per region.
if 'SIMULATION_DURATION_S' not in globals():
    SIMULATION_DURATION_S = 180  # Keep it short while tuning; longer runs explode result sizes.
if 'RANDOM_SEED' not in globals():
    RANDOM_SEED = 42

# I’m not using rng directly here, but I seed anyway to keep any downstream random draws stable.
rng = np.random.default_rng(RANDOM_SEED)

# --- 2) Load (or rebuild) the master satellite table ---
# I only need Satellite_ID + Region (Node_ID is useful for checks but not required for λ).
master_df = None
for path in (OUTPUT_DIR / "master_sample_dataframe.csv", OUTPUT_DIR / "final_sample_master.csv"):
    if path.exists():
        master_df = pd.read_csv(path)
        print(f" Loaded master DataFrame from: {path.name}")
        break

if master_df is None:
    # Fallback rebuild → I prefer to *always* have a way to recreate inputs
    # instead of silently failing because a file went missing.
    adj_file = OUTPUT_DIR / "final_subgraph_adjacency_list.csv"
    pos_file = OUTPUT_DIR / "satellite_positions_with_regions.csv"
    if not adj_file.exists() or not pos_file.exists():
        raise FileNotFoundError("Missing files → cannot rebuild master DataFrame.")

    adj_df = pd.read_csv(adj_file)
    pos_df = pd.read_csv(pos_file)

    # Merge on the most stable keys I expect to exist here.
    # If Node_ID is present, I keep it for sanity checks; otherwise Satellite_ID is enough.
    keep_cols = [c for c in ["Satellite_ID", "Node_ID", "Region"] if c in pos_df.columns]
    master_df = adj_df.merge(pos_df[keep_cols], on=[c for c in ["Satellite_ID", "Node_ID"] if c in keep_cols], how="left")
    print(" Rebuilt master DataFrame from adjacency + positions.")

# --- 3) Normalize region names ---
# I want Regions to match my canonical set; inconsistent text → inconsistent rates.
# If I’ve defined normalize_region_name earlier, I reuse it. Otherwise I keep a tiny alias map.
if 'normalize_region_name' in globals():
    master_df["Region"] = master_df["Region"].map(normalize_region_name)
else:
    _ALIASES = {
        "Oceania": "Australia",
        "Seven Seas (Open Ocean)": "Other",
        "Seven seas (open ocean)": "Other",
        "Ocean": "Other",
        "Antarctica": "Other",
    }
    master_df["Region"] = master_df["Region"].fillna("Other").apply(lambda x: _ALIASES.get(str(x).strip(), str(x).strip()))

# If Node_ID exists, I double-check the Satellite_ID ↔ Node_ID invariant. Quick fail beats silent bugs later.
if {"Satellite_ID", "Node_ID"}.issubset(master_df.columns):
    assert (master_df["Satellite_ID"] == master_df["Node_ID"] + 1).all(), "Satellite_ID must be Node_ID + 1"

# --- 4) High-load rules per region (deliberately aggressive) ---
# I’m intentionally using large peak λ to trigger drops and queuing in later steps.
# Trade-off: too high will inflate CSV sizes. If your disk cries, lower these.
CONGESTION_TRAFFIC_RULES = {
    "Europe":        {"off": 5.0, "peaks": [(7, 19, 50.0)]},
    "North America": {"off": 4.0, "peaks": [(19, 24, 40.0), (0, 7, 40.0)]},
    "Asia":          {"off": 3.0, "peaks": [(1, 13, 35.0)]},
    "Africa":        {"off": 3.0, "peaks": [(7, 19, 25.0)]},
    "South America": {"off": 2.5, "peaks": [(10, 22, 20.0)]},
    "Australia":     {"off": 2.0, "peaks": [(21, 24, 15.0), (0, 9, 15.0)]},
    "Other":         {"off": 2.0, "peaks": []},  # Oceans + misc → keep low so they don’t dominate totals.
}
_DEFAULT_OFF = 2.0

def _hour_in_window(hour: int, start: int, end: int) -> bool:
    """Peak window check that works across midnight. Using half-open intervals [start, end)."""
    h = int(hour) % 24
    return (start <= end and start <= h < end) or (start > end and (h >= start or h < end))

def _congestion_lambda(region: str, hour: int) -> float:
    """Lookup λ for (region, hour). If region not found, fall back to a gentle default."""
    rules = CONGESTION_TRAFFIC_RULES.get(region, {"off": _DEFAULT_OFF, "peaks": []})
    for start, end, rate in rules["peaks"]:
        if _hour_in_window(hour, start, end):
            return float(rate)
    return float(rules["off"])

# --- 5) Apply the model (per-satellite λ) ---
# I store as both lambda_pps and Lambda because downstream steps already expect Lambda.
master_df["lambda_pps"] = master_df["Region"].fillna("Other").apply(lambda r: _congestion_lambda(r, UTC_HOUR))
master_df["Lambda"] = master_df["lambda_pps"].astype("float32")

# Sanity: Regions without rules should get _DEFAULT_OFF (2.0).
if not set(master_df["Region"].unique()).issubset(set(CONGESTION_TRAFFIC_RULES.keys())):
    missing = sorted(set(master_df["Region"].unique()) - set(CONGESTION_TRAFFIC_RULES.keys()))
    print(f" Note: Regions without explicit rules → {_DEFAULT_OFF} pps: {missing}")

# --- 6) Quick summary by region (spot-check the effect size) ---
summary = (
    master_df
    .groupby("Region", dropna=False)["lambda_pps"]
    .agg(count="size", mean_lambda="mean", max_lambda="max")
    .sort_values("mean_lambda", ascending=False)
)
print(f"\n--- Congestion Traffic Summary at {UTC_HOUR:02d}:00 UTC ---")
try:
    display(summary)
except Exception:
    print(summary.to_string())

# --- 7) Persist the exact scenario config (reproducibility matters) ---
config = {
    "scenario": "congestion_highload",
    "UTC_HOUR": int(UTC_HOUR),
    "SIMULATION_DURATION_S": int(SIMULATION_DURATION_S),
    "RANDOM_SEED": int(RANDOM_SEED),
    "CONGESTION_TRAFFIC_RULES": CONGESTION_TRAFFIC_RULES,
}
(OUTPUT_DIR / "traffic_config_congestion.json").write_text(json.dumps(config, indent=2))
print(" Congestion config saved → traffic_config_congestion.json")

# (Optional) I also snapshot the per-satellite λ so later runs can reuse the exact inputs.
lambdas_path = OUTPUT_DIR / "congestion_lambdas_per_satellite.csv"
(master_df[["Satellite_ID", "Region", "Lambda"]]
 .sort_values(["Region", "Satellite_ID"])
 .to_csv(lambdas_path, index=False))
print(f" Per-satellite λ saved → {lambdas_path.name}")


---

In [ ]:
# ====================================================================
# Step 37 — Generate “Congestion” Traffic Events (execution-only)
#
# Why this step?
# I’ve already defined a high-load λ (packets/s) per satellite in Step 36.
# Now I just need to *draw* traffic events from those rates so the rest of
# the pipeline (queues, drops, ML features) can stress-test under load.
#
# Design choices:
# - Vectorized Poisson draws (fast + reproducible).
# - Keep the code execution-only: no modeling decisions here, just use λ.
# - Save both CSV (quick peek) and GZip (space saver) so I don’t regret it later.
# ====================================================================

print("Running the CONGESTION simulation (execution only)...")

# --- 0) Pre-flight checks ---
# If I don’t have the per-satellite λ in memory, this step is meaningless.
if 'master_df' not in globals():
    raise RuntimeError("master_df is missing. Run the traffic model setup (Step 36) first.")

# I use the first λ-like column I can find so the cell is re-runnable even if I renamed earlier.
lambda_col = None
for candidate in ("Congestion_Lambda", "lambda_pps", "Lambda"):
    if candidate in master_df.columns:
        lambda_col = candidate
        break
if lambda_col is None:
    raise RuntimeError("No per-satellite lambda found. Expected one of: 'Congestion_Lambda', 'lambda_pps', 'Lambda'.")

# --- 1) Simulation parameters (fixed here for clarity) ---
# I restate these so it’s obvious what this run used. Keeping them small avoids huge files.
UTC_HOUR = 14               # Should match the hour used to compute λ
SIMULATION_DURATION_S = 180 # 3 minutes is enough to see congestion patterns
RANDOM_SEED = 42            # Deterministic runs are easier to compare
rng = np.random.default_rng(RANDOM_SEED)

# --- 2) Pull vectors out of DataFrame (performance win) ---
# Accessing numpy arrays in a tight loop is much faster than DataFrame columns.
lam_vec = master_df[lambda_col].astype(float).to_numpy()
sat_ids = master_df["Satellite_ID"].astype(np.int32).to_numpy()

# I’d rather fail fast than silently generate garbage.
if not np.isfinite(lam_vec).all():
    raise ValueError("Non-finite lambda values detected.")
if (lam_vec < 0).any():
    raise ValueError("Negative lambda values detected.")

# --- 3) Main draw loop (second-by-second) ---
# I’m keeping it simple: one Poisson vector draw per second.
# Trade-off: building a list of small DataFrames is not the most memory-efficient,
# but it’s straightforward and easy to debug. If this gets too big, I’ll switch
# to chunked writes .
results = []
for t in range(1, SIMULATION_DURATION_S + 1):
    # Single vectorized Poisson draw → one random count per satellite.
    packets = rng.poisson(lam=lam_vec)

    # I store as int32 to keep the file size and memory reasonable.
    results.append(pd.DataFrame({
        "Time": np.int32(t),
        "Satellite_ID": sat_ids,
        "NumPackets_Generated": packets.astype(np.int32)
    }))

print(f" Simulation complete for {SIMULATION_DURATION_S} seconds.")

# --- 4) Combine + save ---
# concat is cleaner than preallocating a giant array here.
congestion_events_df = pd.concat(results, ignore_index=True)

# Enforce dtypes (helps downstream and keeps files smaller).
congestion_events_df = congestion_events_df.astype({
    "Time": "int32",
    "Satellite_ID": "int32",
    "NumPackets_Generated": "int32"
})

# I keep both formats: CSV for quick eyeballing; .gz for disk sanity.
csv_file = OUTPUT_DIR / "congestion_traffic_events_log.csv"
gz_file  = OUTPUT_DIR / "congestion_traffic_events.csv.gz"
congestion_events_df.to_csv(csv_file, index=False)
congestion_events_df.to_csv(gz_file, index=False, compression="gzip")

# --- 5) Sanity check against expectation ---
# For a Poisson process: E[total] = T * sum(λ). The ratio should hover ~1.
expected = SIMULATION_DURATION_S * float(master_df[lambda_col].sum())
actual   = int(congestion_events_df["NumPackets_Generated"].sum())
ratio    = (actual / expected) if expected > 0 else float("nan")

print(f"Sanity check: expected ≈ {expected:.1f}, actual = {actual:,}, ratio = {ratio:.3f}")
print(f"\n Results saved to:\n   {csv_file}\n   {gz_file}")
print("\n--- Sample of Congestion Traffic Events ---")
display(congestion_events_df.head())



---

In [ ]:
# ====================================================================
# Step 38 — Analyze & Visualize Congestion Simulation Results
#
# Why this step?
# I’ve generated a “congestion” traffic log (very high λ per region).
# Now I want to (a) sanity-check that the resulting load looks like
# congestion and (b) produce publication-ready plots + a quick table
# that compares empirical vs. theoretical rates per region.
# ====================================================================

print("Analyzing and visualizing the CONGESTION simulation results...")

# --- 0) Fast fail on prerequisites ---
# If OUTPUT_DIR/master_df aren’t in memory, any merge/paths will break later.
if 'OUTPUT_DIR' not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Run the setup cells first.")
if 'master_df' not in globals():
    raise RuntimeError("master_df is missing. Run the congestion model setup first.")

print("Using OUTPUT_DIR:", OUTPUT_DIR)

# --- 1) Load the congestion events ---
# Prefer the compressed file to save disk/time; fall back to CSV if needed.
events_gz  = OUTPUT_DIR / "congestion_traffic_events.csv.gz"
events_csv = OUTPUT_DIR / "congestion_traffic_events_log.csv"

if events_gz.exists():
    events_df = pd.read_csv(events_gz)
    print(f"Loaded events from: {events_gz.name}")
else:
    # I’m okay with the uncompressed file for quick local looks.
    events_df = pd.read_csv(events_csv)
    print(f"Loaded events from: {events_csv.name}")

# --- 2) Recover the UTC_HOUR used for the run (if possible) ---
# I’d rather the figure title reflect the *actual* hour used when λ was created.
UTC_HOUR = 9  # default if config is missing
cfg_path = OUTPUT_DIR / "traffic_config_congestion.json"
if cfg_path.exists():
    try:
        cfg = json.loads(cfg_path.read_text())
        UTC_HOUR = int(cfg.get("UTC_HOUR", UTC_HOUR))
    except Exception:
        # Not worth failing the analysis over a bad config.
        pass

# --- 3) Attach Region to each event row ---
# I need Region for grouping. This keeps the events table “thin” but informative.
analysis_df = events_df.merge(
    master_df[["Satellite_ID", "Region"]],
    on="Satellite_ID",
    how="left",
    validate="many_to_one"  # Many event rows → one sat row
)

# --- 4) Build a per-second, per-region mean traffic series ---
# Why divide by #sats? Averages are more stable and let me compare regions
# with different sample sizes fairly.
region_counts = (
    master_df.groupby("Region")["Satellite_ID"]
             .nunique()
             .sort_index()
)

# Sum packets per (t, region) → wide frame with regions as columns.
sum_per_t_region = (
    analysis_df
    .groupby(["Time", "Region"])["NumPackets_Generated"]
    .sum()
    .unstack(fill_value=0)
)

# Fill any missing seconds/regions so the time-series is continuous.
T = int(analysis_df["Time"].max())
sum_per_t_region = sum_per_t_region.reindex(index=range(1, T + 1), fill_value=0)
sum_per_t_region = sum_per_t_region.reindex(columns=sorted(region_counts.index.tolist()), fill_value=0)

# Divide by sat count to get mean-per-satellite. If a region had 0 sats
# (shouldn’t happen, but still), avoid divide-by-zero noise.
_safe_counts = region_counts.replace(0, np.nan)
mean_congestion_over_time = sum_per_t_region.div(_safe_counts, axis=1).fillna(0.0)

# --- 5) Plots: time-series + distribution snapshot ---
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))
fig.suptitle(f"Congestion Traffic at {UTC_HOUR:02d}:00 UTC", fontsize=18)

# (a) Mean per region over time — this should sit much higher than the “normal” model.
mean_congestion_over_time.plot(ax=ax1, linewidth=1.5)
ax1.set_title("Mean Traffic per Region Over Time")
ax1.set_xlabel("Time (seconds)")
ax1.set_ylabel("Mean Packets / Second / Satellite")
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.legend(title="Region")

# (b) Histogram of per-(sat,second) packet counts — a quick “does this look congested?” check.
# Trade-off: histogram ignores which satellite it was; that’s fine for a one-glance sanity check.
ax2.hist(analysis_df["NumPackets_Generated"], bins=50, color="coral", edgecolor="black")
ax2.set_title("Distribution of Packet Counts (Per Satellite-Second)")
ax2.set_xlabel("Packets Generated")
ax2.set_ylabel("Frequency")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout(rect=[0, 0, 1, 0.95])

# Save a high-res image for the thesis.
plot_path = OUTPUT_DIR / "congestion_analysis_plots.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
print(f"\nPlots saved to:\n  {plot_path.absolute()}")
plt.show()

# --- 6) Empirical vs. theoretical means (per region) ---
# I want to verify that the observed means align with the λ used to generate them.
# If they’re far apart, something is off in either the model or the generator.
if   "Congestion_Lambda" in master_df.columns:
    theo_series = master_df.groupby("Region")["Congestion_Lambda"].mean().rename("Theo_Mean_Lambda")
elif "lambda_pps" in master_df.columns:
    theo_series = master_df.groupby("Region")["lambda_pps"].mean().rename("Theo_Mean_Lambda")
elif "Lambda" in master_df.columns:
    theo_series = master_df.groupby("Region")["Lambda"].mean().rename("Theo_Mean_Lambda")
else:
    # Not ideal, but I’d rather return *something* than crash here.
    theo_series = pd.Series(dtype=float, name="Theo_Mean_Lambda")

empirical = mean_congestion_over_time.mean().rename("Empirical_Mean_pps")

summary = pd.concat([empirical, theo_series], axis=1)

summary_path = OUTPUT_DIR / "congestion_analysis_summary.csv"
summary.to_csv(summary_path, float_format="%.4f")
print(f"Summary saved to:\n  {summary_path.absolute()}")


---

In [ ]:
# ====================================================================
# Step 39 — Run Queueing Simulation Under CONGESTION
#
# Why this step?
# The congestion traffic model intentionally overloads the network.
# Here I stress-test the queues with a small buffer + low service rate
# to quantify drops and queue build-up under peak load.
# ====================================================================

# --- 0) Minimal prereqs (fail fast) ---
if 'OUTPUT_DIR' not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Run setup first.")
if 'master_df' not in globals():
    raise RuntimeError("master_df is missing. Run the congestion model (λ) step first.")

# --- 1) Load congestion events (prefer gz) ---
events_gz  = OUTPUT_DIR / "congestion_traffic_events.csv.gz"
events_csv = OUTPUT_DIR / "congestion_traffic_events_log.csv"
if events_gz.exists():
    events_df = pd.read_csv(events_gz)
    print(f"Loaded events from: {events_gz.name}")
elif events_csv.exists():
    events_df = pd.read_csv(events_csv)
    print(f"Loaded events from: {events_csv.name}")
else:
    raise FileNotFoundError("No congestion events file found.")

# Validate required columns early (clear errors beat mysterious NaNs later).
req_cols = {"Time", "Satellite_ID", "NumPackets_Generated"}
missing = req_cols - set(events_df.columns)
if missing:
    raise ValueError(f"Events file missing required columns: {missing}")

# Normalize dtypes (smaller ints save memory; also avoids float->int surprises).
events_df["Time"] = events_df["Time"].astype("int32")
events_df["Satellite_ID"] = events_df["Satellite_ID"].astype("int32")
events_df["NumPackets_Generated"] = events_df["NumPackets_Generated"].astype("int32")

# --- 2) Stress parameters (intentionally harsh to trigger congestion) ---
SIMULATION_DURATION_S   = int(events_df["Time"].max())  # simulate full log span
BUFFER_CAPACITY_PACKETS = 1000    # tiny buffer ⇒ easier overflow
SERVICE_RATE_PPS        = 30      # slow service ⇒ queues grow quickly
print(f"Params: T={SIMULATION_DURATION_S}s, Capacity={BUFFER_CAPACITY_PACKETS}, Service={SERVICE_RATE_PPS} pps")

# --- 3) Build arrivals matrix (Time x Satellite) ---
# Why pivot? The vectorized simulator is O(T×N) and much faster on dense arrays.
time_index = pd.Index(range(1, SIMULATION_DURATION_S + 1), name="Time")

# Keep a stable satellite column order from master_df, but include any extra IDs from events (belt & suspenders).
master_sat_ids = set(pd.to_numeric(master_df["Satellite_ID"], errors="coerce").dropna().astype(int).tolist())
event_sat_ids  = set(events_df["Satellite_ID"].unique().tolist())
sat_ids = sorted(master_sat_ids | event_sat_ids)

# Friendly warnings if something looks off.
extra_in_events = event_sat_ids - master_sat_ids
extra_in_master = master_sat_ids - event_sat_ids
if extra_in_events:
    print(f"Warning: {len(extra_in_events)} Satellite_IDs appear in events but not in master_df (they will still be simulated).")
if extra_in_master:
    print(f"Note: {len(extra_in_master)} Satellite_IDs are in master_df but produced no events (columns will be all zeros).")

arrivals_df = (
    events_df
    .pivot_table(index="Time", columns="Satellite_ID",
                 values="NumPackets_Generated", aggfunc="sum", fill_value=0)
    .reindex(index=time_index, fill_value=0)   # ensure every second exists
    .reindex(columns=sat_ids,  fill_value=0)   # ensure every sat exists
    .astype("int32")
)

# Sanity: no packet lost/duplicated during pivot.
raw_sum = int(events_df["NumPackets_Generated"].sum())
matrix_sum = int(arrivals_df.to_numpy().sum())
print(f"Arrivals matrix shape: {arrivals_df.shape} | Raw sum={raw_sum:,} vs Matrix sum={matrix_sum:,} → {'OK' if raw_sum==matrix_sum else 'MISMATCH'}")
display(arrivals_df.head())

# --- 4) Vectorized queueing simulation (time-stepped, all sats in parallel) ---
num_seconds    = SIMULATION_DURATION_S
num_satellites = len(sat_ids)

# Preallocate (fast, predictable memory).
queue_lengths    = np.zeros((num_seconds + 1, num_satellites), dtype=np.int32)
dropped_packets  = np.zeros((num_seconds + 1, num_satellites), dtype=np.int32)
serviced_packets = np.zeros((num_seconds + 1, num_satellites), dtype=np.int32)

q_prev = np.zeros(num_satellites, dtype=np.int32)

for t in range(num_seconds):
    # Pull arrivals for this second as a view (no copy) for speed.
    arrivals = arrivals_df.iloc[t].to_numpy(dtype=np.int32, copy=False)

    # A) Serve backlog from previous second (bounded by service rate).
    served = np.minimum(q_prev, SERVICE_RATE_PPS).astype(np.int32)

    # B/C) Update queue: subtract served, add new arrivals.
    q_after = q_prev - served + arrivals

    # D) Drop overflow above buffer capacity (drops are non-negative by design).
    dropped = np.maximum(0, q_after - BUFFER_CAPACITY_PACKETS).astype(np.int32)

    # E) Cap queue to buffer capacity for the next state.
    q_next = np.minimum(q_after, BUFFER_CAPACITY_PACKETS).astype(np.int32)

    # Store history (t+1 so row 0 stays as initial state).
    queue_lengths[t + 1]    = q_next
    dropped_packets[t + 1]  = dropped
    serviced_packets[t + 1] = served

    # Mass-balance check for first few steps (defensive programming).
    if t < 3:
        if not np.array_equal(q_next, q_prev - served + arrivals - dropped):
            raise RuntimeError(f"Mass-balance failed at t={t}")

    # Roll to next second.
    q_prev = q_next

print("Queueing under CONGESTION completed.")

# --- 5) Tidy results (long format) & save ---
queue_df = pd.DataFrame(queue_lengths[1:], index=time_index, columns=sat_ids)
drop_df  = pd.DataFrame(dropped_packets[1:], index=time_index, columns=sat_ids)
serv_df  = pd.DataFrame(serviced_packets[1:], index=time_index, columns=sat_ids)

queue_long = queue_df.melt(var_name="Satellite_ID", value_name="Queue_Length",    ignore_index=False).reset_index()
drop_long  = drop_df.melt( var_name="Satellite_ID", value_name="Packets_Dropped",  ignore_index=False).reset_index()
serv_long  = serv_df.melt( var_name="Satellite_ID", value_name="Packets_Served",   ignore_index=False).reset_index()

results_df = (
    queue_long
    .merge(drop_long, on=["Time", "Satellite_ID"])
    .merge(serv_long, on=["Time", "Satellite_ID"])
    .astype({"Time":"int32","Satellite_ID":"int32","Queue_Length":"int32","Packets_Dropped":"int32","Packets_Served":"int32"})
)

out_path = OUTPUT_DIR / "queue_congestion_results.csv.gz"
results_df.to_csv(out_path, index=False, compression="gzip")
print(f"Saved: {out_path.absolute()}")

# --- 6) Quick summary & visual cue (should scream “congestion”) ---
total_arrivals = int(arrivals_df.to_numpy(dtype=np.int64).sum())
total_dropped  = int(dropped_packets.sum())
drop_pct       = 100.0 * total_dropped / max(1, total_arrivals)
mean_q         = float(queue_lengths[1:].mean())
p95_q          = float(np.percentile(queue_lengths[1:], 95))

print(
    f"Summary → arrivals={total_arrivals:,} | dropped={total_dropped:,} ({drop_pct:.2f}%) | "
    f"meanQ={mean_q:.2f} | p95Q={p95_q:.0f}"
)

# Plot total drops over time — rising or sustained high values confirm pressure.
plt.figure(figsize=(10, 4))
plt.plot(time_index, drop_df.sum(axis=1))
plt.title("Total Drops per Second (CONGESTION)")
plt.xlabel("Time (s)")
plt.ylabel("Packets Dropped")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


---

In [ ]:
# ============================================================================================
# Step 40 — STRESS-TEST Queueing Simulation
#
# Why this step?
# This scenario deliberately pushes the satellite network beyond its limits.
# I reuse the high-traffic "congestion" input, but apply more extreme queue conditions:
#   → very small buffers (1000 pkts) + very slow processing (30 pps)
# The goal is to measure how the system fails under unsustainable load (worst-case behavior).
# ============================================================================================

print("Running STRESS-TEST queue simulation (aggressive conditions)...")

# --- 0) Prerequisite Checks ---
if 'OUTPUT_DIR' not in globals():
    raise RuntimeError("Missing OUTPUT_DIR. Run earlier setup first.")
if 'master_df' not in globals():
    raise RuntimeError("Missing master_df. Run λ traffic modeling step first.")

# --- 1) Load Congestion Events (Input to Stress Test) ---
events_gz  = OUTPUT_DIR / "congestion_traffic_events.csv.gz"
events_csv = OUTPUT_DIR / "congestion_traffic_events_log.csv"
if events_gz.exists():
    stress_input_df = pd.read_csv(events_gz)
    print(f"Loaded input traffic from: {events_gz.name}")
else:
    stress_input_df = pd.read_csv(events_csv)
    print(f"Loaded input traffic from: {events_csv.name}")

# --- 2) Reshape to Arrival Matrix ---
# Why reshape? The numpy-based simulator runs faster on dense matrix form.
T = int(stress_input_df["Time"].max())
time_index = pd.Index(range(1, T + 1), name="Time")

# Use satellite IDs from master (clean, sorted list)
sat_ids = sorted(master_df["Satellite_ID"].dropna().astype(int).unique().tolist())

arrivals_df = (
    stress_input_df
    .pivot_table(index="Time", columns="Satellite_ID",
                 values="NumPackets_Generated", aggfunc="sum", fill_value=0)
    .reindex(index=time_index, fill_value=0)
    .reindex(columns=sat_ids, fill_value=0)
    .astype("int32")
)
print("Arrivals matrix (Time x Sat):", arrivals_df.shape)

# --- 3) Stress-Test Parameters (Extreme) ---
BUFFER_CAPACITY     = 1000     # Intentionally small → queues overflow easily
SERVICE_RATE_PPS    = 30       # Intentionally slow → can't keep up
INITIAL_QUEUE_LOAD  = 0        # Start from empty (no burst advantage)

print(f"Stress Params: T={T}s, Capacity={BUFFER_CAPACITY}, Service={SERVICE_RATE_PPS}pps")

# --- 4) Preallocate State Arrays (T+1 × Sats) ---
queue_lengths    = np.zeros((T + 1, len(sat_ids)), dtype=np.int32)
dropped_packets  = np.zeros((T + 1, len(sat_ids)), dtype=np.int32)
serviced_packets = np.zeros((T + 1, len(sat_ids)), dtype=np.int32)

# Start with initial queue (zero or truncated if over-capacity)
init_q = np.full(len(sat_ids), INITIAL_QUEUE_LOAD, dtype=np.int32)
q_prev = np.minimum(init_q, BUFFER_CAPACITY)

# --- 5) Main Simulation Loop (Time-Stepped) ---
for t in range(T):
    arrivals = arrivals_df.iloc[t].to_numpy(dtype=np.int32, copy=False)

    # Step A: Serve from queue (limited by service rate)
    served = np.minimum(q_prev, SERVICE_RATE_PPS)

    # Step B–D: Add arrivals → drop excess → update state
    q_after = q_prev - served + arrivals
    dropped = np.maximum(0, q_after - BUFFER_CAPACITY)
    q_next  = np.minimum(q_after, BUFFER_CAPACITY)

    # Store results
    queue_lengths[t + 1]    = q_next
    dropped_packets[t + 1]  = dropped
    serviced_packets[t + 1] = served

    # Optional: assert queue mass-balance for first few seconds
    if t < 3:
        assert np.array_equal(q_next, q_prev - served + arrivals - dropped), f"Mass-balance error at t={t}"

    q_prev = q_next

print("Stress-test simulation completed.")

# --- 6) Tidy Output (Long Format) ---
queue_df  = pd.DataFrame(queue_lengths[1:], index=time_index, columns=sat_ids)
drop_df   = pd.DataFrame(dropped_packets[1:], index=time_index, columns=sat_ids)
served_df = pd.DataFrame(serviced_packets[1:], index=time_index, columns=sat_ids)

queue_long  = queue_df.melt( var_name="Satellite_ID", value_name="Queue_Length",    ignore_index=False).reset_index()
drop_long   = drop_df.melt(  var_name="Satellite_ID", value_name="Packets_Dropped", ignore_index=False).reset_index()
served_long = served_df.melt(var_name="Satellite_ID", value_name="Packets_Served",  ignore_index=False).reset_index()

# Merge all together
stress_results_df = (
    queue_long
    .merge(drop_long,   on=["Time", "Satellite_ID"])
    .merge(served_long, on=["Time", "Satellite_ID"])
    .astype({"Satellite_ID": "int32", "Time": "int32"})
)

# --- 7) Save Results (Compressed + Uncompressed) ---
csv_path = OUTPUT_DIR / "stress_test_results.csv"
gz_path  = OUTPUT_DIR / "stress_test_results.csv.gz"
stress_results_df.to_csv(csv_path, index=False)
stress_results_df.to_csv(gz_path,  index=False, compression="gzip")
print(f"Saved stress results to:\n  {csv_path.name}\n  {gz_path.name}")

# --- 8) Summary Stats (Key Indicators) ---
total_arrivals = int(arrivals_df.to_numpy(dtype=np.int64).sum())
total_dropped  = int(dropped_packets.sum())
drop_pct       = 100.0 * total_dropped / max(1, total_arrivals)
mean_q         = float(queue_lengths[1:].mean())
p95_q          = float(np.percentile(queue_lengths[1:], 95))

print(f"Summary: arrivals={total_arrivals:,} | dropped={total_dropped:,} ({drop_pct:.2f}%)")
print(f"Queue stats → meanQ={mean_q:.2f},  95th percentile Q={p95_q:.0f}")

# --- 9) Preview Results ---
display(stress_results_df.head())


---

In [ ]:
# ============================================================================================
# Step 41 — Detailed Validation of Stress-Test Results
#
# Why this step?
# This is the final validation phase for the stress-test queue simulation. It ensures:
#   - Simulation obeyed physical constraints (queue size, drop logic, etc.)
#   - Output structure is complete (no missing rows, no time gaps)
#   - Results are mathematically and logically sound before I use them in analysis
# ============================================================================================

import json

# --- 0) Load results from disk ---
results_path = OUTPUT_DIR / "stress_test_results.csv"
df = pd.read_csv(results_path)
print(f"Loaded: {results_path.name}")

# --- 1) Validate columns and NaNs ---
required_columns = {"Time", "Satellite_ID", "Queue_Length", "Packets_Dropped", "Packets_Served"}

# Why this check?
# Missing columns or NaNs usually mean earlier steps failed or output was corrupted.
print("Columns:", set(df.columns))
assert required_columns.issubset(df.columns), "Missing required columns"
assert df.isna().sum().sum() == 0, "There are NaN values in the dataset"

# --- 2) Check dataset shape ---
# Why?
# Ensures every satellite has entries for every time step.
T = int(df["Time"].max())
n = int(df["Satellite_ID"].nunique())
print(f"T = {T} seconds, n = {n} satellites")

# --- 3) Check buffer and service limits ---
buffer_cap, service_rate = 1000, 30  # Defaults used in simulation
cfg_path = OUTPUT_DIR / "stress_config.json"
if cfg_path.exists():
    try:
        cfg = json.loads(cfg_path.read_text())
        buffer_cap = int(cfg.get("BUFFER_CAPACITY_STRESS", buffer_cap))
        service_rate = int(cfg.get("SERVICE_RATE_STRESS_PPS", service_rate))
        print(f"Read config from stress_config.json → Capacity={buffer_cap}, Service={service_rate}")
    except Exception:
        print("Could not parse config file; using defaults:", buffer_cap, service_rate)
else:
    print(f"No config found. Using defaults → Capacity={buffer_cap}, Service={service_rate}")

# Enforce physical constraints
assert (df["Queue_Length"] >= 0).all(), "Negative queue length found"
assert (df["Queue_Length"] <= buffer_cap).all(), "Queue exceeds capacity"
assert (df["Packets_Served"] >= 0).all(), "Negative served packets"
assert (df["Packets_Served"] <= service_rate).all(), "Served packets exceed service rate"
assert (df["Packets_Dropped"] >= 0).all(), "Negative dropped packets"

# --- 4) Time coverage per satellite ---
# Why?
# Ensures there are no missing seconds in the time-series for any satellite.
df_sorted = df.sort_values(["Satellite_ID", "Time"]).copy()
time_coverage_ok = (
    df_sorted.groupby("Satellite_ID")["Time"]
    .apply(lambda s: np.array_equal(s.to_numpy(), np.arange(1, T + 1)))
)
assert time_coverage_ok.all(), "Some satellites are missing time steps"

# --- 5) Validate queue math (served ≤ previous queue) ---
# Why?
# Can't serve more than you have in your queue.
df_sorted["Prev_Q"] = df_sorted.groupby("Satellite_ID")["Queue_Length"].shift(1).fillna(0).astype(int)
violations = (df_sorted["Packets_Served"] > df_sorted["Prev_Q"]).sum()
assert violations == 0, f"Found {violations} violations of served > previous queue"

# Check t=1 edge case: served must be 0 if starting with empty queues
first_rows = df_sorted.groupby("Satellite_ID").head(1)
assert (first_rows["Packets_Served"] == 0).all(), "At t=1, served packets must be 0 (empty queue)"

# --- 6) Print total drops ---
# This gives a top-level summary of how severe the congestion was.
total_drops = int(df["Packets_Dropped"].sum())
print("Total dropped packets =", total_drops)

# --- 7) Final verdict ---
print("\nAll checks passed")


---

In [ ]:
# ============================================================================================
# Step 42 — Statistical Analysis of Stress-Test Results
#
# Why this step?
# To quantify and visualize which regions and satellites suffered the most under congestion.
# We evaluate:
#   - Drop rates
#   - Queue delay patterns
#   - Per-region performance
#   - Feature correlations (how traffic, queue size, and packet drops relate)
# ============================================================================================

print(" Statistical analysis + heatmap visualization (combined)...")

# --- 0) Environment checks ---
if 'OUTPUT_DIR' not in globals():
    raise RuntimeError("Missing OUTPUT_DIR.")
if 'master_df' not in globals():
    raise RuntimeError("Missing master_df.")

# --- 1) Load simulation outputs ---
stress_path = OUTPUT_DIR / "stress_test_results.csv.gz"
if stress_path.exists():
    stress_results_df = pd.read_csv(stress_path)
    print(f"Loaded stress results from: {stress_path.name}")
else:
    stress_results_df = pd.read_csv(OUTPUT_DIR / "stress_test_results.csv")
    print("Loaded stress results (CSV fallback).")

events_path = OUTPUT_DIR / "congestion_traffic_events.csv.gz"
if events_path.exists():
    congestion_events_df = pd.read_csv(events_path)
    print(f"Loaded congestion events from: {events_path.name}")
else:
    congestion_events_df = pd.read_csv(OUTPUT_DIR / "congestion_traffic_events_log.csv")
    print("Loaded congestion events (CSV fallback).")

# --- 2) Merge into unified analysis table ---
analysis_df = (
    stress_results_df.merge(
        congestion_events_df,
        on=["Time", "Satellite_ID"],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        master_df[["Satellite_ID", "Region"]],
        on="Satellite_ID",
        how="left",
        validate="many_to_one"
    )
)

# Optimize datatypes
int_cols = ["Time", "Satellite_ID", "Queue_Length", "Packets_Dropped", "Packets_Served", "NumPackets_Generated"]
for col in int_cols:
    if col in analysis_df.columns:
        analysis_df[col] = analysis_df[col].astype("int32")
if "Region" in analysis_df.columns:
    analysis_df["Region"] = analysis_df["Region"].astype("category")

print("All data loaded and merged.")
print("Rows:", len(analysis_df))

# --- 3) Derive queue delay from queue length ---
if 'SERVICE_RATE_STRESS_PPS' in globals():
    service_rate = int(SERVICE_RATE_STRESS_PPS)
    print(f"Using global SERVICE_RATE_STRESS_PPS: {service_rate} pps")
else:
    service_rate = int(analysis_df["Packets_Served"].max())
    print(f"Inferred service rate from data: {service_rate} pps")

def queue_delay_seconds(q):
    return (q / max(service_rate, 1)).astype("float32")

# --- 4) Regional statistics ---
regional_summary = (
    analysis_df
    .groupby("Region", observed=True)
    .agg(
        Mean_Dropped_Packets=("Packets_Dropped", "mean"),
        Variance_Dropped_Packets=("Packets_Dropped", "var"),
        Mean_Queue_Delay_s=("Queue_Length", lambda s: queue_delay_seconds(s).mean()),
        Variance_Queue_Delay_s=("Queue_Length", lambda s: queue_delay_seconds(s).var()),
        P50_Queue_Length=("Queue_Length", "median"),
        P95_Queue_Length=("Queue_Length", lambda s: np.percentile(s, 95)),
        Total_Dropped=("Packets_Dropped", "sum"),
        Total_Arrivals=("NumPackets_Generated", "sum")
    )
    .reset_index()
)

regional_summary["Drop_Rate_%"] = (
    100.0 * regional_summary["Total_Dropped"] / regional_summary["Total_Arrivals"].clip(lower=1)
)

print("\n--- Regional Summary (Top 5 by Drop_Rate_%) ---")
display(
    regional_summary[["Region", "Drop_Rate_%", "P95_Queue_Length", "Mean_Queue_Delay_s"]]
    .sort_values("Drop_Rate_%", ascending=False)
    .head()
)

# --- 5) Correlation analysis ---
target_cols = ["NumPackets_Generated", "Queue_Length", "Packets_Dropped", "Packets_Served"]
pearson_corr  = analysis_df[target_cols].corr(method="pearson")
spearman_corr = analysis_df[target_cols].corr(method="spearman")

print("\n--- Pearson Correlation Matrix ---")
display(pearson_corr)

print("\n--- Spearman Correlation Matrix ---")
display(spearman_corr)

# --- 6) Save outputs to CSV ---
regional_summary_path = OUTPUT_DIR / "stress_regional_summary.csv"
cor_pearson_path      = OUTPUT_DIR / "stress_corr_pearson.csv"
cor_spearman_path     = OUTPUT_DIR / "stress_corr_spearman.csv"

regional_summary.to_csv(regional_summary_path, index=False)
pearson_corr.to_csv(cor_pearson_path)
spearman_corr.to_csv(cor_spearman_path)

print("\nSaved CSVs:")
print(" ", regional_summary_path.absolute())
print(" ", cor_pearson_path.absolute())
print(" ", cor_spearman_path.absolute())

# --- 7) Heatmap Plotting ---
def plot_heatmap(corr_df, title, out_file):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.matshow(corr_df.values, vmin=-1, vmax=1)
    
    ax.set_xticks(range(len(corr_df.columns)))
    ax.set_yticks(range(len(corr_df.index)))
    ax.set_xticklabels(corr_df.columns, rotation=45, ha="left")
    ax.set_yticklabels(corr_df.index)
    
    ax.set_title(title, pad=20, fontsize=14)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Correlation")

    for (i, j), val in np.ndenumerate(corr_df.values):
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig(out_file, dpi=200, bbox_inches="tight")
    print("Saved heatmap to:", out_file)
    plt.show()

# Save heatmaps
heatmap_pearson_path  = OUTPUT_DIR / "stress_corr_pearson_heatmap.png"
heatmap_spearman_path = OUTPUT_DIR / "stress_corr_spearman_heatmap.png"
plot_heatmap(pearson_corr,  "Pearson Correlation (Stress-Test)",  heatmap_pearson_path)
plot_heatmap(spearman_corr, "Spearman Correlation (Stress-Test)", heatmap_spearman_path)


---

In [ ]:
# ============================================================================================
# Step 43 — Save Final Artifacts + Regional Comparison Plots
#
# Why this step?
# To produce all final exports (CSV + PNG) needed for thesis/report, and to visualize
# region-level performance under congestion: both drop rates and queue delays.
# ============================================================================================

print("Final exports & summary plots (non-repetitive)...")

# --- 0) Check required in-memory variables ---
required = ["analysis_df", "regional_summary", "pearson_corr", "spearman_corr", "OUTPUT_DIR"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Missing required variables: {missing}. Run the previous cell first.")

# --- 1) Define all export paths ---
final_stress_csv     = OUTPUT_DIR / "Stress_Test_Full_Results.csv"
final_stress_gz      = OUTPUT_DIR / "Stress_Test_Full_Results.csv.gz"
region_summary_csv   = OUTPUT_DIR / "Stress_Test_Region_Summary.csv"
pearson_csv          = OUTPUT_DIR / "Stress_Test_Corr_Pearson.csv"
spearman_csv         = OUTPUT_DIR / "Stress_Test_Corr_Spearman.csv"
final_plot_path      = OUTPUT_DIR / "Stress_Test_Summary_Plots.png"

# --- 2) Save CSV outputs ---
analysis_df.to_csv(final_stress_csv, index=False)
analysis_df.to_csv(final_stress_gz, index=False, compression="gzip")
regional_summary.to_csv(region_summary_csv, index=False)
pearson_corr.to_csv(pearson_csv)
spearman_corr.to_csv(spearman_csv)

# --- 3) Create and save summary plots ---
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))
fig.suptitle("Network Performance Under Stress by Region", fontsize=18)

# Plot 1: Average packet drop rate per region
ax1.bar(regional_summary["Region"], regional_summary["Mean_Dropped_Packets"])
ax1.set_title("Mean Dropped Packets per Second")
ax1.set_ylabel("Average Dropped Packets")
ax1.tick_params(axis="x", rotation=45)
ax1.grid(axis="y", linestyle="--", alpha=0.7)

# Plot 2: Average queue delay per region
ax2.bar(regional_summary["Region"], regional_summary["Mean_Queue_Delay_s"])
ax2.set_title("Mean Queue Delay (s)")
ax2.set_ylabel("Average Delay (s)")
ax2.tick_params(axis="x", rotation=45)
ax2.grid(axis="y", linestyle="--", alpha=0.7)

# Save the figure to file
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(final_plot_path, dpi=200, bbox_inches="tight")

# --- 4) Confirm export success ---
print("Saved files:")
print(" ", final_stress_csv.name)
print(" ", final_stress_gz.name)
print(" ", region_summary_csv.name)
print(" ", pearson_csv.name)
print(" ", spearman_csv.name)
print(" ", final_plot_path.name)

# --- 5) Show the figure inline ---
plt.show()


In [ ]:
# ============================================================================================
# Step 44 — Network KPIs: Throughput & Packet-Loss Rate (PLR)
#
# Why this step?
# To quantify overall and regional network performance under congestion, using key metrics:
# - Throughput (how much traffic got through)
# - PLR (how much was lost due to overloaded buffers)
# ============================================================================================

print("Computing KPIs (Throughput / PLR) — using in-memory analysis_df")

# --- 0) Sanity check for required dataframes ---
required = ["analysis_df", "OUTPUT_DIR"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Missing objects: {missing}. Please run the previous cell.")

# --- 1) Standardize time column ---
df = analysis_df.copy()
if "Time" in df.columns:
    time_col = "Time"
elif "Time" in df.columns:
    time_col = "Time"
    df = df.rename(columns={"Time": "Time"})
else:
    raise ValueError("analysis_df must contain a time column ('Time' or 'Time')")

# --- 2) Time-series KPIs for the entire network ---
by_time = df.groupby("Time", as_index=False).agg(
    Served_pps=("Packets_Served", "sum"),   # Throughput = packets served per sec
    Dropped=("Packets_Dropped", "sum")      # Congestion symptom = dropped packets
)

# Add traffic arrivals if available, for calculating drop rate
if "NumPackets_Generated" in df.columns:
    gen_t = df.groupby("Time")["NumPackets_Generated"].sum().reset_index(name="Generated")
    by_time = by_time.merge(gen_t, on="Time", how="left")
else:
    by_time["Generated"] = np.nan

# --- 3) Aggregate KPIs (single values) ---
seconds_total   = by_time["Time"].nunique()
served_total    = int(by_time["Served_pps"].sum())
dropped_total   = int(by_time["Dropped"].sum())
generated_total = by_time["Generated"].sum() if by_time["Generated"].notna().any() else np.nan

AVG_PACKET_SIZE_KB = 1.5
pps_mean  = float(by_time["Served_pps"].mean())
mbps_mean = pps_mean * (AVG_PACKET_SIZE_KB * 8) / 1000.0

print("\n### Network-Level KPIs")
print(f" • Mean Throughput = {pps_mean:,.0f} pps  (~{mbps_mean:.2f} Mbps @ {AVG_PACKET_SIZE_KB} KB/packet)")
print(f" • Total Served Packets = {served_total:,}")
if np.isfinite(generated_total):
    plr = 100.0 * dropped_total / max(generated_total, 1)
    print(f" • Packet-Loss Rate (PLR) = {plr:.2f}%")
else:
    print(" • PLR = Skipped (packet arrivals missing)")

# --- 4) Regional KPIs ---
regional_df = None
if "Region" in df.columns:
    reg = df.groupby("Region", dropna=False).agg(
        Served=("Packets_Served", "sum"),
        Dropped=("Packets_Dropped", "sum"),
        Generated=("NumPackets_Generated", "sum")
    ).reset_index()

    if reg["Generated"].notna().any():
        reg["PLR_%"] = 100.0 * reg["Dropped"] / reg["Generated"].clip(lower=1)

    reg["Throughput_pps"]  = reg["Served"] / max(seconds_total, 1)
    reg["Throughput_Mbps"] = reg["Throughput_pps"] * (AVG_PACKET_SIZE_KB * 8) / 1000.0
    regional_df = reg

# --- 5) Save outputs to CSV ---
ts_path = OUTPUT_DIR / "kpi_throughput_timeseries.csv"
by_time.to_csv(ts_path, index=False)
print(f"\nSaved: {ts_path.name}")

if regional_df is not None:
    reg_path = OUTPUT_DIR / "kpi_regional_throughput_plr.csv"
    regional_df.to_csv(reg_path, index=False)
    print(f"Saved: {reg_path.name}")

# --- 6) Plot: Throughput over Time (entire network) ---
plt.figure(figsize=(12, 5))
plt.plot(by_time["Time"], by_time["Served_pps"], linewidth=1.5)
plt.title("Network Throughput Over Time (pps)")
plt.xlabel("Time (s)"); plt.ylabel("Packets Served per Second")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "kpi_throughput_timeseries.png", dpi=180, bbox_inches="tight")
plt.show()

# --- 7) Plot: Packet Loss Rate over Time ---
if by_time["Generated"].notna().any():
    plr_t = (by_time["Dropped"] / by_time["Generated"].clip(lower=1)) * 100
    plt.figure(figsize=(12, 5))
    plt.plot(by_time["Time"], plr_t, linewidth=1.2)
    plt.title("Packet Loss Rate Over Time (%)")
    plt.xlabel("Time (s)"); plt.ylabel("PLR (%)")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "kpi_plr_timeseries.png", dpi=180, bbox_inches="tight")
    plt.show()

# --- 8) Plot: Regional Throughput (bar chart) ---
if regional_df is not None:
    ax = (regional_df.sort_values("Throughput_Mbps", ascending=False)
           .plot(x="Region", y="Throughput_Mbps", kind="bar", figsize=(10,5),
                 legend=False, title="Mean Regional Throughput (Mbps)"))
    ax.set_ylabel("Mbps"); ax.set_xlabel("Region")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "kpi_regional_throughput_bar.png", dpi=180, bbox_inches="tight")
    plt.show()

print("\nDone.")


---
---

## Phase 8: Routing Experiment Setup and Neighborhood Feature Construction

In [ ]:
# ========================================================================================
# Step 45 — Build Fast Lookup Dictionaries
#
# Why this step?
# These dictionaries give constant-time access to dynamic network state variables
# like queue delay, busy status, neighbor lists, and physical distances — which are
# essential for both classical and ML-based routing decisions.
# ========================================================================================

print("Building fast lookup dictionaries...")

# --- 1. Input checks ---
# I want to fail early if any key inputs are missing from memory.
needed = ["analysis_df", "neighbors_df", "edge_list_df"]
missing = [n for n in needed if n not in globals()]
if missing:
    raise RuntimeError(f"Missing objects: {missing}. Run previous prep cells first.")

# --- 2. Service rate for delay calculation ---
# Needed to convert queue length into estimated delay (Queue_Length / Service_Rate).
if "SERVICE_RATE_STRESS_PPS" in globals():
    service_rate = int(SERVICE_RATE_STRESS_PPS)
elif "Packets_Served" in analysis_df.columns:
    service_rate = int(analysis_df["Packets_Served"].max())  # Estimate from max observed
else:
    service_rate = 30  # Default fallback
service_rate = max(service_rate, 1)  # Avoid divide-by-zero

BUSY_THRESHOLD_PPS = 5  # Threshold for considering a node congested
has_arrivals = "NumPackets_Generated" in analysis_df.columns

# --- 3. Add derived columns if missing ---
# These are the columns we'll use to create the dictionaries.
df = analysis_df.copy()

if "Delay_s" not in df.columns:
    df["Delay_s"] = df["Queue_Length"] / service_rate  # Delay proxy in seconds

if "Is_Busy" not in df.columns:
    if has_arrivals:
        df["Is_Busy"] = (df["NumPackets_Generated"] > BUSY_THRESHOLD_PPS).astype("int8")
    else:
        df["Is_Busy"] = 0  # If arrival info is missing, default to not busy

# --- 4. Build dictionaries for fast access ---

# 4a. Queue Delay Map → (Time, Satellite) → Delay_s
# Why? Used by delay-aware or congestion-sensitive algorithms to rank next hops.
delay_map = df.set_index(["Time", "Satellite_ID"])["Delay_s"].to_dict()

# 4b. Busy Status Map → (Time, Satellite) → 0/1 (Not Busy / Busy)
# Why? Lets the algorithm avoid currently overloaded nodes.
busy_map = df.set_index(["Time", "Satellite_ID"])["Is_Busy"].to_dict()

# 4c. Neighbor List Map → Satellite → [neighbor1, neighbor2, ...]
# Why? Required to know who a satellite can forward to at any time.
ndf = neighbors_df.copy()

# Some JSON cleaning — handles possible stringified lists
if "Neighbor_IDs" not in ndf.columns and "Neighbor_IDs_JSON" in ndf.columns:
    ndf["Neighbor_IDs"] = ndf["Neighbor_IDs_JSON"].apply(json.loads)
elif "Neighbor_IDs" in ndf.columns and ndf["Neighbor_IDs"].dtype == object:
    try:
        ndf["Neighbor_IDs"] = ndf["Neighbor_IDs"].apply(
            lambda x: x if isinstance(x, list) else json.loads(x)
        )
    except Exception:
        pass  # Leave as-is if it fails

neighbor_map = (
    ndf.drop_duplicates("Satellite_ID")
       .set_index("Satellite_ID")["Neighbor_IDs"]
       .to_dict()
)

# 4d. Distance Map → (u, v) or (v, u) → Distance_km
# Why? Needed by all routing algorithms for cost computation.
distance_map = {}
for u, v, d in edge_list_df[["From_ID", "To_ID", "Distance_km"]].to_numpy():
    distance_map[(int(u), int(v))] = float(d)
    distance_map[(int(v), int(u))] = float(d)  # Graph is undirected

# --- 5. Summary of constructed maps ---
T = df["Time"].nunique()
N = df["Satellite_ID"].nunique()

print("Lookup dictionaries built:")
print(f" - delay_map entries:   {len(delay_map):,}  (expected ≈ {T*N:,})")
print(f" - busy_map entries:    {len(busy_map):,}")
print(f" - neighbor_map nodes:  {len(neighbor_map):,}")
print(f" - distance_map pairs:  {len(distance_map):,}")


---

In [ ]:
# ========================================================================================
# Step 46 — Vectorized ECEF Conversion and Physical Link Distance Computation
#
# Why this step?
# Satellite positions are given in latitude/longitude (angular), which are hard
# to use for direct distance calculations. Converting to ECEF (x, y, z) 3D space
# lets us compute ISL lengths using simple vector math.
# ========================================================================================

print("Vectorized ECEF utilities ready. (Use for validation or new snapshots.)")

# --- Constants for WGS-84 Earth model and satellite orbit ---
DEFAULT_SATELLITE_ALTITUDE_KM = 1200.0  # Assumed uniform altitude for all LEO nodes

# WGS-84 ellipsoid parameters (used for accurate conversion)
_A  = 6378.137              # Semi-major axis (equatorial radius) in km
_F  = 1.0 / 298.257223563   # Flattening factor of the ellipsoid
_E2 = _F * (2.0 - _F)       # Eccentricity squared

def geodetic_to_ecef_np(lat_deg, lon_deg, h_km=DEFAULT_SATELLITE_ALTITUDE_KM):
    """
    Convert (lat, lon, altitude) to ECEF (x, y, z) coordinates.

    Why?
    Latitude and longitude are angles on a sphere, but routing and distance
    calculations require Cartesian coordinates. ECEF is a fixed 3D coordinate
    system centered at Earth's core. Once in ECEF, distances = fast vector math.

    Vectorized with NumPy for high-speed batch processing of all satellites.
    """
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)

    sin_lat = np.sin(lat)
    cos_lat = np.cos(lat)

    # N = radius of curvature at that latitude
    N = _A / np.sqrt(1.0 - _E2 * sin_lat**2)

    # Standard WGS-84 conversion formula
    x = (N + h_km) * cos_lat * np.cos(lon)
    y = (N + h_km) * cos_lat * np.sin(lon)
    z = (N * (1.0 - _E2) + h_km) * sin_lat

    return np.stack([x, y, z], axis=-1)

def ecef_distance_np(p1_xyz, p2_xyz):
    """
    Compute Euclidean (straight-line) distance between two ECEF points.

    Why?
    Once in (x,y,z), finding the physical length of an inter-satellite link
    is just the norm of the vector difference: ||p1 - p2||
    """
    return np.linalg.norm(p1_xyz - p2_xyz, axis=-1)

def compute_edge_distances_from_positions(positions_df, edges_df, default_alt_km=DEFAULT_SATELLITE_ALTITUDE_KM):
    """
    Compute the real physical distance (in km) for each edge in the network.

    Why?
    This validates or constructs the ISL link lengths from satellite positions.
    Needed for distance-aware routing or threshold-based ISL filtering.

    Input:
        - positions_df: must contain columns [Satellite_ID, Latitude, Longitude]
        - edges_df:     must contain columns [From_ID, To_ID]
    Output:
        - Numpy array of physical distances, same order as rows in edges_df
    """
    # Pre-index for fast lookups
    pos = positions_df.set_index("Satellite_ID")[["Latitude", "Longitude"]]

    # Lookup the lat/lon of each endpoint in each edge
    lat1 = pos.loc[edges_df["From_ID"], "Latitude"].to_numpy()
    lon1 = pos.loc[edges_df["From_ID"], "Longitude"].to_numpy()
    lat2 = pos.loc[edges_df["To_ID"], "Latitude"].to_numpy()
    lon2 = pos.loc[edges_df["To_ID"], "Longitude"].to_numpy()

    # Convert to 3D Cartesian
    p1 = geodetic_to_ecef_np(lat1, lon1, default_alt_km)
    p2 = geodetic_to_ecef_np(lat2, lon2, default_alt_km)

    # Vectorized distance computation
    return ecef_distance_np(p1, p2)


---

In [ ]:
# ========================================================================================
# Step 47 — Create Neighbor-Aware Features for ML
#
# Why this step?
# Local features (queue length, drops) only describe an individual satellite.
# But routing depends on *neighbor conditions*: if neighbors are busy or congested,
# that directly impacts path performance. These features encode that context.
# ========================================================================================

print("Neighbor-aware features — execution only (reusing in-memory data)…")

# --- 0) Minimal checks ---
needed = ["analysis_df", "neighbors_df", "OUTPUT_DIR"]
missing = [n for n in needed if n not in globals()]
if missing:
    raise RuntimeError(f"Missing objects: {missing}. Run prior analysis/graph prep cells first.")

df  = analysis_df.copy()
OUT = Path(OUTPUT_DIR)

# --- 1) Ensure required columns & basics ---
req = {"Time","Satellite_ID","Queue_Length","NumPackets_Generated"}
if not req.issubset(df.columns):
    raise ValueError(f"analysis_df must have columns {req}, found {set(df.columns)}")

# Service rate needed for delay proxy
if "SERVICE_RATE_PPS" in globals():
    service_rate = int(SERVICE_RATE_PPS)
elif "SERVICE_RATE_STRESS_PPS" in globals():
    service_rate = int(SERVICE_RATE_STRESS_PPS)
elif "Packets_Served" in df.columns:
    service_rate = int(df["Packets_Served"].max())  # infer from data
else:
    service_rate = 30  # conservative fallback
service_rate = max(service_rate, 1)

# --- 2) Ensure Delay_s and Is_Busy ---
if "Delay_s" not in df.columns:
    # Queue delay proxy = queue length ÷ service rate
    df["Delay_s"] = (df["Queue_Length"] / service_rate).astype("float32")

BUSY_THRESHOLD_PPS = 5
if "Is_Busy" not in df.columns:
    # Mark a satellite as “busy” if its arrivals > threshold
    df["Is_Busy"] = (df["NumPackets_Generated"] > BUSY_THRESHOLD_PPS).astype("int8")

# --- 3) Prepare neighbor list ---
ndf = neighbors_df.copy()
if "Neighbor_IDs" not in ndf.columns and "Neighbor_IDs_JSON" in ndf.columns:
    ndf["Neighbor_IDs"] = ndf["Neighbor_IDs_JSON"].apply(json.loads)
elif "Neighbor_IDs" in ndf.columns and ndf["Neighbor_IDs"].dtype == object:
    try:
        ndf["Neighbor_IDs"] = ndf["Neighbor_IDs"].apply(
            lambda x: x if isinstance(x, list) else json.loads(x)
        )
    except Exception:
        pass
node_to_neighbors = ndf[["Satellite_ID","Neighbor_IDs"]].drop_duplicates("Satellite_ID")

# --- 4) Build time-aware edges ---
# Idea: expand the dataset so each (satellite,time) is linked with all its neighbors
time_nodes = df[["Time","Satellite_ID"]].drop_duplicates().rename(columns={"Satellite_ID":"Source"})
edges_df   = (node_to_neighbors.rename(columns={"Satellite_ID":"Source"})
                               .explode("Neighbor_IDs")
                               .rename(columns={"Neighbor_IDs":"Neighbor"}))
time_edges = time_nodes.merge(edges_df, on="Source", how="left")

# --- 5) Attach neighbor stats ---
neighbor_stats = time_edges.merge(
    df[["Time","Satellite_ID","Delay_s","Is_Busy"]],
    left_on=["Time","Neighbor"],
    right_on=["Time","Satellite_ID"],
    how="left"
)

# --- 6) Aggregate neighbor stats into features ---
agg = (neighbor_stats
       .groupby(["Time","Source"], as_index=False)
       .agg(
           Avg_Neighbor_Delay=("Delay_s","mean"),   # mean delay of neighbors
           Busy_Neighbor_Ratio=("Is_Busy","mean"), # fraction of busy neighbors
           Neighbor_Count=("Is_Busy","size")       # number of neighbors
       ))

# --- 7) Merge back into main DataFrame ---
df = df.merge(agg, left_on=["Time","Satellite_ID"], right_on=["Time","Source"], how="left")
df["Neighbor_Count"]      = df["Neighbor_Count"].fillna(0).astype("int16")
df["Has_Neighbors"]       = (df["Neighbor_Count"] > 0).astype("int8")
df["Avg_Neighbor_Delay"]  = df["Avg_Neighbor_Delay"].fillna(0).astype("float32")
df["Busy_Neighbor_Ratio"] = df["Busy_Neighbor_Ratio"].fillna(0).astype("float32")
if "Source" in df.columns:
    df.drop(columns=["Source"], inplace=True)

# --- 8) Quick validation plots ---
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(16,6))
fig.suptitle("Neighbor-Aware Features", fontsize=18)

# Expectation: satellites with high neighbor delay should also see high own delay
ax1.scatter(df["Avg_Neighbor_Delay"], df["Delay_s"], s=5, alpha=0.2)
ax1.set_title("Own Delay vs Avg Neighbor Delay")
ax1.set_xlabel("Avg Neighbor Delay (s)")
ax1.set_ylabel("Own Delay (s)")
ax1.grid(True, linestyle="--", alpha=0.5)

# Expectation: Busy_Neighbor_Ratio will typically cluster near 0 or 1
ax2.hist(df["Busy_Neighbor_Ratio"], bins=50, alpha=0.7)
ax2.set_title("Busy Neighbor Ratio")
ax2.set_xlabel("Ratio (0 = all idle, 1 = all busy)")
ax2.set_ylabel("Frequency")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout(rect=[0,0,1,0.95])
plot_path = OUT / "final_feature_analysis.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
print(f"Saved plot: {plot_path}")
plt.show()

# --- 9) Save the enriched dataset ---
save_df = df.drop(columns=["Neighbor_IDs","Neighbor_IDs_JSON"], errors="ignore")
final_csv = OUT / "final_enriched_ml_dataset.csv"
save_df.to_csv(final_csv, index=False)
print(f"Saved dataset: {final_csv}")

# Make enriched DataFrame globally available
analysis_df = df


---

In [ ]:
# ========================================================================================
# Step 48 — Routing Experiment Setup
#
# Why this step?
# Up to now, we have enriched the dataset with congestion-aware and neighbor-aware features.
# Now, we need to create a controlled experiment environment where we can test different
# routing strategies (baseline vs. ML). This setup builds the network graph, indexes
# satellites by region & time, and prepares reproducible sampling of source–destination pairs.
# ========================================================================================

print("Setting up routing experiment (non-repetitive, execution-only)...")

# --- 1) Load edge list ---
if 'edge_list_df' not in globals():
    edge_path = OUTPUT_DIR / "satellite_link_edge_list.csv"
    edge_list_df = pd.read_csv(edge_path)
    print(f"edge_list_df loaded from {edge_path.name}")

# --- 2) Sanity checks ---
needed_objects = ["analysis_df", "edge_list_df"]
missing = [n for n in needed_objects if n not in globals()]
if missing:
    raise RuntimeError(f"Missing objects in memory: {missing}")

df       = analysis_df.copy()
edges_df = edge_list_df.copy()

required_cols = {"Time", "Satellite_ID", "Region", "Delay_s", "Has_Neighbors"}
missing_cols  = sorted(list(required_cols - set(df.columns)))
if missing_cols:
    raise ValueError(f"analysis_df missing required columns: {missing_cols}")

# --- 3) Filter edges to experiment subgraph ---
experiment_nodes = set(df["Satellite_ID"].unique())
edges_df = edges_df[
    edges_df["From_ID"].isin(experiment_nodes) &
    edges_df["To_ID"].isin(experiment_nodes)
].drop_duplicates(subset=["From_ID", "To_ID"])

# --- 4) Canonicalize edges to undirected form ---
edge_list_df[["From_ID","To_ID"]] = edge_list_df[["From_ID","To_ID"]].astype(int)
df["Satellite_ID"] = df["Satellite_ID"].astype(int)

experiment_nodes = set(df["Satellite_ID"].unique())
edges_df = edge_list_df[
    edge_list_df["From_ID"].isin(experiment_nodes) &
    edge_list_df["To_ID"].isin(experiment_nodes)
].copy()

u = edges_df[["From_ID","To_ID"]].min(axis=1)
v = edges_df[["From_ID","To_ID"]].max(axis=1)
edges_df = pd.DataFrame({
    "From_ID": u,
    "To_ID":   v,
    "Distance_km": edges_df["Distance_km"].values
}).drop_duplicates(subset=["From_ID","To_ID"])

# --- 5) Add physical propagation time ---
SPEED_OF_LIGHT_KM_S = 299_792.458
edges_df["propagation_Time"] = edges_df["Distance_km"] / SPEED_OF_LIGHT_KM_S

# --- 6) Build base graph (undirected, physical topology) ---
G_base = nx.from_pandas_edgelist(
    edges_df,
    source="From_ID",
    target="To_ID",
    edge_attr=["Distance_km", "propagation_Time"]
)

# --- 7) Region–time index for fast lookups ---
with_neighbors = df[df["Has_Neighbors"] == 1].copy()
region_time_index = (
    with_neighbors.groupby(["Time", "Region"])["Satellite_ID"]
    .apply(list)
    .to_dict()
)

# --- 8) Experiment control knobs ---
SNAPSHOTS_TO_ANALYZE = [30, 60, 90, 120, 150]   # times to test
NUM_PAIRS_TO_TEST    = 20                       # random pairs per scenario
INTERCONTINENTAL_SCENARIOS = [
    ("Asia", "North America"),
    ("Europe", "North America"),
    ("Europe", "South America"),
]
rng = random.Random(RANDOM_SEED if "RANDOM_SEED" in globals() else 42)

# --- 9) Summary printout ---
available_times = sorted(df["Time"].unique().tolist())
print("\n--- Experiment Setup Complete ---")
print(f"Snapshots available (first 10): {available_times[:10]} ...")
print(f"Selected snapshots: {SNAPSHOTS_TO_ANALYZE}")
print(f"Pairs per scenario: {NUM_PAIRS_TO_TEST}")
print(f"Nodes: {len(experiment_nodes)} | Edges: {G_base.number_of_edges()}")
print("---------------------------------------------------------------")

# --- 10) Helper: sample source–destination pairs ---
def sample_sd_pairs_at_time(t, src_region, dst_region, k=NUM_PAIRS_TO_TEST, rng_obj=rng):
    """
    Sample up to `k` random source–destination pairs at time `t`.

    Why?
    To fairly evaluate routing strategies, we need a consistent set of test pairs
    across regions and time snapshots. This helper quickly looks up eligible
    satellites from the region_time_index and samples reproducibly.
    """
    src_list = region_time_index.get((t, src_region), [])
    dst_list = region_time_index.get((t, dst_region), [])
    if not src_list or not dst_list:
        return []

    pairs = set()
    max_possible = len(src_list) * len(dst_list)
    while len(pairs) < k and len(pairs) < max_possible:
        s = rng_obj.choice(src_list)
        d = rng_obj.choice(dst_list)
        if s != d:
            pairs.add((s, d))

    return list(pairs)


---

In [ ]:
# ============================================================================================
# Step 49 — Routing Experiment (DistanceOnly vs. DelayAware)
#
# Why this step?
# This is the core evaluation phase: I compare two routing algorithms on real snapshot data:
#   - DistanceOnly: chooses shortest physical paths (fast but congestion-unaware)
#   - DelayAware:  chooses lowest-latency paths (accounts for queueing delay)
#
# I also extract the "NextHop" and full path for DelayAware to use later in ML training.
# ============================================================================================

print("Starting the main routing experiment (clean rewrite, with NextHop for both algos)...")

# --- 0) Global Seeds for Reproducibility ---
SEED = 42
rng = random.Random(SEED)
np.random.seed(SEED)

# --- 1) Helper Functions ---

def build_graphs_for_snapshot(snapshot_df, edges_df):
    """
    Builds two graphs:
    - G_dist: undirected, weights = propagation delay
    - G_delay: directed, weights = propagation delay + target node queue delay

    Also builds:
    - qmap[sat_id] = queue delay
    - qmap[(sat_id, "trend")] = short-term queue trend
    """

    # Current queue delays
    queue_delay_map = (
        snapshot_df
        .set_index("Satellite_ID")["Delay_s"]
        .to_dict()
    )

    # Temporal queue trends
    queue_trend_map = (
        snapshot_df
        .set_index("Satellite_ID")["Queue_Trend"]
        .fillna(0.0)
        .to_dict()
    )

    # Unified feature-serving map
    qmap = {}

    # Delay entries
    for sid, delay in queue_delay_map.items():
        qmap[int(sid)] = float(delay)

    # Trend entries
    for sid, trend in queue_trend_map.items():
        qmap[(int(sid), "trend")] = float(trend)

    G_dist = nx.Graph()
    G_delay = nx.DiGraph()

    for row in edges_df.itertuples(index=False):

        u = int(row.From_ID)
        v = int(row.To_ID)

        prop_s = float(row.Distance_km) / SPEED_OF_LIGHT_KM_S

        # Pure propagation graph
        if not G_dist.has_edge(u, v):
            G_dist.add_edge(u, v, weight=prop_s)

        # Delay-aware graph
        qu = qmap.get(u, 0.0)
        qv = qmap.get(v, 0.0)

        G_delay.add_edge(
            u,
            v,
            weight=prop_s + qv
        )

        G_delay.add_edge(
            v,
            u,
            weight=prop_s + qu
        )

    return G_dist, G_delay, qmap


def get_true_e2e_delay(path, G_dist, qmap):
    """
    End-to-end delay definition in this thesis:
    sum of propagation times along the path
    + sum of node-level queueing delays for all nodes except the source
      (this includes the destination ingress/processing queue).
    """

    total = 0.0

    # Propagation delay
    for i in range(len(path) - 1):
        total += G_dist[path[i]][path[i + 1]]["weight"]

    # Queueing delay
    total += sum(
        float(qmap.get(n, 0.0))
        for n in path[1:]
    )

    return total


def _pairs_for(t, ra, rb, k):
    return (
        sample_sd_pairs_at_time(t, ra, rb, k=k, rng_obj=rng)
        if "sample_sd_pairs_at_time" in globals()
        else []
    )


# --- 2) Ensure Inputs are Available ---

if "timeline_df" not in globals():

    timeline_path = OUTPUT_DIR / "final_enriched_ml_dataset.csv"

    timeline_df = pd.read_csv(timeline_path)

    print(f"Loaded timeline_df from {timeline_path.name}")


if "edges_df" not in globals():

    edges_path = OUTPUT_DIR / "satellite_link_edge_list.csv"

    edges_df = pd.read_csv(edges_path)

    print(f"Loaded edges_df from {edges_path.name}")


# Optional: early skip for unreachable pairs
comp_path = OUTPUT_DIR / "component_labels_by_node.csv"

comp_labels = (
    pd.read_csv(comp_path)
    .set_index("Node")["Component"]
    .to_dict()
    if comp_path.exists()
    else None
)

# --- 3) Main Experiment Loop ---

final_path = OUTPUT_DIR / "final_routing_comparison_results.csv"

final_path.unlink(missing_ok=True)

all_results = []
teacher_paths = []  # stores DelayAware path for each (src, dst)

for t_snapshot in SNAPSHOTS_TO_ANALYZE:

    print(f"\n--- Processing snapshot at t={t_snapshot}s ---")

    # Keep history up to current snapshot
    # so temporal trend becomes meaningful
    snapshot_df = timeline_df[
        timeline_df["Time"] <= t_snapshot
    ].copy()

    if snapshot_df.empty:

        print(f"No data found for t={t_snapshot}s. Skipping.")

        continue

    # Ensure correct temporal ordering
    snapshot_df = snapshot_df.sort_values(
        ["Satellite_ID", "Time"]
    ).copy()

    # Causal temporal trend:
    # Queue_Trend(t) = Queue_Length(t) - Queue_Length(t-1)
    snapshot_df["Queue_Trend"] = (
        snapshot_df
        .groupby("Satellite_ID")["Queue_Length"]
        .diff()
        .fillna(0.0)
    )

    # Keep only current rows for routing
    current_snapshot_df = snapshot_df[
        snapshot_df["Time"] == t_snapshot
    ].copy()

    # Build graphs + feature-serving maps
    # using history-aware temporal features
    G_dist, G_delay, qmap = build_graphs_for_snapshot(
        snapshot_df,
        edges_df
    )

    # From this point onward,
    # use only current snapshot rows
    snapshot_df = current_snapshot_df

    for region_a, region_b in INTERCONTINENTAL_SCENARIOS:

        pairs = _pairs_for(
            t_snapshot,
            region_a,
            region_b,
            NUM_PAIRS_TO_TEST
        )

        # Fallback sampling if no precomputed pairs exist
        if not pairs:

            nodes_a = snapshot_df.loc[
                snapshot_df["Region"] == region_a,
                "Satellite_ID"
            ].tolist()

            nodes_b = snapshot_df.loc[
                snapshot_df["Region"] == region_b,
                "Satellite_ID"
            ].tolist()

            if not nodes_a or not nodes_b:
                continue

            pairs_set = set()

            tries = 0

            while (
                len(pairs_set) < NUM_PAIRS_TO_TEST
                and tries < 20 * NUM_PAIRS_TO_TEST
            ):

                s = rng.choice(nodes_a)
                d = rng.choice(nodes_b)

                if s != d:
                    pairs_set.add((s, d))

                tries += 1

            pairs = list(pairs_set)

        for s, d in pairs:

            # Optional early skip for disconnected pairs
            if (
                comp_labels
                and comp_labels.get(int(s)) != comp_labels.get(int(d))
            ):
                continue

            try:

                path_do = nx.dijkstra_path(
                    G_dist,
                    s,
                    weight="weight",
                    target=d
                )

                path_da = nx.dijkstra_path(
                    G_delay,
                    s,
                    weight="weight",
                    target=d
                )

            except nx.NetworkXNoPath:
                continue

            # Calculate true end-to-end delays
            e2e_do = get_true_e2e_delay(
                path_do,
                G_dist,
                qmap
            )

            e2e_da = get_true_e2e_delay(
                path_da,
                G_dist,
                qmap
            )

            # Extract first-hop decisions
            nh_do = int(path_do[1]) if len(path_do) > 1 else None

            nh_da = int(path_da[1]) if len(path_da) > 1 else None

            all_results.append({
                "Time": t_snapshot,
                "Scenario": f"{region_a} -> {region_b}",
                "Algorithm": "DistanceOnly",
                "Source": s,
                "Destination": d,
                "E2E_Delay_s": e2e_do,
                "Hop_Count": len(path_do) - 1,
                "NextHop": nh_do
            })

            all_results.append({
                "Time": t_snapshot,
                "Scenario": f"{region_a} -> {region_b}",
                "Algorithm": "DelayAware",
                "Source": s,
                "Destination": d,
                "E2E_Delay_s": e2e_da,
                "Hop_Count": len(path_da) - 1,
                "NextHop": nh_da
            })

            teacher_paths.append({
                "Time": t_snapshot,
                "Source": s,
                "Destination": d,
                "Path": path_da
            })

# --- 4) Save Results ---

if not all_results:

    print("\nExperiment finished, but no valid routes found.")

else:

    results_df = pd.DataFrame(all_results)

    key_cols = [
        "Time",
        "Scenario",
        "Algorithm",
        "Source",
        "Destination"
    ]

    results_df = (
        results_df
        .drop_duplicates(subset=key_cols)
        .reset_index(drop=True)
    )

    results_df["NextHop"] = (
        results_df["NextHop"]
        .astype("Int64")
    )

    results_df["Hop_Count"] = (
        results_df["Hop_Count"]
        .astype("Int64")
    )

    results_df.to_csv(final_path, index=False)

    print(
        f"\nExperiment complete. Results saved to:\n"
        f"  {final_path.resolve()}"
    )

    # --- 5) Sanity Checks ---

    print("\nSanity checks:")

    print("  NextHop non-null ratio per algorithm:")

    print(
        results_df
        .groupby("Algorithm")["NextHop"]
        .apply(lambda s: s.notna().mean())
    )

    print(
        "  Max DistanceOnly delay (s):",
        results_df.query(
            "Algorithm == 'DistanceOnly'"
        )["E2E_Delay_s"].max()
    )

    print("\n--- Sample of Final Results ---")

    display(results_df.head())

    for t in sorted(results_df["Time"].dropna().unique()):

        out_t = OUTPUT_DIR / f"Routing_Results_t{int(t)}.csv"

        out_t.unlink(missing_ok=True)

        results_df[
            results_df["Time"] == t
        ].to_csv(out_t, index=False)

        print("  Saved:", out_t.name)

    # --- 6) Save Teacher Paths for ML (JSON) ---

    tp_file = OUTPUT_DIR / "teacher_paths.json"

    with open(tp_file, "w") as f:
        json.dump(teacher_paths, f, indent=2)

    print(f"Teacher paths saved to {tp_file}")

---

In [ ]:
# ============================================================================================
# Step 50 — Validate Routing Results
#
#  Why this step?
# Before I use the routing output for analysis or ML training, I need to confirm that:
# - The data is complete and correctly recorded
# - The "NextHop" decisions are valid (i.e., legal neighbors)
# - The DelayAware routing actually outperforms DistanceOnly in most cases
# ============================================================================================

# --- 1) Load results + check structure ---
res   = pd.read_csv(OUTPUT_DIR / "final_routing_comparison_results.csv")
edges = pd.read_csv(OUTPUT_DIR / "satellite_link_edge_list.csv")

#  I expect each row to describe a single (src, dst, time, algo) routing result.
required_cols = [
    "Time", "Scenario", "Algorithm", "Source", "Destination",
    "E2E_Delay_s", "Hop_Count", "NextHop"
]
missing_cols = [c for c in required_cols if c not in res.columns]
print("Missing required columns:", missing_cols)

#  Only 'NextHop' is allowed to have nulls (e.g., unreachable cases)
print("\nNull counts in required columns:")
print(res[required_cols].isna().sum())

# --- 2) Check NextHop logic by algorithm ---
#  NextHop should be present for both algorithms (used to be only for DelayAware).
# If this is ever < 100%, some samples are invalid or disconnected.
da_nonnull = res.query("Algorithm == 'DelayAware'")["NextHop"].notna().mean()
do_nonnull = res.query("Algorithm == 'DistanceOnly'")["NextHop"].notna().mean()
print("\nNextHop non-null ratio (DelayAware):", da_nonnull)  # Expected: ≈ 1.0
print("NextHop non-null ratio (DistanceOnly):", do_nonnull)  # Also expected: ≈ 1.0

# --- 3) Confirm NextHop is a real neighbor ---
#  This validates the core integrity of the dataset. Each NextHop must be a *physical link*.

# Convert to undirected edge list so neighbor checks are symmetric
e = edges.copy()
e["a"] = e[["From_ID", "To_ID"]].min(axis=1)
e["b"] = e[["From_ID", "To_ID"]].max(axis=1)
e = e.drop_duplicates(subset=["a", "b"])[["a", "b"]]

# Build (Source, NextHop) pairs and canonicalize them
da = res[res["Algorithm"] == "DelayAware"].dropna(subset=["NextHop"]).copy()
da["a"] = da[["Source", "NextHop"]].min(axis=1)
da["b"] = da[["Source", "NextHop"]].max(axis=1)

# This merge attempts to "validate" each (Source, NextHop) pair against true links
ok_neighbors = da.merge(e, on=["a", "b"], how="left")
#  Success rate = % of DelayAware NextHops that are physical neighbors
print("\nNextHop is neighbor ratio:", (~ok_neighbors["a"].isna()).mean())

# --- 4) Measure DelayAware win-rate ---
#  This tells me how often DelayAware actually improves E2E delay.
# The pivot puts the two algorithm results side-by-side per test case.
pivot = res.pivot_table(
    index=["Time", "Scenario", "Source", "Destination"],
    columns="Algorithm",
    values="E2E_Delay_s",
    aggfunc="first"
).dropna()

# Win = DelayAware delay < DistanceOnly delay
win_rate = (pivot["DelayAware"] < pivot["DistanceOnly"]).mean()
print(f"\n DelayAware win-rate: {win_rate:.2%}")

# --- 5) Compare median delays ---
#  Medians give a clearer view of typical behavior (less distorted by outliers).
medians = res.groupby("Algorithm")["E2E_Delay_s"].median()
print("\nMedian E2E delay (s):")
print(medians)

---
---

## Phase 9: Adding Loss and Throughput Maps, Executing the Three-Way Comparison, and Summaries

In [ ]:
# ====================================================================================
# Step 51 — Build Per-Node DropRate Map from Stress Test Results
#
#  Why this step?
# In high-traffic simulations, some satellites get overwhelmed and drop packets.
# I want a quick way to check how congested each satellite was — at each second.
# This drop-rate map will be used later for:
#  • Feature engineering (congestion-awareness)
#  • Analysis (identify choke points)
#  • Routing decisions (avoid congested nodes)
# ====================================================================================

# --- 0) Numerical safety trick ---
# Avoid divide-by-zero if both dropped and served packets = 0.
eps = 1e-12

# --- 1) Load stress test output ---
stress_df = pd.read_csv(OUTPUT_DIR / "stress_test_results.csv")

# --- 2) Normalize the time column name ---
# I'm standardizing to 'Time' so that future joins/merges stay consistent.
if "Time" not in stress_df.columns and "Time" in stress_df.columns:
    stress_df = stress_df.rename(columns={"Time": "Time"})

# --- 3) Compute the DropRate per (satellite, second) ---
# The core metric here: what % of traffic was dropped at each node, per second?
stress_df["DropRate"] = (
    stress_df["Packets_Dropped"] / 
    (stress_df["Packets_Dropped"] + stress_df["Packets_Served"] + eps)
)

# --- 4) Build a fast lookup dictionary: (Time, SatID) → DropRate ---
# DataFrames are great for analysis, but dictionaries are faster for lookup in routing loops.
# I convert the DataFrame into a dense hash map for later use.
drop_rate_map = {
    (int(row.Time), int(row.Satellite_ID)): float(row.DropRate)
    for row in stress_df.itertuples()
}

# --- 5) Confirmation ---
print(f" Drop-rate map ready — {len(drop_rate_map):,} entries")


---

In [ ]:
# ====================================================================================
# Step 52 — Routing Comparison with DropRate & NormThroughput
#
#  Why this step?
# Until now, I compared routing algorithms (DistanceOnly vs DelayAware) purely by delay.
# But real networks also care about:
#   • Reliability (drops) → Path_DropRate
#   • Efficiency (throughput) → NormThroughput
#
# Adding these metrics makes the evaluation fairer and closer to real-world KPIs.
# ====================================================================================

# --- 0) I/O paths -------------------------------------------------------------
RESULTS_PAIRS_PATH = OUTPUT_DIR / "final_routing_comparison_results.csv"
EDGES_PATH         = OUTPUT_DIR / "satellite_link_edge_list.csv"
TIMELINE_PATH      = OUTPUT_DIR / "final_enriched_ml_dataset.csv"
STRESS_PATH        = OUTPUT_DIR / "stress_test_results.csv"

SPEED_OF_LIGHT_KM_S = 299_792.458
eps = 1e-12   # Small constant to avoid division-by-zero

# --- Helper: NormThroughput ---------------------------------------------------
def compute_norm_throughput(drop_rate_series, e2e_delay_series, eps=1e-12):
    """
    Normalized throughput = (1 - drop_rate) / delay.
    
    Why?
    - (1 - drop_rate) → how much traffic actually survives
    - 1 / delay       → how quickly it gets through
    Multiplying them gives a single score that rewards fast + reliable paths.
    """
    drop  = pd.Series(drop_rate_series, copy=True).clip(0, 1)
    delay = pd.Series(e2e_delay_series, copy=True).clip(lower=eps)
    return (1.0 - drop) / delay

# --- 1) Load inputs -----------------------------------------------------------
results_pairs = pd.read_csv(RESULTS_PAIRS_PATH)   # Source–destination test pairs
edges_df      = pd.read_csv(EDGES_PATH)           # Physical links
timeline_df   = pd.read_csv(TIMELINE_PATH)        # Per-node features over time

# --- Delay_s Calculation ------------------------------------------------------
# Why? I need a consistent queue delay estimate for every node in every snapshot.

# Normalize column names
col_ren = {}
if "Time" in timeline_df.columns and "Time" not in timeline_df.columns:
    col_ren["Time"] = "Time"
if "Queue" in timeline_df.columns and "Queue_Length" not in timeline_df.columns:
    col_ren["Queue"] = "Queue_Length"
timeline_df = timeline_df.rename(columns=col_ren)

# Ensure key columns exist
for col in ["Satellite_ID", "Queue_Length"]:
    if col not in timeline_df.columns:
        raise KeyError(f"Required column missing: {col}")

# Clean queue length (avoid negatives/NaNs)
timeline_df["Queue_Length"] = (
    timeline_df["Queue_Length"].astype(float).fillna(0.0).clip(lower=0.0)
)

# Service rate hierarchy: prefer row-level → node-median → fallback
SERVICE_RATE_PPS = 1000
if "Packets_Served" in timeline_df.columns:
    timeline_df["Row_ServiceRate"] = timeline_df["Packets_Served"].astype(float)
    svc_map = (
        timeline_df.groupby("Satellite_ID")["Packets_Served"]
        .median()
        .rename("Node_ServiceRate")
    )
    timeline_df = timeline_df.merge(svc_map, on="Satellite_ID", how="left")
else:
    timeline_df["Row_ServiceRate"]  = np.nan
    timeline_df["Node_ServiceRate"] = np.nan

def _choose_rate(row, fallback=SERVICE_RATE_PPS):
    if np.isfinite(row["Row_ServiceRate"]) and row["Row_ServiceRate"] >= 1:
        return float(row["Row_ServiceRate"])
    if np.isfinite(row["Node_ServiceRate"]) and row["Node_ServiceRate"] >= 1:
        return float(row["Node_ServiceRate"])
    return float(fallback)

timeline_df["ServiceRate"] = timeline_df.apply(_choose_rate, axis=1)
timeline_df["Delay_s"]     = timeline_df["Queue_Length"] / timeline_df["ServiceRate"]

# Diagnostics — why useful? Helps catch silent data issues.
print(f"[Delay_s] non-finite ServiceRate: {(~np.isfinite(timeline_df['ServiceRate'])).mean():.2%}")
print(f"[Delay_s] ServiceRate < 1: {(timeline_df['ServiceRate'] < 1).mean():.2%}")
print(f"[Delay_s] missing Delay_s: {timeline_df['Delay_s'].isna().mean():.2%}")

# --- 2) Build distance-only graph ---------------------------------------------
# Simple baseline: only propagation delay, ignoring queues.
G_dist = nx.from_pandas_edgelist(edges_df, "From_ID", "To_ID", edge_attr=["Distance_km"])
for _, _, data in G_dist.edges(data=True):
    data["prop_s"] = data["Distance_km"] / SPEED_OF_LIGHT_KM_S

# --- 3) Delay-aware graph builder ---------------------------------------------
def build_delay_graph_directed_at_t(t):
    """
    Build a snapshot-specific directed graph:
    - Each edge includes propagation delay + neighbor’s queue delay.
    """
    snap = timeline_df[timeline_df["Time"] == t]
    q_map = snap.set_index("Satellite_ID")["Delay_s"].to_dict()
    G = nx.DiGraph()
    for row in edges_df.itertuples(index=False):
        u, v, d_km = int(row.From_ID), int(row.To_ID), float(row.Distance_km)
        prop = d_km / SPEED_OF_LIGHT_KM_S
        qu = float(q_map.get(u, 0.0))
        qv = float(q_map.get(v, 0.0))
        G.add_edge(u, v, weight=prop + qv, prop_s=prop)
        G.add_edge(v, u, weight=prop + qu, prop_s=prop)
    return G, q_map

# --- 4) Path delay calculator -------------------------------------------------
def e2e_delay_seconds(path, q_map, G):
    """
    End-to-end delay = sum of propagation + sum of queue delays (excluding destination).
    """
    if not path or len(path) < 2:
        return float("inf")
    prop_sum  = sum(G[u][v].get("prop_s", 0.0) for u, v in zip(path[:-1], path[1:]))
    queue_sum = sum(float(q_map.get(n, 0.0)) for n in path[:-1])
    return prop_sum + queue_sum

# --- 5) DropRate map ----------------------------------------------------------
drop_rate_map = {}
try:
    stress_df = pd.read_csv(STRESS_PATH)
    if "Time" not in stress_df.columns and "Time" in stress_df.columns:
        stress_df = stress_df.rename(columns={"Time": "Time"})
    stress_df["DropRate"] = stress_df["Packets_Dropped"] / (
        stress_df["Packets_Dropped"] + stress_df["Packets_Served"] + eps
    )
    drop_rate_map = {(int(r.Time), int(r.Satellite_ID)): float(r.DropRate) for r in stress_df.itertuples()}
except FileNotFoundError:
    print(" No stress test results found → DropRate=0 will be assumed.")

def path_drop_rate(path, t):
    """
    Probability that at least one packet is dropped along the path.
    """
    p_succ = 1.0
    for node in path[1:]:  # exclude source, include relays + dest
        p_succ *= (1.0 - drop_rate_map.get((t, int(node)), 0.0))
    return 1.0 - p_succ

# --- 6) Run experiment --------------------------------------------------------
out_rows = []
pairs_iter = results_pairs[["Time", "Source", "Destination"]].drop_duplicates().itertuples(index=False)

for t, s, d in pairs_iter:
    t, s, d = int(t), int(s), int(d)
    G_delay, qmap = build_delay_graph_directed_at_t(t)

    # DistanceOnly routing
    try:
        p_do = nx.dijkstra_path(G_dist, s, d, weight="prop_s")
        e_do = e2e_delay_seconds(p_do, qmap, G_dist)
        out_rows.append(dict(
            Time=t, Algorithm="DistanceOnly", Source=s, Destination=d,
            E2E_Delay_s=e_do, Hop_Count=len(p_do)-1,
            Path_DropRate=path_drop_rate(p_do, t)
        ))
    except nx.NetworkXNoPath:
        pass

    # DelayAware routing
    try:
        p_da = nx.dijkstra_path(G_delay, s, d, weight="weight")
        e_da = e2e_delay_seconds(p_da, qmap, G_delay)
        out_rows.append(dict(
            Time=t, Algorithm="DelayAware", Source=s, Destination=d,
            E2E_Delay_s=e_da, Hop_Count=len(p_da)-1,
            Path_DropRate=path_drop_rate(p_da, t)
        ))
    except nx.NetworkXNoPath:
        pass

# --- 7) Results & Save --------------------------------------------------------
aug_df = pd.DataFrame(out_rows)
aug_df["Norm_Throughput"] = compute_norm_throughput(
    aug_df["Path_DropRate"], aug_df["E2E_Delay_s"]
)

# Aggregate summary: mean per algorithm
final_comparison = (
    aug_df.groupby("Algorithm", as_index=False)[["E2E_Delay_s","Path_DropRate","Norm_Throughput"]]
          .mean()
          .sort_values("E2E_Delay_s")
)

print("\n--- Final Comparison (averages) ---")
print(final_comparison)

# Save full + summary
out_rows_path = OUTPUT_DIR / "final_routing_comparison_results_v2.csv"
aug_df.to_csv(out_rows_path, index=False)
print(f"Saved detailed results → {out_rows_path}")

out_agg_path = OUTPUT_DIR / "final_routing_comparison_results_v2_summary.csv"
final_comparison.to_csv(out_agg_path, index=False)
print(f"Saved summary → {out_agg_path}")


---

In [ ]:
# ========================================================================================
# Step 53 — Final Summary: Delay, DropRate, and NormThroughput
#
#  Why this step?
# These summaries are the final distilled evidence that compares routing algorithms.
# I want:
#   • Per-time summaries → to show how each algo behaves at each snapshot
#   • Overall summary    → final judgment: which algorithm is better?
#
# These CSVs will go in my thesis + evaluation section.
# ========================================================================================



# --- 1) Load the final results (from Step 52) ----------------------------------
aug = pd.read_csv(OUTPUT_DIR / "final_routing_comparison_results_v2.csv")

# --- 2) Group-by-time summary --------------------------------------------------
# Purpose: track how each algorithm performs across different time snapshots

summary = (
    aug.groupby(["Time", "Algorithm"], as_index=False)
       .agg(
           Mean_E2E=("E2E_Delay_s", "mean"),
           Mean_Drop=("Path_DropRate", "mean"),
           Mean_NormTP=("Norm_Throughput", "mean"),
           Paths=("Source", "count")  # Just counts how many test routes in that group
       )
       .sort_values(["Time", "Algorithm"])
)

print("\n--- Per-Time Summary (head) ---")
display(summary.head())

# --- 3) Overall comparison table -----------------------------------------------
# Purpose: final, single-row-per-algorithm summary of overall performance
# I want this table to appear in my thesis results section.

pivot = (
    aug.groupby("Algorithm", as_index=False)
       .agg(
           Mean_E2E=("E2E_Delay_s", "mean"),
           Mean_Drop=("Path_DropRate", "mean"),
           Mean_NormTP=("Norm_Throughput", "mean"),
           Paths=("Source", "count")
       )
       .sort_values("Mean_E2E")
)

print("\n--- Overall Algorithm Comparison ---")
display(pivot)

# --- 4) Save both summary tables -----------------------------------------------
summary_path = OUTPUT_DIR / "routing_summary_triple_metrics.csv"
pivot_path   = OUTPUT_DIR / "routing_overall_triple_metrics.csv"

summary.to_csv(summary_path, index=False)
pivot.to_csv(pivot_path, index=False)

print(f"\n Saved per-time summary → {summary_path.name}")
print(f" Saved overall metrics summary → {pivot_path.name}")


---

In [ ]:
# ==========================================================================================
# Step 54 — Final Performance Comparison Plots (Triple-Metric)
#
#  Why this step?
# Tables are good for precision, but visual plots make the performance tradeoffs
# crystal clear. Here I compare algorithms across all 3 metrics:
#   • End-to-End Delay (lower is better)
#   • Path Drop-Rate (lower is better)
#   • Normalized Throughput (higher is better)
# ==========================================================================================


print(" Visualizing routing performance on THREE metrics (Delay, DropRate, NormThroughput)...")

# --- 0) Global settings -------------------------------------------------------
plt.rcParams["figure.dpi"] = 120   # High-res plots for publication
USE_LOG_Y_DELAY = False            # Option to log-scale delay axis if skewed
SAVE_FIGS = True                   # Save plots for thesis

# Output folder (robustly defined)
OUTPUT_DIR = Path(OUTPUT_DIR) if 'OUTPUT_DIR' in globals() else Path("output_t_1_sec_analysis")

# --- 1) Load the best available results --------------------------------------
candidates = [
    OUTPUT_DIR / "final_routing_comparison_results_v2.csv", # Preferred (has all 3 metrics)
    OUTPUT_DIR / "final_routing_comparison_results.csv",   # Fallback
]

results_df = None
for p in candidates:
    if p.exists():
        results_df = pd.read_csv(p)
        print(f" Results loaded from: {p.name}")
        break
if results_df is None:
    raise FileNotFoundError("No results file found. Please run Step 52 first.")

# --- 2) Minimal cleaning ------------------------------------------------------
# Remove any invalid rows (e.g., NaNs, negative values)
mask_valid = (
    np.isfinite(results_df["E2E_Delay_s"]) &
    np.isfinite(results_df["Path_DropRate"]) &
    np.isfinite(results_df["Norm_Throughput"]) &
    (results_df["E2E_Delay_s"] >= 0) &
    (results_df["Path_DropRate"] >= 0) & (results_df["Path_DropRate"] <= 1)
)
results_df = results_df[mask_valid].copy()
if results_df.empty:
    raise ValueError("Results are empty after cleaning.")

# --- 3) Aggregate: mean + 95th percentile ------------------------------------
# Why? Mean shows “typical” performance. P95 shows “near worst-case” performance.
overall = (
    results_df.groupby("Algorithm", as_index=False)
              .agg(
                   Mean_E2E=("E2E_Delay_s", "mean"),
                   P95_E2E =("E2E_Delay_s", lambda s: np.percentile(s, 95)),
                   Mean_Drop=("Path_DropRate", "mean"),
                   P95_Drop =("Path_DropRate", lambda s: np.percentile(s, 95)),
                   Mean_NormTP=("Norm_Throughput", "mean"),
                   Paths=("Source", "count")
               )
              .sort_values(["Mean_E2E", "Mean_Drop", "Mean_NormTP"],
                           ascending=[True, True, False])
)

print("\n--- Overall (all scenarios & times) ---")
display(overall)

# --- 4) Final Plots -----------------------------------------------------------
fig, axes = plt.subplots(1,3, figsize=(14,4), constrained_layout=True)

# Plot 1 — Delay: lower is better
overall.plot(x="Algorithm", y="Mean_E2E", kind="bar", ax=axes[0], legend=False,
             title="Mean E2E Delay (s)")
axes[0].set_ylabel("Seconds")
if USE_LOG_Y_DELAY:
    axes[0].set_yscale("log")

# Plot 2 — Drop-Rate: lower is better
overall.plot(x="Algorithm", y="Mean_Drop", kind="bar", ax=axes[1], legend=False,
             title="Mean Path Drop-Rate")
axes[1].set_ylabel("Percent (%)")
axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=1))

# Plot 3 — Norm Throughput: higher is better
overall.plot(x="Algorithm", y="Mean_NormTP", kind="bar", ax=axes[2], legend=False,
             title="Mean Normalized Throughput")
axes[2].set_ylabel("Normalized Throughput (1/s)")

# Styling
for ax in axes:
    ax.grid(axis='y', linestyle='--', alpha=0.6)

if SAVE_FIGS:
    out_file = OUTPUT_DIR / "overall_algorithms_triple_metrics.png"
    plt.savefig(out_file, dpi=200, bbox_inches="tight")
    print(" Saved:", out_file)

plt.show()


---

In [ ]:
# ==========================================================================================
# Step 55 — Post-Hoc Diagnostic: DelayAware vs DistanceOnly
#
#  Why this step?
# The headline win-rate (DA > DO) is good, but it doesn’t explain the *mechanics*.
# This diagnostic digs deeper:
#   • How big are the delay savings in seconds?
#   • Does DelayAware trade more hops for less delay?
#   • How often does DelayAware find a nearly congestion-free path?
# ==========================================================================================


# --- 1) Load results robustly ------------------------------------------------
res = pd.read_csv(OUTPUT_DIR / "final_routing_comparison_results.csv").copy()

# Old files might use "Time"; normalize for consistency.
if "Time" not in res.columns and "Time" in res.columns:
    res = res.rename(columns={"Time": "Time"})

# Use Scenario if present; otherwise just Time + endpoints
index_cols = ["Time", "Source", "Destination"]
if "Scenario" in res.columns:
    index_cols.insert(1, "Scenario")

# --- 2) Pivot for direct side-by-side comparison -----------------------------
# Why pivot? It lines up DistanceOnly and DelayAware results row-by-row,
# so we can compare them directly for each route tested.
pairs = (
    res.pivot_table(
        index=index_cols,
        columns="Algorithm",
        values=["E2E_Delay_s", "Hop_Count"],
        aggfunc="first"
    )
    .dropna()  # Only keep pairs where both algorithms produced a valid path
)

# --- 3) Delay savings --------------------------------------------------------
# Positive Delta_s means DelayAware was faster on that route.
pairs["Delta_s"] = (
    pairs["E2E_Delay_s"]["DistanceOnly"] -
    pairs["E2E_Delay_s"]["DelayAware"]
)
win_rate = (pairs["Delta_s"] > 0).mean()
print(f"Win-rate (DelayAware better): {100*win_rate:.1f}%")

# A full distribution summary shows not just the mean but also the
# tails (90th, 95th, 99th percentiles). This answers: *how big can the gains get?*
print("\n--- Summary of Delay Savings (seconds) ---")
print(pairs["Delta_s"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# --- 4) Hop-count trade-off --------------------------------------------------
# Hypothesis: DA achieves lower delay by using *more hops*,
# detouring around congested nodes. This is expected behavior.
pairs["Delta_hops"] = (
    pairs["Hop_Count"]["DelayAware"] -
    pairs["Hop_Count"]["DistanceOnly"]
)
print("\n--- Hop Count Difference ---")
print(f"ΔHops (DA - DO): median = {pairs['Delta_hops'].median()}, "
      f"mean = {pairs['Delta_hops'].mean():.2f}")

# --- 5) Congestion-free path frequency ---------------------------------------
# If E2E delay < 50 ms on an intercontinental path, queueing is negligible.
# This check tells us: *How often does DA find a “perfectly clean” path?*
near_prop = (pairs["E2E_Delay_s"]["DelayAware"] < 0.05).mean()
print("\n--- Congestion-Free Paths ---")
print(f"Share of DelayAware paths with E2E < 50 ms: {100*near_prop:.1f}%")


---

## Phase 10: Building the Supervised 'Next-Hop' Decision Dataset and Data Alignment

In [ ]:

# ==========================================================================================
# Step 56 — Load All Data for Model Training (Robust)
#
#  Why this step?
# Training an ML model requires three aligned inputs:
#   1) Routing results → contain the teacher’s decisions (NextHop = target variable).
#   2) Enriched features → contain the congestion/time-varying state of each node.
#   3) Edge list → contains the static topology (who can connect to whom).
#
# This script loads them all, validates schema, enforces correct dtypes,
# and checks that timestamps overlap — ensuring that features and labels can
# actually be paired during supervised learning.
# ==========================================================================================


print(" Loading all datasets for machine learning...")

# --- 1) Load Routing Results --------------------------------------------------
# This file has the "ground truth" NextHop decisions from the DelayAware algorithm.
# These are the *labels* that the ML model will try to imitate.
routing_candidates = [
    OUTPUT_DIR / "final_routing_comparison_with_nexthop.csv",  # Preferred: explicit NextHop
    OUTPUT_DIR / "final_routing_comparison_results.csv",       # Fallback if above missing
]

routing_df = None
for path in routing_candidates:
    if path.exists():
        routing_df = pd.read_csv(path)
        print(f" Loaded routing results from: {path.name} ({len(routing_df):,} rows)")
        break
if routing_df is None:
    raise FileNotFoundError(" No routing results found. Please run the main routing experiment first.")

# The NextHop column is the label for ML — warn if absent.
has_nexthop = "NextHop" in routing_df.columns
if not has_nexthop:
    print(" Warning: 'NextHop' not found → must be reconstructed later from teacher paths.")

# --- 2) Load Features and Edge List ------------------------------------------
# Features = inputs (what the model sees).
# Edges = topology (hard constraint: candidate NextHops must be neighbors).
features_path = OUTPUT_DIR / "final_enriched_ml_dataset.csv"
edges_path    = OUTPUT_DIR / "satellite_link_edge_list.csv"

features_df = pd.read_csv(features_path)
edges_df    = pd.read_csv(edges_path)

print(f" Loaded features dataset ({len(features_df):,} rows)")
print(f" Loaded network edge list ({len(edges_df):,} rows)")

# --- 3) Schema Checks + Fallbacks --------------------------------------------
# These checks prevent subtle bugs when merging later.

# Routing schema must contain key experiment outputs.
required_routing = {"Time", "Scenario", "Algorithm", "Source", "Destination", "E2E_Delay_s", "Hop_Count"}
missing_routing = sorted(required_routing - set(routing_df.columns))
if missing_routing:
    raise ValueError(f"routing_df is missing columns: {missing_routing}")

# Features schema must contain the essential congestion/neighbor-aware columns.
required_features = {
    "Time", "Satellite_ID", "Delay_s", "Queue_Length",
    "NumPackets_Generated", "Packets_Dropped",
    "Avg_Neighbor_Delay", "Busy_Neighbor_Ratio"
}
missing_features = sorted(required_features - set(features_df.columns))
if missing_features:
    # Safer to fill with 0 than to crash — but this indicates data gaps
    print(f" Warning: features_df is missing columns: {missing_features}")
    for col in missing_features:
        features_df[col] = 0

# Edge list schema must always have the connectivity info.
required_edges = {"From_ID", "To_ID", "Distance_km"}
missing_edges = sorted(required_edges - set(edges_df.columns))
if missing_edges:
    raise ValueError(f"edges_df is missing required columns: {missing_edges}")

# --- 4) Type Normalization ----------------------------------------------------
# Why force types? Ensures join keys are consistent (int32),
# saves memory, and prevents hard-to-debug mismatches.
if "Time" in features_df.columns:
    features_df["Time"] = features_df["Time"].astype("int32")

for col in ["Source", "Destination"]:
    if col in routing_df.columns:
        routing_df[col] = routing_df[col].astype("int32")

if "Satellite_ID" in features_df.columns:
    features_df["Satellite_ID"] = features_df["Satellite_ID"].astype("int32")
edges_df["From_ID"] = edges_df["From_ID"].astype("int32")
edges_df["To_ID"]   = edges_df["To_ID"].astype("int32")

# --- 5) Time Overlap Check ----------------------------------------------------
# Why? Routing results are at selected snapshots, features are continuous.
# Without overlap, you cannot pair features (X) with NextHop labels (y).
common_times = sorted(set(routing_df["Time"]).intersection(set(features_df["Time"])))
if common_times:
    t0 = common_times[0]
    sample = features_df[features_df["Time"] == t0][["Satellite_ID", "Delay_s"]].head(3)
    print(f"\n Sanity check passed. Shared timestamps found (e.g., t={t0}):\n{sample}")
else:
    print("\n Warning: No shared timestamps found! Features and labels may be misaligned.")

# --- 6) Quick Preview ---------------------------------------------------------
# A final visual sanity check before moving on to dataset assembly.
print("\n Routing Results (sample):")
display(routing_df.head(3))

print("\n Enriched Features (sample):")
display(features_df.head(3))

print("\n Network Topology (sample):")
display(edges_df.head(3))


---

In [ ]:

# ==========================================================================================
# Step 57 — Create a Supervised Learning Dataset (Multi-Hop Neighbor-Ranking)
#
#  Why this step?
# We want to train an ML model to predict the teacher’s routing decision (NextHop).
# For each hop of each DelayAware path, we generate:
#   - candidate neighbors (inputs, features)
#   - the chosen neighbor (output, Label=1)
#
# This converts the routing experiment into a supervised classification dataset.
# ==========================================================================================


print("Creating the supervised learning dataset (multi-hop, neighbor-ranking)...")

# --- 0) Inputs required -------------------------------------------------------
assert "timeline_df" in globals(), "timeline_df not found → load final_enriched_ml_dataset first"
assert "edges_df"    in globals(), "edges_df not found → load satellite_link_edge_list.csv first"

# --- 1) Neighbor map (undirected) ---------------------------------------------
# Why? Fast lookup: given node u, get all physical neighbors v.
nbrs = {}
for u, v in edges_df[["From_ID", "To_ID"]].astype(int).itertuples(index=False):
    nbrs.setdefault(int(u), []).append(int(v))
    nbrs.setdefault(int(v), []).append(int(u))

# --- 2) Position index (for distance-to-destination heuristic) ----------------
# Why? Helps create geometric features: how much closer/further a neighbor is to dest.
pos_idx = positions_df.set_index("Node_ID")[["Latitude_deg", "Longitude_deg"]].copy()
pos_idx.columns = ["lat", "lon"]

def haversine_km(a, b):
    """Compute great-circle distance (km) between satellites a and b."""
    lat1, lon1 = pos_idx.loc[a, ["lat","lon"]]
    lat2, lon2 = pos_idx.loc[b, ["lat","lon"]]
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = p2 - p1
    dl = math.radians(lon2 - lon1)
    h = (math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2)
    return 2*R*math.asin(min(1.0, math.sqrt(h)))

# --- 3) Teacher paths (DelayAware) --------------------------------------------
# These are the "labels": the ground truth paths from the DelayAware algorithm.
teacher = {}
tp_file = OUTPUT_DIR / "teacher_paths.json"

if tp_file.exists():
    print("Loading teacher paths from teacher_paths.json ...")
    with open(tp_file) as f:
        for r in json.load(f):
            teacher[(int(r["Time"]), int(r["Source"]), int(r["Destination"]))] = [int(x) for x in r["Path"]]
else:
    print("No teacher_paths.json found → recomputing (slower)...")
    for t in SNAPSHOTS_TO_ANALYZE:
        snap = timeline_df[timeline_df["Time"] == t]
        if snap.empty: continue
        G_dist, G_delay, qmap = build_graphs_for_snapshot(snap, edges_df)
        for (ra, rb) in INTERCONTINENTAL_SCENARIOS:
            pairs = _pairs_for(t, ra, rb, NUM_PAIRS_TO_TEST)
            if not pairs:
                nodes_a = snap.loc[snap["Region"] == ra, "Satellite_ID"].tolist()
                nodes_b = snap.loc[snap["Region"] == rb, "Satellite_ID"].tolist()
                pairs = [(rng.choice(nodes_a), rng.choice(nodes_b))] if nodes_a and nodes_b else []
            for s, d in pairs:
                try:
                    p = nx.dijkstra_path(G_delay, int(s), int(d), weight="weight")
                    teacher[(int(t), int(s), int(d))] = [int(x) for x in p]
                except nx.NetworkXNoPath:
                    pass

print("Teacher path count:", len(teacher))

# --- 4) Build candidate rows --------------------------------------------------
# Each (u,v) neighbor candidate at hop step_i becomes a row.
rows, mismatch = [], 0

for key, path in teacher.items():

    # Robust unpacking
    # (handles any extra metadata stored inside teacher keys)
    t = int(key[0])
    s = int(key[1])
    d = int(key[2])

    # ----------------------------------------------------------------
    # Keep full history up to current snapshot
    # so temporal features remain causal + train-serving consistent
    # ----------------------------------------------------------------
    snap = timeline_df[
        timeline_df["Time"] <= t
    ].copy()

    if snap.empty:
        continue

    # Ensure proper temporal ordering
    snap = snap.sort_values(
        ["Satellite_ID", "Time"]
    ).copy()

    # Short-term queue trend
    # Queue_Trend(t) = Queue_Length(t) - Queue_Length(t-1)
    snap["Queue_Trend"] = (
        snap.groupby("Satellite_ID")["Queue_Length"]
        .diff()
        .fillna(0.0)
    )

    # Build graphs + feature-serving maps
    G_dist, G_delay, qmap = build_graphs_for_snapshot(
        snap,
        edges_df
    )

    # Keep only current snapshot rows afterward
    snap = snap[
        snap["Time"] == t
    ].copy()

    for step_i in range(len(path) - 1):

        u = int(path[step_i])
        best_v = int(path[step_i + 1])

        neighbors = nbrs.get(u, [])

        if not neighbors:
            continue

        if best_v not in neighbors:
            mismatch += 1
            continue

        du = haversine_km(
            u,
            d
        )  # distance u→destination

        for v in neighbors:

            v = int(v)

            # Pure propagation delay
            prop_uv_s = float(
                G_dist[u][v]["weight"]
            )

            # Queueing delay at candidate node
            qv_s = float(
                qmap.get(v, 0.0)
            )

            # Geometric progress toward destination
            dv = haversine_km(v, d)

            rows.append((
                t,
                s,
                d,
                step_i,
                u,
                v,
                prop_uv_s,
                qv_s,
                float(dv - du),
                int(v == best_v)
            ))

train_cols = [
    "Time",
    "Source",
    "Destination",
    "PathStep",
    "U",
    "V",
    "prop_uv_s",
    "queue_v_s",
    "delta_km_to_dest",
    "Label"
]

train_df = pd.DataFrame(
    rows,
    columns=train_cols
)
# --- 5) Quality checks --------------------------------------------------------
print("\nDataset Quality Report:")
print("  total candidate rows:", len(train_df))
print("  positives (teacher-chosen):", int(train_df["Label"].sum()))
print("  mismatches (teacher best_v not in neighbors):", mismatch)

grp = train_df.groupby(["Time","Source","Destination","PathStep"])["Label"].sum()
print("  groups with NO positive:", int((grp == 0).sum()))
print("  groups with >1 positive:", int((grp > 1).sum()))

display(train_df.head())

# --- 6) Save ------------------------------------------------------------------
out_path = OUTPUT_DIR / "train_neighbor_rank.csv"
train_df.to_csv(out_path, index=False)
print(f"\nSaved training dataset to:\n  {out_path.resolve()}")


---

In [ ]:
# ==========================================================================================
# Step 58 — Data Alignment & Filtering – Ensuring Valid Candidates
#
#  Why this step?
# The candidate dataset from Step 57 may include satellites not in the experiment
# or inconsistent IDs. Before ML training, we must:
#   1. Align with simulation positions and results (pos_df, full_sim_df).
#   2. Filter out invalid nodes (keep only those that appear in both datasets).
#   3. Save a clean version for downstream ML training.
# ==========================================================================================


print("Aligning candidate labels to experiment nodes...")

# --- 0) Load candidate labels (from Step 57) ---------------------------------
cand_path = OUTPUT_DIR / "train_neighbor_rank.csv"
if not cand_path.exists():
    raise FileNotFoundError(f"Expected file not found: {cand_path}")

candidate_labels_df = pd.read_csv(cand_path)

# Standardize column names → consistency across all steps
if "U" in candidate_labels_df.columns:
    candidate_labels_df.rename(columns={"U": "Source"}, inplace=True)
if "V" in candidate_labels_df.columns:
    candidate_labels_df.rename(columns={"V": "Candidate_NextHop"}, inplace=True)

# --- 1) Ensure pos_df is available -------------------------------------------
# pos_df gives static info (latitude/longitude) of satellites.
if "pos_df" not in globals():
    if "positions_df" in globals():
        pos_df = positions_df.copy()
        # Normalize column names
        if "Latitude" not in pos_df.columns and "Latitude_deg" in pos_df.columns:
            pos_df.rename(columns={"Latitude_deg": "Latitude"}, inplace=True)
        if "Longitude" not in pos_df.columns and "Longitude_deg" in pos_df.columns:
            pos_df.rename(columns={"Longitude_deg": "Longitude"}, inplace=True)
        # Keep only essential fields
        pos_df = pos_df[["Satellite_ID", "Latitude", "Longitude"]].drop_duplicates("Satellite_ID")
    else:
        raise ValueError("Neither pos_df nor positions_df found. Load positions first!")

# --- 2) Ensure full_sim_df is available --------------------------------------
# full_sim_df gives dynamic info (queues, delays, drops) for satellites over time.
if "full_sim_df" not in globals():
    fs_path = OUTPUT_DIR / "full_simulation_results.csv"
    if not fs_path.exists():
        raise FileNotFoundError(f"full_simulation_results.csv not found in {OUTPUT_DIR}")
    full_sim_df = pd.read_csv(fs_path)

# Normalize time column name
if "Time" not in full_sim_df.columns and "Time" in full_sim_df.columns:
    full_sim_df.rename(columns={"Time": "Time"}, inplace=True)

# Ensure Delay_s exists (fallback if missing)
if "Delay_s" not in full_sim_df.columns and "Queue_Length" in full_sim_df.columns:
    full_sim_df["Delay_s"] = full_sim_df["Queue_Length"] / 1000.0  # Fallback scaling

# --- 3) Build valid node set --------------------------------------------------
# Only keep satellites that are present in BOTH static positions and dynamic results.
valid_nodes = set(pos_df["Satellite_ID"]).intersection(full_sim_df["Satellite_ID"])

# --- 4) Filter candidate labels ----------------------------------------------
# Defensive step: remove duplicate columns if any
candidate_labels_df = candidate_labels_df.loc[:, ~candidate_labels_df.columns.duplicated()].copy()

before = len(candidate_labels_df)

# Keep only rows where Source, Destination, and Candidate_NextHop are valid nodes
candidate_labels_df = candidate_labels_df[
    candidate_labels_df["Source"].isin(valid_nodes) &
    candidate_labels_df["Destination"].isin(valid_nodes) &
    candidate_labels_df["Candidate_NextHop"].isin(valid_nodes)
].copy()

after = len(candidate_labels_df)
print(f"Filtered candidate labels to experiment nodes: kept {after} / {before}")

# --- 5) Save -----------------------------------------------------------------
candidate_labels_out = OUTPUT_DIR / "ml_candidate_labels.csv"
candidate_labels_df.to_csv(candidate_labels_out, index=False)
print(f"Candidate labels saved to: {candidate_labels_out}")

---
---

## Phase 11: Training the Classifier Model and Final Evaluation of the Routing Agent

In [ ]:
# ==========================================================================================
# Step 59 — Sanity Check: Validate Inter-Satellite Link (ISL) Distances
#
# Why this matters:
# In a real LEO constellation, inter-satellite laser links (ISLs) have a maximum
# feasible distance (≈3000 km at ~1200 km altitude). Any edge longer than this
# is either a geometry bug or an unrealistic link. Allowing those into training
# would poison the dataset and make the ML agent learn from impossible scenarios.
# ==========================================================================================

DISTANCE_THRESHOLD_KM = 3000.0  # Physical feasibility cap

# --- 1) Topology validation (edges_df) ---------------------------------------
# Here I confirm that the *core network definition* (edges_df) never includes
# a link longer than 3000 km. This is the strongest guardrail: if the raw graph
# itself is bad, every downstream analysis will be corrupted.
assert edges_df["Distance_km"].max() <= DISTANCE_THRESHOLD_KM + 1e-9, \
    "Error: Found edges longer than 3000 km in edges_df! Check distance calculations."

# --- 2) Feature dataset validation (features_df) ------------------------------
# The ML features are per-hop (u→v). They must only reference links that are
# physically valid edges. A merge against edges_df catches two problems:
#   (a) a hop longer than 3000 km, or
#   (b) a hop that isn’t even in the official edge list.
if {"u", "v"}.issubset(features_df.columns):
    merged = features_df.merge(
        edges_df[["From_ID", "To_ID", "Distance_km"]]
               .rename(columns={"From_ID": "u", "To_ID": "v"}),
        on=["u", "v"],
        how="left"
    )

    # Fail fast if any impossible or missing links are detected
    assert merged["Distance_km"].max() <= DISTANCE_THRESHOLD_KM + 1e-9, \
        "Error: features_df contains edges >3000 km or edges not in edges_df!"

    # --- 2.1) Neighbor consistency check --------------------------------------
    # Every NextHop candidate must be a true neighbor in edges_df.
    nh_check = features_df.merge(
        edges_df[["From_ID", "To_ID"]].rename(columns={"From_ID": "u", "To_ID": "v"}).drop_duplicates(),
        on=["u", "v"],
        how="left",
        indicator=True
    )
    neighbor_ratio = (nh_check["_merge"] == "both").mean()
    print(f" NextHop-is-neighbor ratio: {neighbor_ratio:.6f} (should be 1.0)")

    assert (nh_check["_merge"] == "both").all(), \
        "Error: Some (u,v) pairs in features_df are not actual neighbors in edges_df!"

print(" Sanity check passed: All ISL distances are valid and NextHop pairs are consistent.")



In [ ]:
# ================================================================
# Step 60 — Model Training (Classifier Approach with Random Forest)
#
# Why I’m doing this:
# here I frame the problem as classification: for each candidate next-hop, the
# label is 1 if it matches the teacher’s (DelayAware) choice, else 0.
# Basically, the model is learning to mimic the teacher’s routing
# decisions hop by hop.
#
# Outputs from this step:
#   - rf_classifier_model.joblib  → trained model + metadata
#   - rf_classifier_test_group_winners.csv → group-level evaluation results
# ================================================================

# ----------------------------------------------------------------
# Setup: output folder
# ----------------------------------------------------------------
# I keep all results under a dedicated folder so reruns won’t
# create a mess in the notebook directory.
OUTPUT_DIR = Path("output_t_1_sec_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ================================================================
# Helper Functions (feature engineering)
# ================================================================
def _haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance (km) between two lat/lon points."""
    # Using haversine because it’s simple and good enough
    # for satellite-level distances on Earth’s surface.
    R = 6371.0
    lat1 = np.radians(lat1.astype(float)); lon1 = np.radians(lon1.astype(float))
    lat2 = np.radians(lat2.astype(float)); lon2 = np.radians(lon2.astype(float))
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return R * (2*np.arctan2(np.sqrt(a), np.sqrt(1-a)))


def add_degree_features(df: pd.DataFrame, edges_df: pd.DataFrame) -> pd.DataFrame:
    """Attach node degrees for source and candidate nodes."""
    # Degree gives a rough sense of how “connected” a satellite is.
    # More neighbors = potentially more routing options.
    req = {"Source", "Candidate_NextHop"}
    assert req.issubset(df.columns), f"missing {req - set(df.columns)}"

    # Normalize edge-list column names
    e = edges_df.copy()
    rename = {}
    if "From_ID" not in e.columns and "u" in e.columns: rename["u"] = "From_ID"
    if "To_ID"   not in e.columns and "v" in e.columns: rename["v"] = "To_ID"
    e.rename(columns=rename, inplace=True)

    assert {"From_ID", "To_ID"}.issubset(e.columns), "edges_df must have From_ID/To_ID"

    # Count degree by stacking both endpoints
    deg = pd.concat([e["From_ID"], e["To_ID"]]).value_counts().rename_axis("Node").reset_index(name="degree")

    # Merge back to candidates
    df = df.merge(deg.rename(columns={"Node": "Source", "degree": "Source_Degree"}), on="Source", how="left")
    df = df.merge(deg.rename(columns={"Node": "Candidate_NextHop", "degree": "Candidate_Degree"}),
                  on="Candidate_NextHop", how="left")
    return df


def add_congestion_features(df: pd.DataFrame, sim_df: pd.DataFrame) -> pd.DataFrame:
    """Add queue delays for Source and Candidate, plus their difference."""
    # Even if a link is short, it’s not useful if the candidate is congested.
    s = sim_df.copy()

    # Sometimes delay isn’t given directly → approximate from queue length.
    # I assume 1000 packets/sec service rate (a design choice).
    if "Delay_s" not in s.columns and "Queue_Length" in s.columns:
        s["Delay_s"] = s["Queue_Length"] / 1000.0

    req = {"Time", "Satellite_ID", "Delay_s"}
    assert req.issubset(s.columns), f"sim_df needs {req}"

    q_src = s[["Time", "Satellite_ID", "Delay_s"]].rename(
        columns={"Satellite_ID": "Source", "Delay_s": "Source_Queue_Delay"})
    q_cnd = s[["Time", "Satellite_ID", "Delay_s"]].rename(
        columns={"Satellite_ID": "Candidate_NextHop", "Delay_s": "Candidate_Queue_Delay"})

    df = df.merge(q_src, on=["Time", "Source"], how="left")
    df = df.merge(q_cnd, on=["Time", "Candidate_NextHop"], how="left")

    # A relative signal: is candidate’s queue worse than source’s?
    df["Delay_Delta"] = df["Candidate_Queue_Delay"] - df["Source_Queue_Delay"]

    return df


def add_temporal_features(df: pd.DataFrame,
                          sim_df: pd.DataFrame) -> pd.DataFrame:
    """Add short-term queue trend features for Source and Candidate."""

    # Queue trend captures whether congestion is growing or shrinking.
    # I use only past information (t - 1 → t) to avoid future leakage.

    req = {"Time", "Satellite_ID", "Queue_Length"}
    assert req.issubset(sim_df.columns), f"sim_df needs {req}"

    # Sort by satellite and time so temporal differences are meaningful.
    s = sim_df.sort_values(["Satellite_ID", "Time"]).copy()

    # First-order temporal difference:
    # positive  → queue increasing
    # negative  → queue draining
    s["Queue_Trend"] = (
        s.groupby("Satellite_ID")["Queue_Length"]
        .diff()
    )

    # Keep only rows where previous observation exists.
    t = s[["Time", "Satellite_ID", "Queue_Trend"]].dropna()

    # Attach trend feature for current Source satellite.
    df = df.merge(
        t.rename(columns={
            "Satellite_ID": "Source",
            "Queue_Trend": "Source_Queue_Trend"
        }),
        on=["Time", "Source"],
        how="left"
    )

    # Attach trend feature for candidate next-hop satellite.
    df = df.merge(
        t.rename(columns={
            "Satellite_ID": "Candidate_NextHop",
            "Queue_Trend": "Candidate_Queue_Trend"
        }),
        on=["Time", "Candidate_NextHop"],
        how="left"
    )

    return df


def add_geometric_features(df: pd.DataFrame, pos_df: pd.DataFrame, edges_df: pd.DataFrame) -> pd.DataFrame:
    """Add propagation delays and closeness flag using geometry."""
    # Distance still matters. Even if congestion is low, a far candidate
    # introduces propagation delay. So I give the model this signal.
    req = {"Source", "Candidate_NextHop", "Destination"}
    assert req.issubset(df.columns), f"missing {req - set(df.columns)}"

    # Bring in positions
    p = pos_df.rename(columns={"Satellite_ID": "sid", "Latitude": "lat", "Longitude": "lon"})
    s = p.rename(columns={"sid": "Source", "lat": "s_lat", "lon": "s_lon"})[["Source", "s_lat", "s_lon"]]
    c = p.rename(columns={"sid": "Candidate_NextHop", "lat": "c_lat", "lon": "c_lon"})[["Candidate_NextHop", "c_lat", "c_lon"]]
    d = p.rename(columns={"sid": "Destination", "lat": "d_lat", "lon": "d_lon"})[["Destination", "d_lat", "d_lon"]]

    df = df.merge(s, on="Source", how="left").merge(c, on="Candidate_NextHop", how="left").merge(d, on="Destination", how="left")

    # Candidate vs Source propagation delay to destination
    df["Prop_Delay_Cand_To_Dest"]   = _haversine_km(df["c_lat"], df["c_lon"], df["d_lat"], df["d_lon"]) / SPEED_OF_LIGHT_KM_S
    df["Prop_Delay_Source_To_Dest"] = _haversine_km(df["s_lat"], df["s_lon"], df["d_lat"], df["d_lon"]) / SPEED_OF_LIGHT_KM_S
    df["Is_Closer"] = (df["Prop_Delay_Cand_To_Dest"] < df["Prop_Delay_Source_To_Dest"]).astype("int8")

    # Source→Candidate propagation from edge-list
    e = edges_df.copy()
    rename = {}
    if "From_ID" not in e.columns and "u" in e.columns: rename["u"] = "From_ID"
    if "To_ID"   not in e.columns and "v" in e.columns: rename["v"] = "To_ID"
    if "Distance_km" not in e.columns:
        for cand in ("dist_km", "distance_km", "DistanceKm"):
            if cand in e.columns: rename[cand] = "Distance_km"
    e.rename(columns=rename, inplace=True)

    assert {"From_ID", "To_ID", "Distance_km"}.issubset(e.columns), "edges_df must have From_ID/To_ID/Distance_km"

    # Build lookup dictionary (make it symmetric)
    dist_map = {(int(u), int(v)): float(dkm) for u, v, dkm in e[["From_ID", "To_ID", "Distance_km"]].itertuples(index=False)}
    dist_map.update({(v, u): d for (u, v), d in list(dist_map.items())})

    df["Prop_Delay_Source_To_Cand"] = [
        dist_map.get((int(sid), int(cid)), np.nan)
        for sid, cid in zip(df["Source"], df["Candidate_NextHop"])
    ]
    df["Prop_Delay_Source_To_Cand"] = df["Prop_Delay_Source_To_Cand"] / SPEED_OF_LIGHT_KM_S

    # Drop temp lat/lon columns
    df.drop(columns=["s_lat", "s_lon", "c_lat", "c_lon", "d_lat", "d_lon"], inplace=True, errors="ignore")

    return df

print("Feature helpers ready ✓ (degree, geometric, congestion, temporal)")

# ================================================================
# 1) Label preparation (Teacher = DelayAware) — only PathStep==0
# ================================================================
cand = pd.read_csv(OUTPUT_DIR / "ml_candidate_labels.csv")
res  = pd.read_csv(OUTPUT_DIR / "final_routing_comparison_results.csv")

# Keep only the first hop decision (source’s immediate next-hop)
cand = cand[cand["PathStep"] == 0].copy()

# Teacher labels come from DelayAware algorithm
teacher = (res[res["Algorithm"] == "DelayAware"]
           .loc[:, ["Time", "Source", "Destination", "NextHop"]]
           .rename(columns={"NextHop": "Teacher_NextHop"}))

# Merge to align candidates with teacher’s choice
df = cand.merge(teacher, on=["Time", "Source", "Destination"], how="inner")
df["y_from_teacher"] = (df["Candidate_NextHop"] == df["Teacher_NextHop"]).astype("int8")

# Sanity: consistent data types
for c in ["Time","Source","Destination","Candidate_NextHop"]:
    cand[c] = cand[c].astype(int)
for c in ["Time","Source","Destination","NextHop"]:
    res[c] = res[c].astype(int)

# If label already existed, validate it
if "Label" in df.columns:
    assert (df["Label"].astype("int8") == df["y_from_teacher"]).all(), \
        "Mismatch between Label and Teacher in PathStep==0"
    df["y"] = df["Label"].astype("int8")
else:
    df["y"] = df["y_from_teacher"]

# Check: each (Time, Source, Destination) group must have exactly one positive
grp = df.groupby(["Time", "Source", "Destination"])["y"].sum()
assert (grp == 1).all(), "Each group must have exactly one y=1 (PathStep==0)."

# Save labeled dataset for inspection
df.to_csv(OUTPUT_DIR / "ml_classifier_labeled_candidates.csv", index=False)
print(f"Labels OK — groups={len(grp)}, positives={int(df['y'].sum())}, rows={len(df)}")

# ================================================================
# 2) Feature engineering
# ================================================================
required_funcs = [
    "add_degree_features",
    "add_geometric_features",
    "add_congestion_features",
    "add_temporal_features"
]
for fn in required_funcs:
    assert fn in globals(), f"Function {fn} is missing — run feature steps first."

for var in ["pos_df", "edges_df", "full_sim_df"]:
    assert var in globals(), f"{var} not defined — run data loading first."

# Apply all feature functions
featured = df.copy()
featured = add_degree_features(featured, edges_df)
featured = add_geometric_features(featured, pos_df, edges_df)
featured = add_congestion_features(featured, full_sim_df)
featured = add_temporal_features(featured, full_sim_df)

feature_columns = [
    "Source_Degree", "Candidate_Degree",
    "Prop_Delay_Source_To_Cand", "Prop_Delay_Cand_To_Dest", "Prop_Delay_Source_To_Dest",
    "Is_Closer", "Source_Queue_Delay", "Candidate_Queue_Delay", "Delay_Delta",
    "Source_Queue_Trend", "Candidate_Queue_Trend"
]

missing = [c for c in feature_columns if c not in featured.columns]
assert not missing, f"Missing feature columns: {missing}"

final_cls = featured.dropna(subset=feature_columns + ["y"]).copy()
X = final_cls[feature_columns]
y = final_cls["y"].astype("int8")

print(f"Features ready — X={X.shape}, positives={int(y.sum())}, negatives={int((1-y).sum())}")

# ================================================================
# 3) Group-aware train/test split (no leakage)
# ================================================================
# Important: split by groups (Time, Source, Destination) so that
# the same routing problem doesn’t leak into both train and test.
groups = (final_cls["Time"].astype(str) + "_" +
          final_cls["Source"].astype(str) + "_" +
          final_cls["Destination"].astype(str))

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))

Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
ytr, yte = y.iloc[tr_idx], y.iloc[te_idx]

print(f"Split → Train: {len(Xtr)} | Test: {len(Xte)} | Groups: {groups.nunique()} total")

# ================================================================
# 4) Train RandomForestClassifier + offline evaluation
# ================================================================
# Why RF? Strong baseline, handles nonlinearities, minimal tuning.
clf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,        # avoid overfitting tiny groups
    class_weight="balanced",   # compensate for class imbalance
    n_jobs=-1,                 # use all CPU cores
    random_state=42,
    oob_score=True             # sanity check via out-of-bag samples
)
clf.fit(Xtr, ytr)
print("OOB score:", getattr(clf, "oob_score_", None))

# Sample-level report (basic precision/recall/F1)
pred = clf.predict(Xte)
print("=== Classification Report (sample-level) ===")
print(classification_report(yte, pred, digits=3))

# Group-level metric: how often the top predicted candidate
# matches the teacher’s choice (this is what really matters).
te_df = final_cls.iloc[te_idx].copy()
te_df["p1"] = clf.predict_proba(Xte)[:, 1]
winners = te_df.loc[
    te_df.groupby(["Time", "Source", "Destination"])["p1"].idxmax(),
    ["Time", "Source", "Destination", "Candidate_NextHop", "Teacher_NextHop", "p1"]
]
group_top1 = (winners["Candidate_NextHop"] == winners["Teacher_NextHop"]).mean()
print(f"Group Top-1 Accuracy: {group_top1:.3f}")

# ================================================================
# 5) Save model + helper outputs
# ================================================================
# Save both model and metadata so I can reload later with context
art = {
    "model": clf,
    "features": feature_columns,
    "meta": {
        "type": "RandomForestClassifier",
        "labeling": "binary_per_candidate (teacher=DelayAware)",
        "train_size": len(Xtr),
        "test_size": len(Xte),
        "group_top1": float(group_top1)
    }
}
joblib.dump(art, OUTPUT_DIR / "rf_classifier_model.joblib")
winners.to_csv(OUTPUT_DIR / "rf_classifier_test_group_winners.csv", index=False)

print("Saved → rf_classifier_model.joblib")
print("Saved → rf_classifier_test_group_winners.csv")
print("Step 60 (Classifier) — DONE.")


In [ ]:
print(type(teacher))

first_key = next(iter(teacher.keys()))

print("First key:")
print(first_key)

print("Type of first key:")
print(type(first_key))

In [ ]:
print(teacher.columns.tolist())

In [ ]:
print([x for x in globals().keys() if "teacher" in x.lower()])

In [ ]:
print(type(teacher_paths))
print()

print("Length:")
print(len(teacher_paths))
print()

print("First item:")
print(teacher_paths[0])
print()

print("Type of first item:")
print(type(teacher_paths[0]))

In [ ]:
rows, mismatch = [], 0

for item in teacher_paths:

    t = int(item["Time"])
    s = int(item["Source"])
    d = int(item["Destination"])

    path = item["Path"]

    snap = timeline_df[
        timeline_df["Time"] <= t
    ].copy()

    if snap.empty:
        continue

    # Temporal trend computation
    snap = snap.sort_values(
        ["Satellite_ID", "Time"]
    ).copy()

    snap["Queue_Trend"] = (
        snap.groupby("Satellite_ID")["Queue_Length"]
        .diff()
        .fillna(0.0)
    )

    G_dist, G_delay, qmap = build_graphs_for_snapshot(
        snap,
        edges_df
    )

    # Keep only current snapshot afterward
    snap = snap[
        snap["Time"] == t
    ].copy()

    for step_i in range(len(path) - 1):

        u = int(path[step_i])
        best_v = int(path[step_i + 1])

        neighbors = nbrs.get(u, [])

        if not neighbors:
            continue

        if best_v not in neighbors:
            mismatch += 1
            continue

        du = haversine_km(u, d)

        for v in neighbors:

            v = int(v)

            prop_uv_s = float(
                G_dist[u][v]["weight"]
            )

            qv_s = float(
                qmap.get(v, 0.0)
            )

            dv = haversine_km(v, d)

            rows.append((
                t,
                s,
                d,
                step_i,
                u,
                v,
                prop_uv_s,
                qv_s,
                float(dv - du),
                int(v == best_v)
            ))

In [ ]:
print("Rows built:", len(rows))
print("Mismatch:", mismatch)

print("\nFirst row:")
print(rows[0])

In [ ]:
train_df = pd.DataFrame(rows, columns=train_cols)

print(train_df.head())
print(train_df.shape)

In [ ]:
print(final_cls.columns.tolist())
print()
print(final_cls.shape)

In [ ]:
print("Source_Queue_Trend" in final_cls.columns)
print("Candidate_Queue_Trend" in final_cls.columns)

In [ ]:
print("=== Non-zero trend entries ===")

non_zero = [
    (k, v)
    for k, v in qmap.items()
    if isinstance(k, tuple)
    and k[1] == "trend"
    and v != 0
]

print("Count:", len(non_zero))

print("\nFirst 20:")
for item in non_zero[:20]:
    print(item)

In [ ]:
# === Multi-seed eval with FIX 2 (11 features, proper trend) ===
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score

# Make sure we use the FIXED 11-feature list
FEATURES_11 = [
    'Source_Degree', 'Candidate_Degree',
    'Prop_Delay_Source_To_Cand', 'Prop_Delay_Cand_To_Dest', 'Prop_Delay_Source_To_Dest',
    'Is_Closer', 'Source_Queue_Delay', 'Candidate_Queue_Delay', 'Delay_Delta',
    'Source_Queue_Trend', 'Candidate_Queue_Trend'
]

groups_for_split = (final_cls["Time"].astype(str) + "_" +
                    final_cls["Source"].astype(str) + "_" +
                    final_cls["Destination"].astype(str))

def compute_top1(test_df, proba_col):
    correct, total = 0, 0
    for _, grp in test_df.groupby(["Time", "Source", "Destination"]):
        if grp["y"].sum() == 0: continue
        if grp.loc[grp[proba_col].idxmax(), "y"] == 1: correct += 1
        total += 1
    return correct / total

SEEDS = [42, 123, 456, 789, 2024, 1, 7, 314, 99, 2025]

print(f"{'Seed':<6} {'Top-1':>8} {'OOB':>8} {'BalAcc':>8} {'Prec':>8} {'Rec':>8}")
print("-" * 50)
results = []
for seed in SEEDS:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    tr, te = next(gss.split(final_cls, final_cls['y'], groups=groups_for_split))
    
    train_df = final_cls.iloc[tr]
    test_df = final_cls.iloc[te].copy()
    
    rf = RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, class_weight='balanced',
        oob_score=True, random_state=seed, n_jobs=-1
    )
    rf.fit(train_df[FEATURES_11], train_df['y'])
    
    test_df['proba'] = rf.predict_proba(test_df[FEATURES_11])[:, 1]
    pred = (test_df['proba'] >= 0.5).astype(int)
    
    r = {
        'Seed': seed,
        'Top1': compute_top1(test_df, 'proba'),
        'OOB': rf.oob_score_,
        'BalAcc': balanced_accuracy_score(test_df['y'], pred),
        'Prec': precision_score(test_df['y'], pred, zero_division=0),
        'Rec': recall_score(test_df['y'], pred, zero_division=0),
    }
    results.append(r)
    print(f"{r['Seed']:<6} {r['Top1']:>8.4f} {r['OOB']:>8.4f} "
          f"{r['BalAcc']:>8.4f} {r['Prec']:>8.4f} {r['Rec']:>8.4f}")

import pandas as pd
res_df = pd.DataFrame(results)
print(f"\nMean ± Std:")
for col in ['Top1', 'OOB', 'BalAcc', 'Prec', 'Rec']:
    print(f"  {col:<8}: {res_df[col].mean():.4f} ± {res_df[col].std(ddof=1):.4f}")

# Save for later
res_df.to_csv('multi_seed_fix2_11feat.csv', index=False)
print("\n✓ Saved multi_seed_fix2_11feat.csv")

In [ ]:
# === Compute BCa Bootstrap 95% CIs ===
from scipy import stats
import numpy as np

def bca_bootstrap_ci(data, n_boot=10000, alpha=0.05, seed=42):
    data = np.asarray(data)
    n = len(data)
    rng = np.random.RandomState(seed)
    boot = np.array([np.mean(rng.choice(data, n, replace=True)) for _ in range(n_boot)])
    obs = np.mean(data)
    p = (boot < obs).mean()
    if p in (0, 1):
        return float(np.percentile(boot, 100*alpha/2)), float(np.percentile(boot, 100*(1-alpha/2)))
    z0 = stats.norm.ppf(p)
    jack = np.array([np.mean(np.delete(data, i)) for i in range(n)])
    jm = jack.mean()
    num = np.sum((jm - jack)**3)
    den = 6 * (np.sum((jm - jack)**2))**1.5
    a = num/den if den > 0 else 0
    z_a = stats.norm.ppf(alpha/2)
    z_1a = stats.norm.ppf(1-alpha/2)
    p1 = stats.norm.cdf(z0 + (z0+z_a)/(1 - a*(z0+z_a)))
    p2 = stats.norm.cdf(z0 + (z0+z_1a)/(1 - a*(z0+z_1a)))
    return float(np.percentile(boot, 100*p1)), float(np.percentile(boot, 100*p2))

res_df = pd.read_csv('multi_seed_fix2_11feat.csv')

print("BCa Bootstrap 95% CIs (FIX 2, 11 features):")
print("-" * 60)
for col in ['Top1', 'OOB', 'BalAcc', 'Prec', 'Rec']:
    vals = res_df[col].values
    mean = np.mean(vals)
    std = np.std(vals, ddof=1)
    lo, hi = bca_bootstrap_ci(vals)
    print(f"  {col:<8}: {mean:.4f} ± {std:.4f}  BCa CI: [{lo:.4f}, {hi:.4f}]")

In [ ]:
# === Y-randomization control ===
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

print("Y-randomization (shuffle labels, retrain):")
print(f"{'Seed':<6} {'Top1 (real)':>12} {'Top1 (Y-rand)':>14}")
print("-" * 35)

y_rand_results = []
for seed in [42, 123, 456, 789, 2024, 1, 7, 314, 99, 2025]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    tr, te = next(gss.split(final_cls, final_cls['y'], groups=groups_for_split))
    train_df = final_cls.iloc[tr]
    test_df = final_cls.iloc[te].copy()
    
    # Shuffle labels in training
    rng = np.random.RandomState(seed)
    y_shuf = rng.permutation(train_df['y'].values)
    
    rf_yr = RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, class_weight='balanced',
        random_state=seed, n_jobs=-1
    )
    rf_yr.fit(train_df[FEATURES_11], y_shuf)
    test_df['proba'] = rf_yr.predict_proba(test_df[FEATURES_11])[:, 1]
    
    yr_top1 = compute_top1(test_df, 'proba')
    real_top1 = res_df[res_df['Seed'] == seed]['Top1'].values[0]
    y_rand_results.append(yr_top1)
    print(f"{seed:<6} {real_top1:>12.4f} {yr_top1:>14.4f}")

print(f"\nY-rand mean ± std: {np.mean(y_rand_results):.4f} ± {np.std(y_rand_results, ddof=1):.4f}")

# Wilcoxon
from scipy.stats import wilcoxon
W, p = wilcoxon(res_df['Top1'].values, y_rand_results)
print(f"Wilcoxon (real vs Y-rand): W = {W:.2f}, p = {p:.6f}")

In [ ]:
# === Held-out time evaluation ===
TRAIN_TIMES = [30, 60, 90]
TEST_TIMES = [120, 150]

train_mask = final_cls["Time"].isin(TRAIN_TIMES)
test_mask = final_cls["Time"].isin(TEST_TIMES)

print(f"Train times {TRAIN_TIMES}: {train_mask.sum()} rows, "
      f"{final_cls.loc[train_mask].groupby(['Time','Source','Destination']).ngroups} groups")
print(f"Test times {TEST_TIMES}:  {test_mask.sum()} rows, "
      f"{final_cls.loc[test_mask].groupby(['Time','Source','Destination']).ngroups} groups")

# Check unseen pairs
tr_pairs = set(zip(final_cls.loc[train_mask, "Source"], final_cls.loc[train_mask, "Destination"]))
te_pairs = set(zip(final_cls.loc[test_mask, "Source"], final_cls.loc[test_mask, "Destination"]))
unseen = te_pairs - tr_pairs
print(f"Unseen pairs in test: {len(unseen)}/{len(te_pairs)} ({100*len(unseen)/len(te_pairs):.0f}%)")

print(f"\n{'Seed':<6} {'Top1':>8} {'BalAcc':>8} {'Prec':>8} {'Rec':>8}")
print("-" * 40)
ho_results = []
for seed in [42, 123, 456, 789, 2024, 1, 7, 314, 99, 2025]:
    rf = RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, class_weight='balanced',
        random_state=seed, n_jobs=-1
    )
    rf.fit(final_cls.loc[train_mask, FEATURES_11], final_cls.loc[train_mask, 'y'])
    
    test_eval = final_cls.loc[test_mask].copy()
    test_eval['proba'] = rf.predict_proba(test_eval[FEATURES_11])[:, 1]
    pred = (test_eval['proba'] >= 0.5).astype(int)
    
    r = {
        'Seed': seed,
        'Top1': compute_top1(test_eval, 'proba'),
        'BalAcc': balanced_accuracy_score(test_eval['y'], pred),
        'Prec': precision_score(test_eval['y'], pred, zero_division=0),
        'Rec': recall_score(test_eval['y'], pred, zero_division=0),
    }
    ho_results.append(r)
    print(f"{seed:<6} {r['Top1']:>8.4f} {r['BalAcc']:>8.4f} {r['Prec']:>8.4f} {r['Rec']:>8.4f}")

ho_df = pd.DataFrame(ho_results)
print(f"\nHeld-out time Mean ± Std:")
for col in ['Top1', 'BalAcc', 'Prec', 'Rec']:
    print(f"  {col:<8}: {ho_df[col].mean():.4f} ± {ho_df[col].std(ddof=1):.4f}")

ho_df.to_csv('held_out_time_fix2.csv', index=False)

In [ ]:
import pandas as pd

imp = pd.DataFrame({
    "Feature": FEATURES_11,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

print(imp)

In [ ]:
# ========================================================================================
# Step 61 — Diagnostics for Overfitting
#
# Goal:
# I want to check if my classifier is really learning useful patterns
# or if it’s just memorizing labels. This is important for credibility.
#
# Main metric:
#   Group Top-1 → for each (Time, Source, Destination), does the
#   top predicted candidate (argmax of probability) match the Teacher?
#
# Two quick checks I decided to include:
#   (A) Simple learning curve with 3 training sizes (20%, 60%, 100%)
#       → should show better accuracy as more data is used.
#   (B) Y-Randomization with 5 repeats
#       → if labels are shuffled, accuracy should collapse to random.
# ========================================================================================


RANDOM_SEED = 42
OUTPUT_DIR = Path("output_t_1_sec_analysis")

# ---------- 0) Prepare input data (reproducible) ----------
# If Step 60 variables are still in memory, I reuse them.
# Otherwise, I rebuild from saved CSV. This keeps the notebook
# self-contained (so I can re-run without the whole pipeline).
if 'final_cls' in globals() and 'feature_columns' in globals():
    use_df = final_cls.copy()
    feats  = feature_columns
else:
    raw = pd.read_csv(OUTPUT_DIR / "ml_classifier_labeled_candidates.csv")

    # Check if feature functions are loaded; if not, try importing
    need_funcs = ["add_degree_features","add_geometric_features",
                  "add_congestion_features","add_temporal_features"]
    have_all = all(fn in globals() for fn in need_funcs)

    if not have_all:
        try:
            from features import add_degree_features, add_geometric_features, \
                                add_congestion_features, add_temporal_features
            have_all = True
        except Exception:
            pass
    assert have_all, "Feature functions not available. Please run Step 2.5/Step 60 first."

    # Rebuild features exactly like in Step 60 (so results are consistent)
    tmp = raw.copy()
    tmp = add_degree_features(tmp, edges_df)
    tmp = add_geometric_features(tmp, pos_df, edges_df)
    tmp = add_congestion_features(tmp, full_sim_df)
    tmp = add_temporal_features(tmp, full_sim_df)

    feats = [
        "Source_Degree","Candidate_Degree",
        "Prop_Delay_Source_To_Cand","Prop_Delay_Cand_To_Dest","Prop_Delay_Source_To_Dest",
        "Is_Closer","Source_Queue_Delay","Candidate_Queue_Delay","Delay_Delta",
        "Source_Queue_Trend","Candidate_Queue_Trend"
    ]
    use_df = tmp.dropna(subset=feats + ["y"]).copy()

# Build feature matrix + labels + group IDs
X = use_df[feats].to_numpy()
y = use_df["y"].astype('int8').to_numpy()
groups = (use_df["Time"].astype(str) + "_" +
          use_df["Source"].astype(str) + "_" +
          use_df["Destination"].astype(str)).to_numpy()

# Consistent split → prevents leakage across groups
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))
Xtr, Xte = X[tr_idx], X[te_idx]
Ytr, Yte = y[tr_idx], y[te_idx]
DF_te = use_df.iloc[te_idx].copy()

print(f"Train={len(Xtr)} | Test={len(Xte)} | Groups(total)={use_df[['Time','Source','Destination']].drop_duplicates().shape[0]}")

# ---------- Helper: compute Group Top-1 ----------
def group_top1(df_test: pd.DataFrame, proba_pos: np.ndarray) -> float:
    """Check if the candidate with the highest probability matches Teacher."""
    tmp = df_test.copy()
    tmp["p1"] = proba_pos
    idx = tmp.groupby(["Time","Source","Destination"])["p1"].idxmax()
    winners = tmp.loc[idx, ["Candidate_NextHop","Teacher_NextHop"]]
    return float((winners["Candidate_NextHop"] == winners["Teacher_NextHop"]).mean())

# ---------- 1) Train base model and baseline accuracy ----------
# I reuse the same RF setup as Step 60 to keep things consistent.
base = RandomForestClassifier(
    n_estimators=400, max_depth=None, min_samples_leaf=2,
    class_weight="balanced", n_jobs=-1, random_state=RANDOM_SEED, oob_score=True
).fit(Xtr, Ytr)

print("OOB score (rough sanity):", getattr(base, "oob_score_", None))

p_test = base.predict_proba(Xte)[:,1]
gtop1_full = group_top1(DF_te, p_test)
bal_acc_full = balanced_accuracy_score(Yte, base.predict(Xte))
print(f"Group Top-1 (full train) = {gtop1_full:.3f} | Balanced Acc (test) = {bal_acc_full:.3f}")

# ---------- 2) Simple learning curve (3 points) ----------
# Idea: as I feed the model more data, performance should improve.
fracs = [0.2, 0.6, 1.0]
top1_list, bal_tr_list, bal_te_list = [], [], []

rng = np.random.RandomState(RANDOM_SEED)
for f in fracs:
    k = max(64, int(f * len(Xtr)))  # avoid tiny training sets
    pick = rng.choice(len(Xtr), size=k, replace=False)
    Xm, Ym = Xtr[pick], Ytr[pick]
    m = clone(base).fit(Xm, Ym)

    bal_tr_list.append(balanced_accuracy_score(Ym, m.predict(Xm)))
    bal_te_list.append(balanced_accuracy_score(Yte, m.predict(Xte)))
    top1_list.append(group_top1(DF_te, m.predict_proba(Xte)[:,1]))

plt.figure(figsize=(7,4.5))
plt.plot(fracs, top1_list, marker='o')
plt.xlabel("Fraction of Training Data")
plt.ylabel("Group Top-1 (Test)")
plt.title("Learning Curve — Group Top-1")
plt.grid(True); plt.show()

# ---------- 3) Y-Randomization (5 repeats + random baseline) ----------
# If I shuffle labels, the model should collapse near random guessing.
cand_per_group = DF_te.groupby(["Time","Source","Destination"]).size().to_numpy()
rand_baseline = float(1.0 / cand_per_group.mean())  # expected random hit rate

gtop1_shuf = []
for r in range(5):
    Ytr_shuf = Ytr.copy()
    rng.shuffle(Ytr_shuf)  # destroy label → only noise remains
    m_shuf = clone(base).fit(Xtr, Ytr_shuf)
    gtop1_shuf.append(group_top1(DF_te, m_shuf.predict_proba(Xte)[:,1]))
gtop1_shuf = np.array(gtop1_shuf)

plt.figure(figsize=(6,4))
vals = [gtop1_full, gtop1_shuf.mean(), rand_baseline]
labs = ["Real y", "Shuffled y (mean)", "Random baseline"]
bars = plt.bar(labs, vals, alpha=0.85)
for b, v in zip(bars, vals):
    plt.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontweight="bold")
plt.ylabel("Group Top-1"); plt.title("Y-Randomization Check")
plt.grid(axis='y'); plt.show()

# ---------- 4) Save diagnostics ----------
diag = pd.DataFrame({
    "train_frac": fracs,
    "train_bal_acc": bal_tr_list,
    "test_bal_acc": bal_te_list,
    "test_group_top1": top1_list
})
diag.to_csv(OUTPUT_DIR / "step61_classifier_diagnostics.csv", index=False)

print("Saved: step61_classifier_diagnostics.csv")
print(f"Random baseline ≈ {rand_baseline:.3f} | Shuffled y mean = {gtop1_shuf.mean():.3f} ± {gtop1_shuf.std():.3f}")



---

In [ ]:
from pathlib import Path
import joblib

OUTPUT_DIR = Path("output_t_1_sec_analysis")
print(OUTPUT_DIR.exists())
print((OUTPUT_DIR / "rf_classifier_model.joblib").exists())

In [ ]:
# ==============================================================================
# Step 62 — Final Evaluation: DistanceOnly vs DelayAware vs ML-Agent (Classifier)
#
# Why I’m doing this:
# This is the big comparison step. I now put the ML agent (classifier from Step 60)
# against two baselines:
#   * DistanceOnly   → shortest path in propagation time (ignores congestion)
#   * DelayAware     → shortest path in propagation + queue delay (oracle baseline)
#   * ML-Agent  → my classifier-based greedy agent (argmax P(y=1))
#
# I’ll compare them on three key metrics:
#   - End-to-End Delay
#   - Path Drop Rate
#   - Normalized Throughput
#
# Outputs:
#   - ml_agent_final_eval_triple_metrics_classifier.csv   (raw per-sample results)
#   - ml_agent_final_eval_summary_classifier.csv          (mean per algorithm)
#   - 3 bar plots (Delay, DropRate, NormThroughput)
# ==============================================================================
import joblib

print("\n=== Step 62: Final Evaluation (Delay + DropRate + NormThroughput) — Classifier ===")

# --- 1) Load trained CLASSIFIER model -----------------------------------------

mdl = joblib.load(OUTPUT_DIR / "rf_classifier_model.joblib")
rf = mdl["model"]
TRAIN_FEATURES = mdl["features"]
if not TRAIN_FEATURES:
    raise ValueError("No feature list found in saved classifier. Check Step 60 output.")

# Make sure features match exactly what the model was trained on
expected = {
    "Source_Degree","Candidate_Degree","Prop_Delay_Source_To_Cand",
    "Prop_Delay_Cand_To_Dest","Prop_Delay_Source_To_Dest","Is_Closer",
    "Source_Queue_Delay","Candidate_Queue_Delay","Delay_Delta",
    "Source_Queue_Trend","Candidate_Queue_Trend"
}
assert set(TRAIN_FEATURES) == expected, f"Feature mismatch: {set(TRAIN_FEATURES)^expected}"
print("Loaded classifier features:", TRAIN_FEATURES)

# --- 2) Feature extraction helpers (same logic as Step 60) --------------------
SPEED_OF_LIGHT_KM_S = globals().get("SPEED_OF_LIGHT_KM_S", 299_792.458)
_pos = pos_df.set_index("Satellite_ID")[["Latitude","Longitude"]].to_dict("index")

def _haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(float(lat1)); lon1 = np.radians(float(lon1))
    lat2 = np.radians(float(lat2)); lon2 = np.radians(float(lon2))
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return R * (2*np.arctan2(np.sqrt(a), np.sqrt(1-a)))

def _straight_prop_seconds(a, b):
    """Approx propagation delay (straight-line haversine)."""
    pa = _pos.get(int(a)); pb = _pos.get(int(b))
    if pa is None or pb is None:
        return float("inf")
    d_km = _haversine_km(pa["Latitude"], pa["Longitude"], pb["Latitude"], pb["Longitude"])
    return d_km / SPEED_OF_LIGHT_KM_S

def featurize_row(u, v, d, G_dist, qmap):
    """Extract features for one candidate hop u→v when heading to dest d."""
    # Note: I keep it identical to Step 60 feature logic
    src_deg  = G_dist.degree[u]
    cand_deg = G_dist.degree[v]
    prop_uv_s = G_dist[u][v]["weight"]                # ISL propagation (sec)
    prop_vd_s = _straight_prop_seconds(v, d)          # candidate→dest
    prop_ud_s = _straight_prop_seconds(u, d)          # source→dest
    is_closer = int(prop_vd_s < prop_ud_s)
    q_src = qmap.get(u, 0.0)
    q_cnd = qmap.get(v, 0.0)
    delay_delta  = q_cnd - q_src
    q_src_trend  = qmap.get((u, "trend"), 0.0)
    q_cnd_trend  = qmap.get((v, "trend"), 0.0)
    return {
        "Source_Degree": src_deg,
        "Candidate_Degree": cand_deg,
        "Prop_Delay_Source_To_Cand": prop_uv_s,
        "Prop_Delay_Cand_To_Dest": prop_vd_s,
        "Prop_Delay_Source_To_Dest": prop_ud_s,
        "Is_Closer": is_closer,
        "Source_Queue_Delay": q_src,
        "Candidate_Queue_Delay": q_cnd,
        "Delay_Delta": delay_delta,
        "Source_Queue_Trend": q_src_trend,
        "Candidate_Queue_Trend": q_cnd_trend
    }

def features_matrix(u, v, d, G_dist, qmap):
    """Wrap featurize_row into a one-row DataFrame (for sklearn)."""
    f = featurize_row(u, v, d, G_dist, qmap)
    return pd.DataFrame([[f[k] for k in TRAIN_FEATURES]], columns=TRAIN_FEATURES)

# --- 3) ML agent policy (classifier argmax) -----------------------------------
def ml_agent_classifier_route(source, dest, G_dist, qmap, max_hops=50, allow_backtrack=False):
    """Routing policy: greedy argmax P(y=1) from RF classifier."""
    u = int(source); d = int(dest)
    path = [u]; visited = {u}
    for _ in range(max_hops):
        if u == d:
            return path, False
        nbrs = list(G_dist.neighbors(u))
        if not nbrs:
            return None, False
        last = path[-2] if len(path) >= 2 else None
        if not allow_backtrack and last is not None:
            nbrs = [v for v in nbrs if v != last] or nbrs
        scored = []
        for v in nbrs:
            try:
                X = features_matrix(u, v, d, G_dist, qmap)
                p1 = float(rf.predict_proba(X)[0, 1])
                dest_prop = float(X.loc[X.index[0], "Prop_Delay_Cand_To_Dest"])
            except Exception:
                return None, True
            scored.append((p1, dest_prop, int(v)))
        # Pick neighbor with max probability; break ties by closeness
        scored.sort(key=lambda z: (-z[0], z[1]))
        v_best = next((v for _, __, v in scored if v not in visited), None)
        if v_best is None:
            return None, False
        path.append(v_best); visited.add(v_best); u = v_best
    return None, False

# --- 4) DropRate & NormThroughput helpers -------------------------------------
eps = 1e-12
try:
    stress_df = pd.read_csv(OUTPUT_DIR / "stress_test_results.csv")
    stress_df["DropRate"] = stress_df["Packets_Dropped"] / (
        stress_df["Packets_Dropped"] + stress_df["Packets_Served"] + eps
    )
    drop_rate_map = {(int(r.Time), int(r.Satellite_ID)): float(r.DropRate)
                     for r in stress_df.itertuples()}
    print("Loaded stress_test_results.csv for drop-rate mapping.")
except FileNotFoundError:
    drop_rate_map = {}
    print("stress_test_results.csv not found — assuming zero drop rate.")

def path_drop_rate(path, t):
    """Prob of at least one packet drop along path at time t."""
    p_succ = 1.0
    for n in path[1:]:
        p_succ *= (1.0 - drop_rate_map.get((int(t), int(n)), 0.0))
    return 1.0 - p_succ

def path_norm_throughput(path, t, e2e):
    """Normalize throughput = success prob / delay."""
    drop = path_drop_rate(path, t)
    if e2e is None or e2e <= 0:
        return 0.0
    return (1.0 - drop) / float(e2e)

# --- 5) Evaluation loop -------------------------------------------------------
rng = np.random.default_rng(42)
rows = []

for t_snapshot in SNAPSHOTS_TO_ANALYZE:

    # ==========================================================
    # Temporal-aware snapshot construction
    # ==========================================================
    # Use all historical snapshots up to current time
    # so Queue_Trend can be computed leakage-free.
    # ==========================================================

    snap = timeline_df[
        timeline_df["Time"] <= t_snapshot
    ].copy()

    if snap.empty:
        continue

    # Sort temporally per satellite
    snap = snap.sort_values(
        ["Satellite_ID", "Time"]
    ).copy()

    # Temporal congestion trend
    snap["Queue_Trend"] = (
        snap.groupby("Satellite_ID")["Queue_Length"]
        .diff()
        .fillna(0.0)
    )

    # Build graphs using temporal-aware snapshot
    G_dist, G_delay, qmap = build_graphs_for_snapshot(
        snap,
        edges_df
    )

    # Keep only current timestep afterward
    snap = snap[
        snap["Time"] == t_snapshot
    ].copy()

    # ==========================================================
    # Scenario evaluation
    # ==========================================================

    for (region_a, region_b) in INTERCONTINENTAL_SCENARIOS:

        pairs = _pairs_for(
            t_snapshot,
            region_a,
            region_b,
            NUM_PAIRS_TO_TEST
        )

        if not pairs:

            # Fallback random sampling
            nodes_a = snap.loc[
                snap["Region"] == region_a,
                "Satellite_ID"
            ].tolist()

            nodes_b = snap.loc[
                snap["Region"] == region_b,
                "Satellite_ID"
            ].tolist()

            if not nodes_a or not nodes_b:
                continue

            pairs = []
            tries = 0

            while (
                len(pairs) < NUM_PAIRS_TO_TEST
                and tries < 20 * NUM_PAIRS_TO_TEST
            ):

                s = rng.choice(nodes_a)
                d = rng.choice(nodes_b)

                if s != d:
                    pairs.append((int(s), int(d)))

                tries += 1

        # ======================================================
        # Evaluate each source-destination pair
        # ======================================================

        for s, d in pairs:

            # --------------------------------------------------
            # DistanceOnly baseline
            # --------------------------------------------------
            try:

                p_do = nx.dijkstra_path(
                    G_dist,
                    s,
                    d,
                    weight="weight"
                )

                e2e_do = get_true_e2e_delay(
                    p_do,
                    G_dist,
                    qmap
                )

                rows.append({
                    "Time": int(t_snapshot),
                    "Algorithm": "DistanceOnly",
                    "E2E_Delay_s": float(e2e_do),
                    "Path_DropRate": path_drop_rate(
                        p_do,
                        t_snapshot
                    ),
                    "Norm_Throughput": path_norm_throughput(
                        p_do,
                        t_snapshot,
                        e2e_do
                    )
                })

            except nx.NetworkXNoPath:
                continue

            # --------------------------------------------------
            # DelayAware oracle baseline
            # --------------------------------------------------
            try:

                p_da = nx.dijkstra_path(
                    G_delay,
                    s,
                    d,
                    weight="weight"
                )

                e2e_da = get_true_e2e_delay(
                    p_da,
                    G_dist,
                    qmap
                )

                rows.append({
                    "Time": int(t_snapshot),
                    "Algorithm": "DelayAware",
                    "E2E_Delay_s": float(e2e_da),
                    "Path_DropRate": path_drop_rate(
                        p_da,
                        t_snapshot
                    ),
                    "Norm_Throughput": path_norm_throughput(
                        p_da,
                        t_snapshot,
                        e2e_da
                    )
                })

            except nx.NetworkXNoPath:
                continue

            # --------------------------------------------------
            # ML-Agent (Classifier)
            # --------------------------------------------------

            p_ml, used_fb = ml_agent_classifier_route(
                s,
                d,
                G_dist,
                qmap
            )

            # Fallback to DelayAware if ML fails
            if p_ml is None:

                try:
                    p_ml = nx.dijkstra_path(
                        G_delay,
                        s,
                        d,
                        weight="weight"
                    )

                except nx.NetworkXNoPath:
                    continue

            e2e_ml = get_true_e2e_delay(
                p_ml,
                G_dist,
                qmap
            )

            rows.append({
                "Time": int(t_snapshot),
                "Algorithm": "ML-Agent (Classifier)",
                "E2E_Delay_s": float(e2e_ml),
                "Path_DropRate": path_drop_rate(
                    p_ml,
                    t_snapshot
                ),
                "Norm_Throughput": path_norm_throughput(
                    p_ml,
                    t_snapshot,
                    e2e_ml
                )
            })

# --- 6) Aggregate, save, and plot ---------------------------------------------
if not rows:
    print("No results produced.")
else:
    res = pd.DataFrame(rows)

    final_cmp = (
        res.groupby("Algorithm", as_index=False)
           .agg(E2E_Delay_s=("E2E_Delay_s","mean"),
                Path_DropRate=("Path_DropRate","mean"),
                Norm_Throughput=("Norm_Throughput","mean"))
           .sort_values("E2E_Delay_s")
    )
    print("\n=== Final Performance Comparison (Averaged Results) ===")
    print(final_cmp)

    out_rows   = OUTPUT_DIR / "ml_agent_final_eval_triple_metrics_classifier.csv"
    out_summary= OUTPUT_DIR / "ml_agent_final_eval_summary_classifier.csv"
    res.to_csv(out_rows, index=False)
    final_cmp.to_csv(out_summary, index=False)
    print("Saved:", out_rows.name, "and", out_summary.name)

    # Simple bar plots for each metric
    ax = final_cmp.plot(x="Algorithm", y="E2E_Delay_s", kind="bar", rot=0,
                        title="Mean End-to-End Delay (Lower is Better)", figsize=(8,5))
    ax.set_ylabel("Seconds"); ax.set_xlabel("")
    plt.tight_layout(); plt.show()

    ax = final_cmp.plot(x="Algorithm", y="Path_DropRate", kind="bar", rot=0,
                        title="Mean Path Drop Rate (Lower is Better)", figsize=(8,5))
    ax.set_ylabel("Drop Rate"); ax.set_xlabel("")
    plt.tight_layout(); plt.show()

    ax = final_cmp.plot(x="Algorithm", y="Norm_Throughput", kind="bar", rot=0,
                        title="Mean Normalized Throughput (Higher is Better)", figsize=(8,5))
    ax.set_ylabel("Norm Throughput (1/s)"); ax.set_xlabel("")
    plt.tight_layout(); plt.show()


---

In [ ]:

# === Step 63 — Rank the 3 Agents (Classifier outputs) ==========================
#
# Why I’m doing this:
# After Step 62 gave me raw evaluation metrics, I now want to make
# the results easier to compare. Looking at 3 plots and CSVs is messy.
# By turning everything into ranks + a simple “points” system (Borda),
# I can say clearly which agent comes out on top overall.
#
# Agents being compared:
#   - DistanceOnly
#   - DelayAware (teacher/oracle)
#   - ML-Agent (Classifier)
#
# Inputs:
#   - ml_agent_final_eval_summary_classifier.csv   (preferred summary)
#   - ml_agent_final_eval_triple_metrics_classifier.csv (fallback if summary missing)
# ===============================================================================

print("\n=== Step 64: Ranking the Agents (Classifier) ===")

# --- 1) Locate previous outputs ------------------------------------------------
# I try to be flexible:
# - If summary is already in memory (from Step 62), reuse it (fast).
# - Otherwise load the saved summary CSV.
# - If missing, rebuild from the detailed metrics CSV.
# - If none exist, fail loudly → I must have skipped Step 62.
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path("output_t_1_sec_analysis")

summary_csv = OUTPUT_DIR / "ml_agent_final_eval_summary_classifier.csv"
detail_csv  = OUTPUT_DIR / "ml_agent_final_eval_triple_metrics_classifier.csv"

if "final_cmp" in globals():
    summary = final_cmp.copy()
elif summary_csv.exists():
    summary = pd.read_csv(summary_csv)
elif detail_csv.exists():
    df = pd.read_csv(detail_csv)
    summary = (
        df.groupby("Algorithm", as_index=False)
          .agg(E2E_Delay_s=("E2E_Delay_s", "mean"),
               Path_DropRate=("Path_DropRate", "mean"),
               Norm_Throughput=("Norm_Throughput", "mean"))
    )
else:
    raise FileNotFoundError("No evaluation results found. Run Step 62 first.")

# Quick sanity: make sure all 3 algorithms are present
expected_algos = {"DistanceOnly", "DelayAware", "ML-Agent (Classifier)"}
have_algos = set(summary["Algorithm"].unique())
missing = expected_algos - have_algos
if missing:
    print("Warning: expected algorithms not found in summary:", missing)

# --- 2) Clean presentation -----------------------------------------------------
# Round numbers → easier to read, doesn’t change ranking
pretty = summary.copy()
pretty["E2E_Delay_s"]     = pretty["E2E_Delay_s"].round(4)
pretty["Path_DropRate"]   = pretty["Path_DropRate"].round(4)
pretty["Norm_Throughput"] = pretty["Norm_Throughput"].round(6)

print("\n=== Summary (low delay & drop are better; high throughput is better) ===")
print(pretty.sort_values("E2E_Delay_s").to_string(index=False))

# --- 3) Per-metric rankings ----------------------------------------------------
# Instead of juggling decimals, convert each metric into ranks:
#   - 1 = best, ties get same rank
def rank_series(s: pd.Series, *, ascending: bool) -> pd.Series:
    """Ranking helper (ascending=True means smaller is better)."""
    return s.rank(method="dense", ascending=ascending).astype(int)

r_delay = rank_series(summary["E2E_Delay_s"], ascending=True)
r_drop  = rank_series(summary["Path_DropRate"], ascending=True)
r_thr   = rank_series(summary["Norm_Throughput"], ascending=False)

ranks = pd.DataFrame({
    "Algorithm": summary["Algorithm"],
    "Rank_Delay (low best)": r_delay,
    "Rank_DropRate (low best)": r_drop,
    "Rank_Throughput (high best)": r_thr,
})

# --- 4) Borda-style overall score ---------------------------------------------
# Simple points system: best = N pts, next = N-1, ..., worst = 1
# Summing across metrics gives one “overall winner”.
N = len(summary)
points = pd.DataFrame({
    "Algorithm": summary["Algorithm"],
    "Pts_Delay":       (N - r_delay + 1),
    "Pts_DropRate":    (N - r_drop  + 1),
    "Pts_Throughput":  (N - r_thr   + 1),
})
points["Total_Points"] = points[["Pts_Delay","Pts_DropRate","Pts_Throughput"]].sum(axis=1)

# --- 5) Print rankings ---------------------------------------------------------
# Show each metric separately → professor can check fairness
for col, asc in [
    ("E2E_Delay_s", True),
    ("Path_DropRate", True),
    ("Norm_Throughput", False),
]:
    ordered = summary.sort_values(col, ascending=asc)[["Algorithm", col]].reset_index(drop=True)
    ordered.index = ordered.index + 1
    label = {
        "E2E_Delay_s": "Delay (s) — lower is better",
        "Path_DropRate": "Drop Rate — lower is better",
        "Norm_Throughput": "Normalized Throughput — higher is better",
    }[col]
    print(f"\n{label}:")
    print(ordered.to_string())

print("\n=== Rank table (1 = best) ===")
print(ranks.sort_values("Rank_Delay (low best)").to_string(index=False))

print("\n=== Overall ranking (Borda score; higher = better) ===")
print(points.sort_values("Total_Points", ascending=False).to_string(index=False))

# --- 6) Quick winners + save ---------------------------------------------------
# Pick the single best per metric + overall winner
best_delay = summary.loc[summary["E2E_Delay_s"].idxmin(), "Algorithm"]
best_drop  = summary.loc[summary["Path_DropRate"].idxmin(), "Algorithm"]
best_thr   = summary.loc[summary["Norm_Throughput"].idxmax(), "Algorithm"]
overall_winner = points.sort_values("Total_Points", ascending=False).iloc[0]["Algorithm"]

print("\n Winners:")
print(f" - Lowest mean delay:             {best_delay}")
print(f" - Lowest mean drop rate:         {best_drop}")
print(f" - Highest normalized throughput: {best_thr}")
print(" -----------------------------------------")
print(f" - Overall (Borda score):         {overall_winner}")

# Save for later steps (keeps workflow modular)
ranks_out  = OUTPUT_DIR / "agent_rankings_classifier.csv"
points_out = OUTPUT_DIR / "agent_borda_scores_classifier.csv"
ranks.to_csv(ranks_out, index=False)
points.to_csv(points_out, index=False)
print(f"Saved: {ranks_out.name} , {points_out.name}")
